# Black-Box Optimisation Capstone

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


## 1. Setup

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import norm

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    ConstantKernel,
    Matern,
    WhiteKernel,
)

## 2. Initial Data

In [ ]:
INITIAL_DATA_PATH = (
    "/content/drive/MyDrive/Imperial_BBO/data/initial"
)

print("Initial-data folder:")
print(INITIAL_DATA_PATH)

print("\nFiles found:")
for filename in sorted(os.listdir(INITIAL_DATA_PATH)):
    print(filename)

Initial-data folder:
/content/drive/MyDrive/Imperial_BBO/data/initial

Files found:
function_1_inputs.npy
function_1_outputs.npy
function_2_inputs.npy
function_2_outputs.npy
function_3_inputs.npy
function_3_outputs.npy
function_4_inputs.npy
function_4_outputs.npy
function_5_inputs.npy
function_5_outputs.npy
function_6_inputs.npy
function_6_outputs.npy
function_7_inputs.npy
function_7_outputs.npy
function_8_inputs.npy
function_8_outputs.npy


## 3. Load Datasets

In [ ]:

datasets = {}

for function_number in range(1, 9):
    input_path = os.path.join(
        INITIAL_DATA_PATH,
        f"function_{function_number}_inputs.npy",
    )

    output_path = os.path.join(
        INITIAL_DATA_PATH,
        f"function_{function_number}_outputs.npy",
    )

    X = np.load(input_path)
    y = np.load(output_path).reshape(-1)

    datasets[function_number] = {
        "X": X,
        "y": y,
    }

    print(
        f"Function {function_number}: "
        f"inputs {X.shape}, "
        f"outputs {y.shape}, "
        f"best initial output = {np.max(y):.6f}"
    )

Function 1: inputs (10, 2), outputs (10,), best initial output = 0.000000
Function 2: inputs (10, 2), outputs (10,), best initial output = 0.611205
Function 3: inputs (15, 3), outputs (15,), best initial output = -0.034835
Function 4: inputs (30, 4), outputs (30,), best initial output = -4.025542
Function 5: inputs (20, 4), outputs (20,), best initial output = 1088.859618
Function 6: inputs (20, 5), outputs (20,), best initial output = -0.714265
Function 7: inputs (30, 6), outputs (30,), best initial output = 1.364968
Function 8: inputs (40, 8), outputs (40,), best initial output = 9.598482


## 4. Round 1 Results

In [ ]:
round_1_results = {
    1: {
        "x": [0.869082, 0.503499],
        "y": -4.760694916685198e-52,
    },
    2: {
        "x": [0.695224, 0.055659],
        "y": 0.5702046356280807,
    },
    3: {
        "x": [0.839692, 0.013811, 0.000331],
        "y": -0.19056713016385887,
    },
    4: {
        "x": [0.472556, 0.398468, 0.362086, 0.382374],
        "y": -1.1360324509695903,
    },
    5: {
        "x": [0.239337, 0.846790, 0.950799, 0.838420],
        "y": 1302.8485933219035,
    },
    6: {
        "x": [
            0.505435,
            0.279076,
            0.635842,
            0.932969,
            0.005307,
        ],
        "y": -0.3997806456044814,
    },
    7: {
        "x": [
            0.107435,
            0.759977,
            0.260986,
            0.218595,
            0.706498,
            0.617621,
        ],
        "y": 0.1124898567049083,
    },
    8: {
        "x": [
            0.052573,
            0.128542,
            0.058623,
            0.109321,
            0.970058,
            0.683181,
            0.152298,
            0.779592,
        ],
        "y": 9.9223106734526,
    },
}

print("Round 1 results loaded.")

Round 1 results loaded.


## 5. Update Datasets

In [ ]:
for function_number in range(1, 9):
    new_x = np.array(
        round_1_results[function_number]["x"],
        dtype=float,
    ).reshape(1, -1)

    new_y = float(
        round_1_results[function_number]["y"]
    )

    expected_dimensions = datasets[function_number]["X"].shape[1]

    if new_x.shape[1] != expected_dimensions:
        raise ValueError(
            f"Function {function_number}: "
            f"expected {expected_dimensions} values, "
            f"received {new_x.shape[1]}."
        )

    datasets[function_number]["X"] = np.vstack(
        [
            datasets[function_number]["X"],
            new_x,
        ]
    )

    datasets[function_number]["y"] = np.append(
        datasets[function_number]["y"],
        new_y,
    )

    print(
        f"Function {function_number}: "
        f"inputs {datasets[function_number]['X'].shape}, "
        f"outputs {datasets[function_number]['y'].shape}"
    )

Function 1: inputs (11, 2), outputs (11,)
Function 2: inputs (11, 2), outputs (11,)
Function 3: inputs (16, 3), outputs (16,)
Function 4: inputs (31, 4), outputs (31,)
Function 5: inputs (21, 4), outputs (21,)
Function 6: inputs (21, 5), outputs (21,)
Function 7: inputs (31, 6), outputs (31,)
Function 8: inputs (41, 8), outputs (41,)


## 6. Check Updated Data

In [ ]:
assert set(datasets.keys()) == set(range(1, 9))

for function_number in range(1, 9):
    X = datasets[function_number]["X"]
    y = datasets[function_number]["y"]

    assert X.shape[0] == y.shape[0]

    print(
        f"Function {function_number}: "
        f"{X.shape[0]} observations, "
        f"{X.shape[1]} dimensions, "
        f"current best = {np.max(y):.6f}"
    )

Function 1: 11 observations, 2 dimensions, current best = 0.000000
Function 2: 11 observations, 2 dimensions, current best = 0.611205
Function 3: 16 observations, 3 dimensions, current best = -0.034835
Function 4: 31 observations, 4 dimensions, current best = -1.136032
Function 5: 21 observations, 4 dimensions, current best = 1302.848593
Function 6: 21 observations, 5 dimensions, current best = -0.399781
Function 7: 31 observations, 6 dimensions, current best = 1.364968
Function 8: 41 observations, 8 dimensions, current best = 9.922311


## 7. Compare Results

In [ ]:

comparison_rows = []

for function_number in range(1, 9):
    y = datasets[function_number]["y"]

    initial_best = np.max(y[:-1])
    round_1_output = y[-1]
    updated_best = np.max(y)

    comparison_rows.append(
        {
            "Function": function_number,
            "Initial Best": initial_best,
            "Round 1 Output": round_1_output,
            "Updated Best": updated_best,
            "Improved": updated_best > initial_best,
        }
    )

comparison_df = pd.DataFrame(comparison_rows)

comparison_df

,Function,Initial Best,Round 1 Output,Updated Best,Improved
0,1,7.710875e-16,-4.760695e-52,7.710875e-16,False
1,2,6.112052e-01,5.702046e-01,6.112052e-01,False
2,3,-3.483531e-02,-1.905671e-01,-3.483531e-02,False
3,4,-4.025542e+00,-1.136032e+00,-1.136032e+00,True
4,5,1.088860e+03,1.302849e+03,1.302849e+03,True
5,6,-7.142649e-01,-3.997806e-01,-3.997806e-01,True
6,7,1.364968e+00,1.124899e-01,1.364968e+00,False
7,8,9.598482e+00,9.922311e+00,9.922311e+00,True


## 8. Round 2 Results

In [ ]:
round_2_results = {
    1: {
        "x": [0.514232, 0.063435],
        "y": 1.370784784609222e-96,
    },
    2: {
        "x": [0.965674, 0.216783],
        "y": 0.09495924056129348,
    },
    3: {
        "x": [0.692232, 0.468397, 0.303034],
        "y": -0.08132276509860875,
    },
    4: {
        "x": [0.378762, 0.416287, 0.401547, 0.396799],
        "y": 0.36846288672365235,
    },
    5: {
        "x": [0.536437, 0.790238, 0.999914, 0.909757],
        "y": 2006.076446872901,
    },
    6: {
        "x": [
            0.414036,
            0.038018,
            0.331093,
            0.980820,
            0.002463,
        ],
        "y": -0.9485322549641995,
    },
    7: {
        "x": [
            0.926472,
            0.634833,
            0.288693,
            0.796292,
            0.442100,
            0.743127,
        ],
        "y": 0.016812595771486926,
    },
    8: {
        "x": [
            0.101208,
            0.190316,
            0.146255,
            0.030995,
            0.951802,
            0.440253,
            0.255216,
            0.912456,
        ],
        "y": 9.9524647203994,
    },
}

print("Round 2 results loaded.")

Round 2 results loaded.


## 9. Update Datasets with Round 2

In [ ]:
for function_number in range(1, 9):
    new_x = np.array(
        round_2_results[function_number]["x"],
        dtype=float,
    ).reshape(1, -1)

    new_y = float(
        round_2_results[function_number]["y"]
    )

    expected_dimensions = datasets[function_number]["X"].shape[1]

    if new_x.shape[1] != expected_dimensions:
        raise ValueError(
            f"Function {function_number}: "
            f"expected {expected_dimensions} values, "
            f"received {new_x.shape[1]}."
        )

    datasets[function_number]["X"] = np.vstack(
        [
            datasets[function_number]["X"],
            new_x,
        ]
    )

    datasets[function_number]["y"] = np.append(
        datasets[function_number]["y"],
        new_y,
    )

    print(
        f"Function {function_number}: "
        f"inputs {datasets[function_number]['X'].shape}, "
        f"outputs {datasets[function_number]['y'].shape}"
    )

Function 1: inputs (12, 2), outputs (12,)
Function 2: inputs (12, 2), outputs (12,)
Function 3: inputs (17, 3), outputs (17,)
Function 4: inputs (32, 4), outputs (32,)
Function 5: inputs (22, 4), outputs (22,)
Function 6: inputs (22, 5), outputs (22,)
Function 7: inputs (32, 6), outputs (32,)
Function 8: inputs (42, 8), outputs (42,)


## 10. Check Data After Round 2

In [ ]:
for function_number in range(1, 9):
    X = datasets[function_number]["X"]
    y = datasets[function_number]["y"]

    assert X.shape[0] == y.shape[0]

    print(f"Function {function_number}")
    print("Latest input:", X[-1])
    print("Latest output:", y[-1])
    print("Current best:", np.max(y))
    print("Shape:", X.shape, y.shape)
    print()

Function 1
Latest input: [0.514232 0.063435]
Latest output: 1.370784784609222e-96
Current best: 7.710875114502849e-16
Shape: (12, 2) (12,)

Function 2
Latest input: [0.965674 0.216783]
Latest output: 0.09495924056129348
Current best: 0.6112052157614438
Shape: (12, 2) (12,)

Function 3
Latest input: [0.692232 0.468397 0.303034]
Latest output: -0.08132276509860875
Current best: -0.034835313350078584
Shape: (17, 3) (17,)

Function 4
Latest input: [0.378762 0.416287 0.401547 0.396799]
Latest output: 0.36846288672365235
Current best: 0.36846288672365235
Shape: (32, 4) (32,)

Function 5
Latest input: [0.536437 0.790238 0.999914 0.909757]
Latest output: 2006.076446872901
Current best: 2006.076446872901
Shape: (22, 4) (22,)

Function 6
Latest input: [0.414036 0.038018 0.331093 0.98082  0.002463]
Latest output: -0.9485322549641995
Current best: -0.3997806456044814
Shape: (22, 5) (22,)

Function 7
Latest input: [0.926472 0.634833 0.288693 0.796292 0.4421   0.743127]
Latest output: 0.016812595771

## 11. Expected Improvement

In [ ]:
def expected_improvement(
    X_candidates,
    model,
    best_y,
    xi=0.01,
):
    mean, standard_deviation = model.predict(
        X_candidates,
        return_std=True,
    )

    improvement = mean - best_y - xi

    with np.errstate(divide="ignore", invalid="ignore"):
        z = np.divide(
            improvement,
            standard_deviation,
            out=np.zeros_like(improvement),
            where=standard_deviation > 0,
        )

        ei = (
            improvement * norm.cdf(z)
            + standard_deviation * norm.pdf(z)
        )

    ei[standard_deviation == 0] = 0.0

    return ei

## 12. Gaussian Process Model

In [ ]:
def propose_next_point(
    X,
    y,
    n_candidates=200_000,
    xi=0.01,
    local_fraction=0.5,
    local_scale=0.10,
    random_state=42,
):
    dimensions = X.shape[1]

    kernel = (
        ConstantKernel(
            constant_value=1.0,
            constant_value_bounds=(1e-3, 1e3),
        )
        * Matern(
            length_scale=np.ones(dimensions),
            length_scale_bounds=(1e-3, 1e3),
            nu=2.5,
        )
        + WhiteKernel(
            noise_level=1e-6,
            noise_level_bounds=(1e-10, 1e-1),
        )
    )

    model = GaussianProcessRegressor(
        kernel=kernel,
        normalize_y=True,
        n_restarts_optimizer=10,
        random_state=random_state,
    )

    model.fit(X, y)

    rng = np.random.default_rng(random_state)

    # Best observed point
    best_index = np.argmax(y)
    best_x = X[best_index]

    # Split candidates into exploration and exploitation
    n_local = int(n_candidates * local_fraction)
    n_global = n_candidates - n_local

    # Global exploration
    global_candidates = rng.uniform(
        low=0.0,
        high=1.0,
        size=(n_global, dimensions),
    )

    # Local exploitation
    local_candidates = rng.normal(
        loc=best_x,
        scale=local_scale,
        size=(n_local, dimensions),
    )

    # Keep values inside the allowed range
    local_candidates = np.clip(
        local_candidates,
        0.0,
        1.0,
    )

    # Combine both candidate sets
    X_candidates = np.vstack(
        (
            global_candidates,
            local_candidates,
        )
    )

    # Score every candidate
    ei = expected_improvement(
        X_candidates=X_candidates,
        model=model,
        best_y=np.max(y),
        xi=xi,
    )

    best_candidate_index = np.argmax(ei)

    next_point = X_candidates[best_candidate_index]
    best_ei = ei[best_candidate_index]

    return next_point, model, best_ei

## 13. Round 3 Queries

In [ ]:
round_3_strategy = {
    1: {
        "xi": 0.05,
        "local_fraction": 0.20,
        "local_scale": 0.20,
    },
    2: {
        "xi": 0.05,
        "local_fraction": 0.20,
        "local_scale": 0.20,
    },
    3: {
        "xi": 0.05,
        "local_fraction": 0.20,
        "local_scale": 0.20,
    },
    4: {
        "xi": 0.001,
        "local_fraction": 0.70,
        "local_scale": 0.08,
    },
    5: {
        "xi": 0.001,
        "local_fraction": 0.70,
        "local_scale": 0.08,
    },
    6: {
        "xi": 0.01,
        "local_fraction": 0.50,
        "local_scale": 0.12,
    },
    7: {
        "xi": 0.05,
        "local_fraction": 0.20,
        "local_scale": 0.20,
    },
    8: {
        "xi": 0.001,
        "local_fraction": 0.70,
        "local_scale": 0.08,
    },
}

print("Round 3 strategy loaded.")

Round 3 strategy loaded.


## 14. Round 3 Queries

In [ ]:
round_3_queries = {}
models = {}
expected_improvements = {}

for function_number in range(1, 9):
    X = datasets[function_number]["X"]
    y = datasets[function_number]["y"]

    strategy = round_3_strategy[function_number]

    next_point, model, best_ei = propose_next_point(
        X=X,
        y=y,
        n_candidates=200_000,
        xi=strategy["xi"],
        local_fraction=strategy["local_fraction"],
        local_scale=strategy["local_scale"],
        random_state=300 + function_number,
    )

    round_3_queries[function_number] = next_point
    models[function_number] = model
    expected_improvements[function_number] = best_ei

    closest_distance = np.min(
        np.linalg.norm(
            X - next_point,
            axis=1,
        )
    )

    print(f"Function {function_number}")
    print("Strategy:", strategy)
    print("Query:", next_point)
    print("Expected Improvement:", best_ei)
    print("Closest existing distance:", closest_distance)
    print("Fitted kernel:", model.kernel_)
    print()

/usr/local/lib/python3.13/dist-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 1000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


Function 1
Strategy: {'xi': 0.05, 'local_fraction': 0.2, 'local_scale': 0.2}
Query: [0.40074066 0.20028589]
Expected Improvement: 0.0
Closest existing distance: 0.053615689751582075
Fitted kernel: 1.04**2 * Matern(length_scale=[0.0201, 1e+03], nu=2.5) + WhiteKernel(noise_level=4.13e-09)



/usr/local/lib/python3.13/dist-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 1000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


Function 2
Strategy: {'xi': 0.05, 'local_fraction': 0.2, 'local_scale': 0.2}
Query: [0.73874326 1.        ]
Expected Improvement: 0.007583861742973166
Closest existing distance: 0.08183221270198372
Fitted kernel: 0.956**2 * Matern(length_scale=[0.0598, 1e+03], nu=2.5) + WhiteKernel(noise_level=0.0176)



/usr/local/lib/python3.13/dist-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified upper bound 1000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 1000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


...

Function 5
Strategy: {'xi': 0.001, 'local_fraction': 0.7, 'local_scale': 0.08}
Query: [0.61680591 0.44089223 1.         1.        ]
Expected Improvement: 474.53044722457696
Closest existing distance: 0.36965582909240663
Fitted kernel: 6.86**2 * Matern(length_scale=[1e+03, 1e+03, 1.18, 1.52], nu=2.5) + WhiteKernel(noise_level=0.0027)

Function 6
Strategy: {'xi': 0.01, 'local_fraction': 0.5, 'local_scale': 0.12}
Query: [0.50409504 0.35727028 0.97768459 1.         0.06537132]
Expected Improvement: 0.06016346108028455
Closest existing distance: 0.3620405689695695
Fitted kernel: 1.46**2 * Matern(length_scale=[0.565, 0.817, 1.27, 1.3, 1.22], nu=2.5) + WhiteKernel(noise_level=1e-06)



/usr/local/lib/python3.13/dist-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 1000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 2 of parameter k1__k2__length_scale is close to the specified upper bound 1000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


Function 7
Strategy: {'xi': 0.05, 'local_fraction': 0.2, 'local_scale': 0.2}
Query: [0.         0.39648684 0.30899836 0.15236155 0.39071853 0.77684483]
Expected Improvement: 0.00937278596376042
Closest existing distance: 0.15334577056255766
Fitted kernel: 1.07**2 * Matern(length_scale=[3.93, 1e+03, 1e+03, 0.32, 0.162, 0.277], nu=2.5) + WhiteKernel(noise_level=1.49e-08)



/usr/local/lib/python3.13/dist-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 7 of parameter k1__k2__length_scale is close to the specified upper bound 1000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


Function 8
Strategy: {'xi': 0.001, 'local_fraction': 0.7, 'local_scale': 0.08}
Query: [0.06795938 0.050247   0.1911291  0.         1.         0.54115967
 0.21043593 0.85555711]
Expected Improvement: 0.03930513075502613
Closest existing distance: 0.20358563391235515
Fitted kernel: 6.56**2 * Matern(length_scale=[4.54, 6.57, 3.45, 8.04, 23.1, 6.35, 4.66, 1e+03], nu=2.5) + WhiteKernel(noise_level=1.02e-10)



### Function 1 Uncertainty Fallback

In [ ]:
function_number = 1

X = datasets[function_number]["X"]
y = datasets[function_number]["y"]

model = models[function_number]

rng = np.random.default_rng(901)

X_candidates = rng.uniform(
    low=0.0,
    high=1.0,
    size=(300_000, X.shape[1]),
)

predicted_mean, predicted_std = model.predict(
    X_candidates,
    return_std=True,
)

# Avoid points that are too close to observations already tested
distances = np.min(
    np.linalg.norm(
        X_candidates[:, None, :] - X[None, :, :],
        axis=2,
    ),
    axis=1,
)

valid_mask = distances >= 0.10

valid_candidates = X_candidates[valid_mask]
valid_std = predicted_std[valid_mask]

best_index = np.argmax(valid_std)
new_function_1_query = valid_candidates[best_index]

round_3_queries[1] = new_function_1_query
expected_improvements[1] = 0.0

print("New Function 1 query:", new_function_1_query)
print("Predicted uncertainty:", valid_std[best_index])
print(
    "Closest existing distance:",
    np.min(np.linalg.norm(X - new_function_1_query, axis=1)),
)

New Function 1 query: [0.99999947 0.2991063 ]
Predicted uncertainty: 0.0010319015911687541
Closest existing distance: 0.16330487752068834


## 15. Round 3 Submission Values

In [ ]:
print("ROUND 3 SUBMISSION VALUES\n")

round_3_submission_values = {}

for function_number in range(1, 9):
    submission_string = "-".join(
        f"{value:.6f}"
        for value in round_3_queries[function_number]
    )

    round_3_submission_values[function_number] = submission_string

    print(
        f"Function {function_number}: "
        f"{submission_string}"
    )

ROUND 3 SUBMISSION VALUES

Function 1: 0.999999-0.299106
Function 2: 0.738743-1.000000
Function 3: 0.957316-0.817132-0.855848
Function 4: 0.359522-0.428631-0.425310-0.436670
Function 5: 0.616806-0.440892-1.000000-1.000000
Function 6: 0.504095-0.357270-0.977685-1.000000-0.065371
Function 7: 0.000000-0.396487-0.308998-0.152362-0.390719-0.776845
Function 8: 0.067959-0.050247-0.191129-0.000000-1.000000-0.541160-0.210436-0.855557


### 16. Round 3 Summary

In [ ]:
summary_rows = []

for function_number in range(1, 9):
    summary_rows.append(
        {
            "Function": function_number,
            "Dimensions": datasets[function_number]["X"].shape[1],
            "Observations": datasets[function_number]["X"].shape[0],
            "Best Output": np.max(
                datasets[function_number]["y"]
            ),
            "Expected Improvement": expected_improvements[
                function_number
            ],
            "Round 3 Query": round_3_submission_values[
                function_number
            ],
        }
    )

round_3_summary_df = pd.DataFrame(summary_rows)

round_3_summary_df

,Function,Dimensions,Observations,Best Output,Expected Improvement,Round 3 Query
0,1,2,12,7.710875e-16,0.000000,0.999999-0.299106
1,2,2,12,6.112052e-01,0.007584,0.738743-1.000000
2,3,3,17,-3.483531e-02,0.006924,0.957316-0.817132-0.855848
3,4,4,32,3.684629e-01,0.063842,0.359522-0.428631-0.425310-0.436670
4,5,4,22,2.006076e+03,474.530447,0.616806-0.440892-1.000000-1.000000
5,6,5,22,-3.997806e-01,0.060163,0.504095-0.357270-0.977685-1.000000-0.065371
6,7,6,32,1.364968e+00,0.009373,0.000000-0.396487-0.308998-0.152362-0.390719-0...
7,8,8,42,9.952465e+00,0.039305,0.067959-0.050247-0.191129-0.000000-1.000000-0...


## 17. Save Round 3 Queries

In [ ]:
import json

ROUND_3_PATH = (
    "/content/drive/MyDrive/Imperial_BBO/data/week_3"
)

os.makedirs(ROUND_3_PATH, exist_ok=True)

round_3_json = {
    str(function_number): {
        "x": [
            float(value)
            for value in round_3_queries[function_number]
        ],
        "submission_value": round_3_submission_values[
            function_number
        ],
    }
    for function_number in range(1, 9)
}

save_path = os.path.join(
    ROUND_3_PATH,
    "submitted_queries.json",
)

with open(save_path, "w") as file:
    json.dump(
        round_3_json,
        file,
        indent=4,
    )

print("Saved to:")
print(save_path)

Saved to:
/content/drive/MyDrive/Imperial_BBO/data/week_3/submitted_queries.json


## 18. Round 3 Results

In [ ]:
round_3_results = {
    1: {
        "x": [0.999999, 0.299106],
        "y": -5.525818956858191e-172,
    },
    2: {
        "x": [0.738743, 1.000000],
        "y": 0.47517055974165556,
    },
    3: {
        "x": [0.957316, 0.817132, 0.855848],
        "y": -0.04391628945172519,
    },
    4: {
        "x": [0.359522, 0.428631, 0.425310, 0.436670],
        "y": 0.45751587994386655,
    },
    5: {
        "x": [0.616806, 0.440892, 1.000000, 1.000000],
        "y": 2049.1176324257544,
    },
    6: {
        "x": [
            0.504095,
            0.357270,
            0.977685,
            1.000000,
            0.065371,
        ],
        "y": -0.6376211539803871,
    },
    7: {
        "x": [
            0.000000,
            0.396487,
            0.308998,
            0.152362,
            0.390719,
            0.776845,
        ],
        "y": 1.479290443017275,
    },
    8: {
        "x": [
            0.067959,
            0.050247,
            0.191129,
            0.000000,
            1.000000,
            0.541160,
            0.210436,
            0.855557,
        ],
        "y": 9.9258429198891,
    },
}

print("Round 3 results loaded.")

Round 3 results loaded.


## 19. Update Datasets with Round 3 Results

In [ ]:
for function_number in range(1, 9):
    new_x = np.array(
        round_3_results[function_number]["x"],
        dtype=float,
    ).reshape(1, -1)

    new_y = float(
        round_3_results[function_number]["y"]
    )

    expected_dimensions = datasets[function_number]["X"].shape[1]

    if new_x.shape[1] != expected_dimensions:
        raise ValueError(
            f"Function {function_number}: "
            f"expected {expected_dimensions} dimensions, "
            f"received {new_x.shape[1]}."
        )

    datasets[function_number]["X"] = np.vstack(
        [
            datasets[function_number]["X"],
            new_x,
        ]
    )

    datasets[function_number]["y"] = np.append(
        datasets[function_number]["y"],
        new_y,
    )

    print(
        f"Function {function_number}: "
        f"{datasets[function_number]['X'].shape[0]} observations, "
        f"current best = {np.max(datasets[function_number]['y']):.6f}"
    )

Function 1: 13 observations, current best = 0.000000
Function 2: 13 observations, current best = 0.611205
Function 3: 18 observations, current best = -0.034835
Function 4: 33 observations, current best = 0.457516
Function 5: 23 observations, current best = 2049.117632
Function 6: 23 observations, current best = -0.399781
Function 7: 33 observations, current best = 1.479290
Function 8: 43 observations, current best = 9.952465


## 20. Round 3 Diagnostics

In [ ]:
diagnostic_rows = []

for function_number in range(1, 9):
    X = datasets[function_number]["X"]
    y = datasets[function_number]["y"]

    round_3_x = np.array(
        round_3_results[function_number]["x"],
        dtype=float,
    )

    round_3_y = float(
        round_3_results[function_number]["y"]
    )

    previous_best = np.max(y[:-1])
    current_best = np.max(y)

    best_index = np.argmax(y)
    best_x = X[best_index]

    distance_from_best = np.linalg.norm(
        round_3_x - best_x
    )

    diagnostic_rows.append(
        {
            "Function": function_number,
            "Dimensions": X.shape[1],
            "Observations": X.shape[0],
            "Previous Best": previous_best,
            "Round 3 Output": round_3_y,
            "Current Best": current_best,
            "Improved in Round 3": round_3_y > previous_best,
            "Distance from Current Best": distance_from_best,
        }
    )

round_3_diagnostics_df = pd.DataFrame(diagnostic_rows)

round_3_diagnostics_df

,Function,Dimensions,Observations,Previous Best,Round 3 Output,Current Best,Improved in Round 3,Distance from Current Best
0,1,2,13,7.710875e-16,-5.525819e-172,7.710875e-16,False,0.510501
1,2,2,13,6.112052e-01,4.751706e-01,6.112052e-01,False,0.081832
2,3,3,18,-3.483531e-02,-4.391629e-02,-3.483531e-02,False,0.723976
3,4,4,33,3.684629e-01,4.575159e-01,4.575159e-01,True,0.000000
4,5,4,23,2.006076e+03,2.049118e+03,2.049118e+03,True,0.000000
5,6,5,23,-3.997806e-01,-6.376212e-01,-3.997806e-01,False,0.362041
6,7,6,33,1.364968e+00,1.479290e+00,1.479290e+00,True,0.000000
7,8,8,43,9.952465e+00,9.925843e+00,9.952465e+00,False,0.203586


## 21. Round 4 Strategy

In [ ]:
round_4_strategy = {
    # F1: special uncertainty-driven exploration
    1: {
        "mode": "uncertainty",
        "xi": None,
        "local_fraction": 0.0,
        "local_scale": None,
        "min_distance": 0.10,
    },

    # F2: moderate exploration
    2: {
        "mode": "ei",
        "xi": 0.03,
        "local_fraction": 0.35,
        "local_scale": 0.15,
        "min_distance": 0.07,
    },

    # F3: stronger exploration
    3: {
        "mode": "ei",
        "xi": 0.05,
        "local_fraction": 0.20,
        "local_scale": 0.20,
        "min_distance": 0.10,
    },

    # F4: successful region -> exploit
    4: {
        "mode": "ei",
        "xi": 0.001,
        "local_fraction": 0.80,
        "local_scale": 0.06,
        "min_distance": 0.04,
    },

    # F5: successful region -> exploit
    5: {
        "mode": "ei",
        "xi": 0.001,
        "local_fraction": 0.80,
        "local_scale": 0.06,
        "min_distance": 0.05,
    },

    # F6: previous exploration failed -> continue broad search
    6: {
        "mode": "ei",
        "xi": 0.05,
        "local_fraction": 0.25,
        "local_scale": 0.18,
        "min_distance": 0.12,
    },

    # F7: new best -> exploit, but retain some global search
    7: {
        "mode": "ei",
        "xi": 0.005,
        "local_fraction": 0.70,
        "local_scale": 0.08,
        "min_distance": 0.10,
    },

    # F8: strong existing region, but Round 3 did not improve
    8: {
        "mode": "ei",
        "xi": 0.01,
        "local_fraction": 0.50,
        "local_scale": 0.10,
        "min_distance": 0.14,
    },
}

print("Round 4 strategy loaded.")

Round 4 strategy loaded.


## 22. Round 4 Optimiser

In [ ]:
def propose_round_4_point(
    X,
    y,
    mode="ei",
    xi=0.01,
    local_fraction=0.5,
    local_scale=0.10,
    min_distance=0.10,
    n_candidates=250_000,
    random_state=42,
):
    dimensions = X.shape[1]

    kernel = (
        ConstantKernel(
            constant_value=1.0,
            constant_value_bounds=(1e-3, 1e3),
        )
        * Matern(
            length_scale=np.ones(dimensions),
            length_scale_bounds=(1e-3, 1e4),
            nu=2.5,
        )
        + WhiteKernel(
            noise_level=1e-6,
            noise_level_bounds=(1e-10, 1e-1),
        )
    )

    model = GaussianProcessRegressor(
        kernel=kernel,
        normalize_y=True,
        n_restarts_optimizer=10,
        random_state=random_state,
    )

    model.fit(X, y)

    rng = np.random.default_rng(random_state)

    best_index = np.argmax(y)
    best_x = X[best_index]

    if mode == "uncertainty":
        X_candidates = rng.uniform(
            0.0,
            1.0,
            size=(n_candidates, dimensions),
        )

    else:
        n_local = int(n_candidates * local_fraction)
        n_global = n_candidates - n_local

        global_candidates = rng.uniform(
            0.0,
            1.0,
            size=(n_global, dimensions),
        )

        local_candidates = []

        while len(local_candidates) < n_local:
            batch = rng.normal(
                loc=best_x,
                scale=local_scale,
                size=(n_local, dimensions),
            )

            valid = np.all(
                (batch >= 0.0) & (batch <= 1.0),
                axis=1,
            )

            local_candidates.extend(
                batch[valid]
            )

        local_candidates = np.array(
            local_candidates[:n_local]
        )

        X_candidates = np.vstack(
            [
                global_candidates,
                local_candidates,
            ]
        )

    distances = np.min(
        np.linalg.norm(
            X_candidates[:, None, :] - X[None, :, :],
            axis=2,
        ),
        axis=1,
    )

    valid_mask = distances >= min_distance

    X_candidates = X_candidates[valid_mask]
    distances = distances[valid_mask]

    if len(X_candidates) == 0:
        raise ValueError(
            "No valid candidates remain after distance filtering."
        )

    mean, std = model.predict(
        X_candidates,
        return_std=True,
    )

    if mode == "uncertainty":
        scores = std

    else:
        scores = expected_improvement(
            X_candidates=X_candidates,
            model=model,
            best_y=np.max(y),
            xi=xi,
        )

        if np.max(scores) <= 0:
            scores = std

    best_candidate_index = np.argmax(scores)

    next_point = X_candidates[best_candidate_index]

    return {
        "query": next_point,
        "score": scores[best_candidate_index],
        "distance": distances[best_candidate_index],
        "model": model,
        "mean": mean[best_candidate_index],
        "std": std[best_candidate_index],
    }

## 23. Generate Round 4 Queries

In [ ]:
round_4_queries = {}
round_4_results = {}

for function_number in range(1, 9):
    X = datasets[function_number]["X"]
    y = datasets[function_number]["y"]

    strategy = round_4_strategy[function_number]

    result = propose_round_4_point(
        X=X,
        y=y,
        mode=strategy["mode"],
        xi=0.01 if strategy["xi"] is None else strategy["xi"],
        local_fraction=strategy["local_fraction"],
        local_scale=0.10 if strategy["local_scale"] is None else strategy["local_scale"],
        min_distance=strategy["min_distance"],
        n_candidates=250_000,
        random_state=400 + function_number,
    )

    round_4_queries[function_number] = result["query"]
    round_4_results[function_number] = result

    print(f"Function {function_number}")
    print("Query:", result["query"])
    print("Score:", result["score"])
    print("Predicted mean:", result["mean"])
    print("Predicted std:", result["std"])
    print("Closest existing distance:", result["distance"])
    print("Fitted kernel:", result["model"].kernel_)
    print()

Function 1
Query: [0.38884018 0.99999032]
Score: 0.00099862875684854
Predicted mean: -0.0002758710726090183
Predicted std: 0.00099862875684854
Closest existing distance: 0.2209713120308991
Fitted kernel: 1.04**2 * Matern(length_scale=[137, 0.0319], nu=2.5) + WhiteKernel(noise_level=5.89e-06)



/usr/local/lib/python3.13/dist-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 10000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


Function 2
Query: [1.29492149e-06 9.90397900e-01]
Score: 0.002936289284359829
Predicted mean: 0.22952174916091383
Predicted std: 0.22441897825243853
Closest existing distance: 0.5340834102843446
Fitted kernel: 0.974**2 * Matern(length_scale=[0.0668, 1e+04], nu=2.5) + WhiteKernel(noise_level=0.0176)



/usr/local/lib/python3.13/dist-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified upper bound 10000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


Function 3
Query: [0.68565715 0.79406022 0.09249402]
Score: 0.005422764179025288
Predicted mean: -0.0739339755387429
Predicted std: 0.08037464744911633
Closest existing distance: 0.11299699984644812
Fitted kernel: 1.11**2 * Matern(length_scale=[1e+04, 3.69e+03, 0.0229], nu=2.5) + WhiteKernel(noise_level=8.58e-09)

Function 4
Query: [0.34366551 0.44551672 0.45357941 0.42028674]
Score: 0.1099549430562617
Predicted mean: 0.46787844650254407
Predicted std: 0.26371575239997025
Closest existing distance: 0.040051550732191486
Fitted kernel: 2.44**2 * Matern(length_scale=[1.47, 1.21, 1.26, 1.29], nu=2.5) + WhiteKernel(noise_level=0.000327)



...

Function 5
Query: [0.59236772 0.38630042 0.99990354 0.99998561]
Score: 63.12713418680385
Predicted mean: 2097.062302078168
Predicted std: 84.9695336712676
Closest existing distance: 0.05981203795133585
Fitted kernel: 2.07**2 * Matern(length_scale=[6.26e+03, 1e+04, 0.681, 0.878], nu=2.5) + WhiteKernel(noise_level=0.0109)

Function 6
Query: [0.43310895 0.45930461 0.50432728 0.97432518 0.01885052]
Score: 0.026599875437069553
Predicted mean: -0.42760598225062174
Predicted std: 0.14363235208996378
Closest existing distance: 0.23854410459028152
Fitted kernel: 1.46**2 * Matern(length_scale=[0.677, 0.755, 1.13, 1.29, 1.41], nu=2.5) + WhiteKernel(noise_level=1e-06)



/usr/local/lib/python3.13/dist-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 2 of parameter k1__k2__length_scale is close to the specified upper bound 10000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


Function 7
Query: [0.00356188 0.43002423 0.36146591 0.19322616 0.38764743 0.84611758]
Score: 0.014802701262841892
Predicted mean: 1.414160570144078
Predicted std: 0.10174332109616462
Closest existing distance: 0.10182494485097769
Fitted kernel: 0.934**2 * Matern(length_scale=[1.96, 9.73e+03, 1e+04, 0.383, 0.186, 0.301], nu=2.5) + WhiteKernel(noise_level=3.79e-08)



/usr/local/lib/python3.13/dist-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 7 of parameter k1__k2__length_scale is close to the specified upper bound 10000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/gaussian_process/kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-10. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


Function 8
Query: [0.12812325 0.18107801 0.09643291 0.19905344 0.99728799 0.48156462
 0.22061085 0.85931809]
Score: 0.018850674630706228
Predicted mean: 9.976441049599647
Predicted std: 0.026076591851711604
Closest existing distance: 0.1983240576899431
Fitted kernel: 6.49**2 * Matern(length_scale=[4.54, 6.63, 3.59, 7.65, 20.9, 6.8, 4.66, 1e+04], nu=2.5) + WhiteKernel(noise_level=1e-10)



## 24. Round 4 Submission Values

In [ ]:
round_4_submission_values = {}

print("ROUND 4 SUBMISSION VALUES\n")

for function_number in range(1, 9):
    submission_string = "-".join(
        f"{value:.6f}"
        for value in round_4_queries[function_number]
    )

    round_4_submission_values[function_number] = submission_string

    print(
        f"Function {function_number}: "
        f"{submission_string}"
    )

ROUND 4 SUBMISSION VALUES

Function 1: 0.388840-0.999990
Function 2: 0.000001-0.990398
Function 3: 0.685657-0.794060-0.092494
Function 4: 0.343666-0.445517-0.453579-0.420287
Function 5: 0.592368-0.386300-0.999904-0.999986
Function 6: 0.433109-0.459305-0.504327-0.974325-0.018851
Function 7: 0.003562-0.430024-0.361466-0.193226-0.387647-0.846118
Function 8: 0.128123-0.181078-0.096433-0.199053-0.997288-0.481565-0.220611-0.859318


## 25. Save Round 4 Queries

In [ ]:
import json

ROUND_4_PATH = (
    "/content/drive/MyDrive/Imperial_BBO/data/week_4"
)

os.makedirs(ROUND_4_PATH, exist_ok=True)

round_4_json = {
    str(function_number): {
        "x": [
            float(value)
            for value in round_4_queries[function_number]
        ],
        "submission_value": round_4_submission_values[
            function_number
        ],
    }
    for function_number in range(1, 9)
}

save_path = os.path.join(
    ROUND_4_PATH,
    "submitted_queries.json",
)

with open(save_path, "w") as file:
    json.dump(
        round_4_json,
        file,
        indent=4,
    )

print("Saved to:")
print(save_path)

Saved to:
/content/drive/MyDrive/Imperial_BBO/data/week_4/submitted_queries.json


## 26. Round 4 Results

In [ ]:
round_4_results = {
    1: {
        "x": [0.388840, 0.999990],
        "y": 2.0665168552572823e-136,
    },
    2: {
        "x": [0.000001, 0.990398],
        "y": 0.02921568688137048,
    },
    3: {
        "x": [0.685657, 0.794060, 0.092494],
        "y": -0.06530888519597104,
    },
    4: {
        "x": [0.343666, 0.445517, 0.453579, 0.420287],
        "y": -0.22454422776275917,
    },
    5: {
        "x": [0.592368, 0.386300, 0.999904, 0.999986],
        "y": 1957.49526698959,
    },
    6: {
        "x": [
            0.433109,
            0.459305,
            0.504327,
            0.974325,
            0.018851,
        ],
        "y": -0.5753911885392136,
    },
    7: {
        "x": [
            0.003562,
            0.430024,
            0.361466,
            0.193226,
            0.387647,
            0.846118,
        ],
        "y": 1.3492630003784043,
    },
    8: {
        "x": [
            0.128123,
            0.181078,
            0.096433,
            0.199053,
            0.997288,
            0.481565,
            0.220611,
            0.859318,
        ],
        "y": 9.9642905885306,
    },
}

print("Round 4 results loaded.")

Round 4 results loaded.


## 27. Update Datasets with Round 4 Results

In [ ]:
for function_number in range(1, 9):
    new_x = np.array(
        round_4_results[function_number]["x"],
        dtype=float,
    ).reshape(1, -1)

    new_y = float(
        round_4_results[function_number]["y"]
    )

    expected_dimensions = datasets[function_number]["X"].shape[1]

    if new_x.shape[1] != expected_dimensions:
        raise ValueError(
            f"Function {function_number}: "
            f"expected {expected_dimensions} dimensions, "
            f"received {new_x.shape[1]}."
        )

    datasets[function_number]["X"] = np.vstack(
        [
            datasets[function_number]["X"],
            new_x,
        ]
    )

    datasets[function_number]["y"] = np.append(
        datasets[function_number]["y"],
        new_y,
    )

    print(
        f"Function {function_number}: "
        f"{datasets[function_number]['X'].shape[0]} observations, "
        f"current best = {np.max(datasets[function_number]['y']):.6f}"
    )

Function 1: 14 observations, current best = 0.000000
Function 2: 14 observations, current best = 0.611205
Function 3: 19 observations, current best = -0.034835
Function 4: 34 observations, current best = 0.457516
Function 5: 24 observations, current best = 2049.117632
Function 6: 24 observations, current best = -0.399781
Function 7: 34 observations, current best = 1.479290
Function 8: 44 observations, current best = 9.964291


## 28. Round 4 Diagnostics

In [ ]:
diagnostic_rows = []

for function_number in range(1, 9):
    X = datasets[function_number]["X"]
    y = datasets[function_number]["y"]

    round_4_x = np.array(
        round_4_results[function_number]["x"],
        dtype=float,
    )

    round_4_y = float(
        round_4_results[function_number]["y"]
    )

    previous_best = np.max(y[:-1])
    current_best = np.max(y)

    best_index = np.argmax(y)
    best_x = X[best_index]

    distance_from_best = np.linalg.norm(
        round_4_x - best_x
    )

    diagnostic_rows.append(
        {
            "Function": function_number,
            "Dimensions": X.shape[1],
            "Observations": X.shape[0],
            "Previous Best": previous_best,
            "Round 4 Output": round_4_y,
            "Current Best": current_best,
            "Improved in Round 4": round_4_y > previous_best,
            "Distance from Current Best": distance_from_best,
        }
    )

round_4_diagnostics_df = pd.DataFrame(diagnostic_rows)

round_4_diagnostics_df

,Function,Dimensions,Observations,Previous Best,Round 4 Output,Current Best,Improved in Round 4,Distance from Current Best
0,1,2,14,7.710875e-16,2.066517e-136,7.710875e-16,False,0.434020
1,2,2,14,6.112052e-01,2.921569e-02,6.112052e-01,False,0.705529
2,3,3,19,-3.483531e-02,-6.530889e-02,-3.483531e-02,False,0.363206
3,4,4,34,4.575159e-01,-2.245442e-01,4.575159e-01,False,0.040051
4,5,4,24,2.049118e+03,1.957495e+03,2.049118e+03,False,0.059812
5,6,5,24,-3.997806e-01,-5.753912e-01,-3.997806e-01,False,0.238545
6,7,6,34,1.479290e+00,1.349263e+00,1.479290e+00,False,0.101825
7,8,8,44,9.952465e+00,9.964291e+00,9.964291e+00,True,0.000000


## 29. Round 5 Strategy

In [ ]:
round_5_strategy = {
    1: {
        "local_fraction": 0.20,
        "local_scale": 0.20,
        "min_distance": 0.05,
        "beta": 2.5,
    },

    2: {
        "local_fraction": 0.20,
        "local_scale": 0.20,
        "min_distance": 0.04,
        "beta": 2.5,
    },

    3: {
        "local_fraction": 0.30,
        "local_scale": 0.18,
        "min_distance": 0.05,
        "beta": 2.2,
    },

    4: {
        "local_fraction": 0.40,
        "local_scale": 0.12,
        "min_distance": 0.04,
        "beta": 2.0,
    },

    5: {
        "local_fraction": 0.45,
        "local_scale": 0.10,
        "min_distance": 0.05,
        "beta": 2.0,
    },

    6: {
        "local_fraction": 0.30,
        "local_scale": 0.16,
        "min_distance": 0.08,
        "beta": 2.4,
    },

    7: {
        "local_fraction": 0.40,
        "local_scale": 0.12,
        "min_distance": 0.10,
        "beta": 2.2,
    },

    8: {
        "local_fraction": 0.65,
        "local_scale": 0.08,
        "min_distance": 0.12,
        "beta": 1.8,
    },
}

print("Round 5 strategy loaded.")

Round 5 strategy loaded.


## 30. Round 5 Candidate Generator

In [ ]:
def generate_round_5_candidates(
    X,
    y,
    local_fraction=0.5,
    local_scale=0.10,
    min_distance=0.05,
    beta=2.0,
    n_candidates=300_000,
    random_state=42,
):
    dimensions = X.shape[1]

    kernel = (
        ConstantKernel(
            constant_value=1.0,
            constant_value_bounds=(1e-3, 1e3),
        )
        * Matern(
            length_scale=np.ones(dimensions),
            length_scale_bounds=(1e-3, 1e4),
            nu=2.5,
        )
        + WhiteKernel(
            noise_level=1e-6,
            noise_level_bounds=(1e-10, 1e-1),
        )
    )

    model = GaussianProcessRegressor(
        kernel=kernel,
        normalize_y=True,
        n_restarts_optimizer=10,
        random_state=random_state,
    )

    model.fit(X, y)

    rng = np.random.default_rng(random_state)

    best_index = np.argmax(y)
    best_x = X[best_index]
    best_y = np.max(y)

    n_local = int(n_candidates * local_fraction)
    n_global = n_candidates - n_local

    global_candidates = rng.uniform(
        low=0.0,
        high=0.999999,
        size=(n_global, dimensions),
    )

    local_candidates = []

    while len(local_candidates) < n_local:
        batch = rng.normal(
            loc=best_x,
            scale=local_scale,
            size=(n_local, dimensions),
        )

        valid = np.all(
            (batch >= 0.0) & (batch < 1.0),
            axis=1,
        )

        local_candidates.extend(
            batch[valid]
        )

    local_candidates = np.array(
        local_candidates[:n_local]
    )

    X_candidates = np.vstack(
        [
            global_candidates,
            local_candidates,
        ]
    )

    distances = np.min(
        np.linalg.norm(
            X_candidates[:, None, :] - X[None, :, :],
            axis=2,
        ),
        axis=1,
    )

    valid_mask = distances >= min_distance

    X_candidates = X_candidates[valid_mask]
    distances = distances[valid_mask]

    if len(X_candidates) == 0:
        raise ValueError(
            "No valid candidates remain after distance filtering."
        )

    mean, std = model.predict(
        X_candidates,
        return_std=True,
    )

    ei = expected_improvement(
        X_candidates=X_candidates,
        model=model,
        best_y=best_y,
        xi=0.01,
    )

    ucb = mean + beta * std

    uncertainty = std

    exploitation = mean

    acquisition_scores = {
        "EI": ei,
        "UCB": ucb,
        "Uncertainty": uncertainty,
        "Mean": exploitation,
    }

    candidates = {}

    for name, scores in acquisition_scores.items():
        index = np.argmax(scores)

        candidates[name] = {
            "query": X_candidates[index],
            "mean": mean[index],
            "std": std[index],
            "ei": ei[index],
            "ucb": ucb[index],
            "distance": distances[index],
        }

    return {
        "model": model,
        "candidates": candidates,
    }

## 31. Generate Round 5 Candidate Sets

In [ ]:
round_5_candidate_sets = {}

for function_number in range(1, 9):
    X = datasets[function_number]["X"]
    y = datasets[function_number]["y"]

    strategy = round_5_strategy[function_number]

    result = generate_round_5_candidates(
        X=X,
        y=y,
        local_fraction=strategy["local_fraction"],
        local_scale=strategy["local_scale"],
        min_distance=strategy["min_distance"],
        beta=strategy["beta"],
        n_candidates=300_000,
        random_state=500 + function_number,
    )

    round_5_candidate_sets[function_number] = result

    print(f"Function {function_number}")
    print("Fitted kernel:", result["model"].kernel_)

    for name, candidate in result["candidates"].items():
        print(f"\n{name}")
        print("Query:", candidate["query"])
        print("Predicted mean:", candidate["mean"])
        print("Predicted std:", candidate["std"])
        print("EI:", candidate["ei"])
        print("UCB:", candidate["ucb"])
        print("Closest existing distance:", candidate["distance"])

    print("\n" + "-" * 60 + "\n")

Function 1
Fitted kernel: 1.04**2 * Matern(length_scale=[0.0207, 2.15e+03], nu=2.5) + WhiteKernel(noise_level=2.36e-10)

EI
Query: [0.14594254 0.0097746 ]
Predicted mean: -0.00025116714811665095
Predicted std: 0.0009643622938317528
EI: 9.63709151195452e-31
UCB: 0.0021597385864627313
Closest existing distance: 0.18044099387427653

UCB
Query: [0.46633645 0.61743529]
Predicted mean: -0.0002269716656646106
Predicted std: 0.0009599529550787891
EI: 7.4285445439409725e-31
UCB: 0.002172910722032362
Closest existing distance: 0.1946326671622407

Uncertainty
Query: [0.19576873 0.73843196]
Predicted mean: -0.0002574510611169265
Predicted std: 0.0009646608061415277
EI: 9.309671518148166e-31
UCB: 0.002154200954236893
Closest existing distance: 0.12604462051179344

Mean
Query: [0.69458061 0.59324215]
Predicted mean: 0.0002846762429035271
Predicted std: 0.0005516731970269172
EI: 3.1763333749939265e-74
UCB: 0.00166385923547082
Closest existing distance: 0.09885026443549841

---------------------------

/usr/local/lib/python3.13/dist-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified upper bound 10000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


Function 3
Fitted kernel: 1.48**2 * Matern(length_scale=[1e+04, 658, 0.0853], nu=2.5) + WhiteKernel(noise_level=0.0168)

EI
Query: [0.25110879 0.95654293 0.3979069 ]
Predicted mean: -0.036970457770061235
Predicted std: 0.05020685015792979
EI: 0.014544300183820264
UCB: 0.07348461257738431
Closest existing distance: 0.16021245908094228

UCB
Query: [0.94565474 0.06037024 0.65511789]
Predicted mean: -0.09588515293566129
Predicted std: 0.09423566331919876
EI: 0.012276270825591025
UCB: 0.11143330636657602
Closest existing distance: 0.5229547920377684

Uncertainty
Query: [0.23662722 0.01578733 0.66776376]
Predicted mean: -0.10202247121924803
Predicted std: 0.09565650725485525
EI: 0.011359028302460324
UCB: 0.10842184474143353
Closest existing distance: 0.3050927163940679

Mean
Query: [0.65587888 0.93289402 0.37136285]
Predicted mean: -0.03370920580222635
Predicted std: 0.03426031305730761
EI: 0.009686871575823265
UCB: 0.0416634829238504
Closest existing distance: 0.2895134559569185

----------

/usr/local/lib/python3.13/dist-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified upper bound 10000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 10000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


Function 5
Fitted kernel: 1.61**2 * Matern(length_scale=[1e+04, 1e+04, 0.642, 0.778], nu=2.5) + WhiteKernel(noise_level=0.011)

EI
Query: [0.68472985 0.24770103 0.99989329 0.99991817]
Predicted mean: 2032.7927984422956
Predicted std: 85.00170314004312
EI: 26.36759147739945
UCB: 2202.796204722382
Closest existing distance: 0.1665544643132645

UCB
Query: [0.68472985 0.24770103 0.99989329 0.99991817]
Predicted mean: 2032.7927984422956
Predicted std: 85.00170314004312
EI: 26.36759147739945
UCB: 2202.796204722382
Closest existing distance: 0.1665544643132645

Uncertainty
Query: [0.00809962 0.86771887 0.99927323 0.00174666]
Predicted mean: 185.54850105207237
Predicted std: 405.47034941058075
EI: 0.00017511991440031204
UCB: 996.4891998732339
Closest existing distance: 0.5839227551352687

Mean
Query: [0.68472985 0.24770103 0.99989329 0.99991817]
Predicted mean: 2032.7927984422956
Predicted std: 85.00170314004312
EI: 26.36759147739945
UCB: 2202.796204722382
Closest existing distance: 0.16655446

/usr/local/lib/python3.13/dist-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 2 of parameter k1__k2__length_scale is close to the specified upper bound 10000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


Function 7
Fitted kernel: 0.884**2 * Matern(length_scale=[1.35, 0.835, 1e+04, 0.589, 0.239, 0.379], nu=2.5) + WhiteKernel(noise_level=9.99e-07)

EI
Query: [0.00957446 0.38783356 0.17736232 0.05578326 0.38169955 0.71896328]
Predicted mean: 1.496884655891067
Predicted std: 0.06507793704774723
EI: 0.029936018049090723
UCB: 1.6400561173961108
Closest existing distance: 0.17393562353197572

UCB
Query: [0.00812109 0.54940062 0.3343497  0.00305086 0.35452248 0.71652926]
Predicted mean: 1.392737985431578
Predicted std: 0.14172729811462836
EI: 0.020900529796982103
UCB: 1.7045380412837603
Closest existing distance: 0.22656837127363946

Uncertainty
Query: [0.03803383 0.98811267 0.18806554 0.01931023 0.99163969 0.0108929 ]
Predicted mean: 0.23241449501001965
Predicted std: 0.33690166200450045
EI: 7.683200887514537e-06
UCB: 0.9735981514199207
Closest existing distance: 0.7425085832071187

Mean
Query: [0.00442619 0.34480756 0.44566734 0.11082414 0.38200967 0.76000058]
Predicted mean: 1.5063589278745

/usr/local/lib/python3.13/dist-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 7 of parameter k1__k2__length_scale is close to the specified upper bound 10000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


Function 8
Fitted kernel: 6.4**2 * Matern(length_scale=[4.62, 6.62, 3.61, 8.03, 20.5, 6.85, 4.73, 1e+04], nu=2.5) + WhiteKernel(noise_level=1.67e-09)

EI
Query: [0.15447186 0.15868285 0.0208147  0.00212351 0.99015771 0.34472196
 0.19303431 0.90596104]
Predicted mean: 9.935917298045052
Predicted std: 0.05436556915203066
EI: 0.0076912101977167305
UCB: 10.033775322518707
Closest existing distance: 0.18684917089477393

UCB
Query: [0.15222295 0.05752497 0.0112999  0.0472053  0.96840214 0.30923083
 0.05334662 0.78650402]
Predicted mean: 9.879207138994937
Predicted std: 0.09214040583378677
EI: 0.007221043879098211
UCB: 10.045059869495754
Closest existing distance: 0.33160855648081905

Uncertainty
Query: [0.97899868 0.01875331 0.95638809 0.7290911  0.92841261 0.03199917
 0.02193771 0.09332003]
Predicted mean: 6.188584406942242
Predicted std: 0.3047371551512517
EI: 2.3788583058986813e-37
UCB: 6.737111286214495
Closest existing distance: 1.0861035519316717

Mean
Query: [0.1167475  0.22272474 0.0

## 32. Function 4 Diversified Candidates

In [ ]:
X = datasets[4]["X"]
y = datasets[4]["y"]

f4_diversified = generate_round_5_candidates(
    X=X,
    y=y,
    local_fraction=0.20,
    local_scale=0.18,
    min_distance=0.12,
    beta=2.2,
    n_candidates=300_000,
    random_state=604,
)

print("Function 4 diversified candidates")
print("Fitted kernel:", f4_diversified["model"].kernel_)

for name, candidate in f4_diversified["candidates"].items():
    print(f"\n{name}")
    print("Query:", candidate["query"])
    print("Predicted mean:", candidate["mean"])
    print("Predicted std:", candidate["std"])
    print("EI:", candidate["ei"])
    print("UCB:", candidate["ucb"])
    print("Closest existing distance:", candidate["distance"])

Function 4 diversified candidates
Fitted kernel: 2.72**2 * Matern(length_scale=[1.59, 1.37, 1.45, 1.44], nu=2.5) + WhiteKernel(noise_level=0.00131)

EI
Query: [0.35288772 0.44516543 0.30237609 0.45761945]
Predicted mean: -0.7515445139762935
Predicted std: 0.6497054536194748
EI: 0.0076358571539745165
UCB: 0.6778074839865513
Closest existing distance: 0.12262723315149486

UCB
Query: [0.35288772 0.44516543 0.30237609 0.45761945]
Predicted mean: -0.7515445139762935
Predicted std: 0.6497054536194748
EI: 0.0076358571539745165
UCB: 0.6778074839865513
Closest existing distance: 0.12262723315149486

Uncertainty
Query: [0.00605657 0.181593   0.99939269 0.99474286]
Predicted mean: -26.972334866950245
Predicted std: 6.6184958299574355
EI: 2.452481588846253e-05
UCB: -12.411644041043886
Closest existing distance: 0.5913632864051291

Mean
Query: [0.42110072 0.4457389  0.34322709 0.50069237]
Predicted mean: -0.7027330457456014
Predicted std: 0.5775408839642547
EI: 0.004569239328134654
UCB: 0.567856898

## 33. Function 4 Balanced Candidates

In [ ]:
X = datasets[4]["X"]
y = datasets[4]["y"]

f4_balanced = generate_round_5_candidates(
    X=X,
    y=y,
    local_fraction=0.30,
    local_scale=0.15,
    min_distance=0.08,
    beta=2.1,
    n_candidates=300_000,
    random_state=704,
)

print("Function 4 balanced candidates")
print("Fitted kernel:", f4_balanced["model"].kernel_)

for name, candidate in f4_balanced["candidates"].items():
    print(f"\n{name}")
    print("Query:", candidate["query"])
    print("Predicted mean:", candidate["mean"])
    print("Predicted std:", candidate["std"])
    print("EI:", candidate["ei"])
    print("UCB:", candidate["ucb"])
    print("Closest existing distance:", candidate["distance"])

Function 4 balanced candidates
Fitted kernel: 2.72**2 * Matern(length_scale=[1.59, 1.37, 1.45, 1.44], nu=2.5) + WhiteKernel(noise_level=0.00131)

EI
Query: [0.37144384 0.45442929 0.34593106 0.45904661]
Predicted mean: -0.25522622319740584
Predicted std: 0.5175409662293405
EI: 0.01912433692939127
UCB: 0.8316098058842094
Closest existing distance: 0.08723193846996465

UCB
Query: [0.37144384 0.45442929 0.34593106 0.45904661]
Predicted mean: -0.25522622319740584
Predicted std: 0.5175409662293405
EI: 0.01912433692939127
UCB: 0.8316098058842094
Closest existing distance: 0.08723193846996465

Uncertainty
Query: [0.01775238 0.17725395 0.99916615 0.98018687]
Predicted mean: -26.683020802963494
Predicted std: 6.417522542358375
EI: 1.6094912351662127e-05
UCB: -13.206223464010906
Closest existing distance: 0.5739000784736701

Mean
Query: [0.4052641  0.39248817 0.37719985 0.46458766]
Predicted mean: -0.16798886711630523
Predicted std: 0.45255534862668034
EI: 0.016439338330413555
UCB: 0.782377364999

## 34. Round 5 Submission Values

In [ ]:
round_5_queries = {
    1: np.array([0.46633645, 0.61743529]),
    2: np.array([0.69693232, 0.62970224]),
    3: np.array([0.94565474, 0.06037024, 0.65511789]),
    4: np.array([0.37144384, 0.45442929, 0.34593106, 0.45904661]),
    5: np.array([0.68472985, 0.24770103, 0.99989329, 0.99991817]),
    6: np.array([0.58174862, 0.32379123, 0.58472237, 0.99701867, 0.00326090]),
    7: np.array([0.00957446, 0.38783356, 0.17736232, 0.05578326, 0.38169955, 0.71896328]),
    8: np.array([0.11674750, 0.22272474, 0.09007611, 0.08553602, 0.99738531, 0.49787580, 0.19874271, 0.81573130]),
}

round_5_submission_values = {}

print("ROUND 5 SUBMISSION VALUES\n")

for function_number in range(1, 9):
    submission_string = "-".join(
        f"{value:.6f}"
        for value in round_5_queries[function_number]
    )

    round_5_submission_values[function_number] = submission_string

    print(
        f"Function {function_number}: "
        f"{submission_string}"
    )

ROUND 5 SUBMISSION VALUES

Function 1: 0.466336-0.617435
Function 2: 0.696932-0.629702
Function 3: 0.945655-0.060370-0.655118
Function 4: 0.371444-0.454429-0.345931-0.459047
Function 5: 0.684730-0.247701-0.999893-0.999918
Function 6: 0.581749-0.323791-0.584722-0.997019-0.003261
Function 7: 0.009574-0.387834-0.177362-0.055783-0.381700-0.718963
Function 8: 0.116748-0.222725-0.090076-0.085536-0.997385-0.497876-0.198743-0.815731


## 35. Save Round 5 Queries

In [ ]:
import os
import json

week_5_dir = "/content/drive/MyDrive/Imperial_BBO/data/week_5"

os.makedirs(
    week_5_dir,
    exist_ok=True,
)

round_5_queries_to_save = {
    f"Function {function_number}": [
        round(float(value), 6)
        for value in round_5_queries[function_number]
    ]
    for function_number in range(1, 9)
}

save_path = os.path.join(
    week_5_dir,
    "submitted_queries.json",
)

with open(save_path, "w") as file:
    json.dump(
        round_5_queries_to_save,
        file,
        indent=4,
    )

print(f"Saved to: {save_path}")

Saved to: /content/drive/MyDrive/Imperial_BBO/data/week_5/submitted_queries.json


## 36. Round 5 Results

In [ ]:
round_5_results = {
    1: {
        "x": [0.466336, 0.617435],
        "y": 7.58064335600162e-19,
    },
    2: {
        "x": [0.696932, 0.629702],
        "y": 0.5140985605502743,
    },
    3: {
        "x": [0.945655, 0.060370, 0.655118],
        "y": -0.1684900356287598,
    },
    4: {
        "x": [0.371444, 0.454429, 0.345931, 0.459047],
        "y": -0.6661602804925972,
    },
    5: {
        "x": [0.684730, 0.247701, 0.999893, 0.999918],
        "y": 2142.4534883778606,
    },
    6: {
        "x": [
            0.581749,
            0.323791,
            0.584722,
            0.997019,
            0.003261,
        ],
        "y": -0.5996891250147741,
    },
    7: {
        "x": [
            0.009574,
            0.387834,
            0.177362,
            0.055783,
            0.381700,
            0.718963,
        ],
        "y": 1.063849901140438,
    },
    8: {
        "x": [
            0.116748,
            0.222725,
            0.090076,
            0.085536,
            0.997385,
            0.497876,
            0.198743,
            0.815731,
        ],
        "y": 9.9610706217204,
    },
}

print("Round 5 results loaded.")

Round 5 results loaded.


## 37. Update Datasets with Round 5 Results

In [ ]:
for function_number in range(1, 9):
    new_x = np.array(
        round_5_results[function_number]["x"],
        dtype=float,
    ).reshape(1, -1)

    new_y = float(
        round_5_results[function_number]["y"]
    )

    expected_dimensions = datasets[function_number]["X"].shape[1]

    if new_x.shape[1] != expected_dimensions:
        raise ValueError(
            f"Function {function_number}: "
            f"expected {expected_dimensions} dimensions, "
            f"received {new_x.shape[1]}."
        )

    datasets[function_number]["X"] = np.vstack(
        [
            datasets[function_number]["X"],
            new_x,
        ]
    )

    datasets[function_number]["y"] = np.append(
        datasets[function_number]["y"],
        new_y,
    )

    print(
        f"Function {function_number}: "
        f"{datasets[function_number]['X'].shape[0]} observations, "
        f"current best = {np.max(datasets[function_number]['y']):.6f}"
    )

Function 1: 15 observations, current best = 0.000000
Function 2: 15 observations, current best = 0.611205
Function 3: 20 observations, current best = -0.034835
Function 4: 35 observations, current best = 0.457516
Function 5: 25 observations, current best = 2142.453488
Function 6: 25 observations, current best = -0.399781
Function 7: 35 observations, current best = 1.479290
Function 8: 45 observations, current best = 9.964291


## 38. Round 5 Diagnostics

In [ ]:
diagnostic_rows = []

for function_number in range(1, 9):
    X = datasets[function_number]["X"]
    y = datasets[function_number]["y"]

    round_5_x = np.array(
        round_5_results[function_number]["x"],
        dtype=float,
    )

    round_5_y = float(
        round_5_results[function_number]["y"]
    )

    previous_best = np.max(y[:-1])
    current_best = np.max(y)

    best_index = np.argmax(y)
    best_x = X[best_index]

    distance_from_best = np.linalg.norm(
        round_5_x - best_x
    )

    diagnostic_rows.append(
        {
            "Function": function_number,
            "Dimensions": X.shape[1],
            "Observations": X.shape[0],
            "Previous Best": previous_best,
            "Round 5 Output": round_5_y,
            "Current Best": current_best,
            "Improved in Round 5": round_5_y > previous_best,
            "Distance from Current Best": distance_from_best,
        }
    )

round_5_diagnostics_df = pd.DataFrame(diagnostic_rows)

round_5_diagnostics_df

,Function,Dimensions,Observations,Previous Best,Round 5 Output,Current Best,Improved in Round 5,Distance from Current Best
0,1,2,15,7.710875e-16,7.580643e-19,7.710875e-16,False,0.288816
1,2,2,15,6.112052e-01,5.140986e-01,6.112052e-01,False,0.296917
2,3,3,20,-3.483531e-02,-1.684900e-01,-3.483531e-02,False,0.779943
3,4,4,35,4.575159e-01,-6.661603e-01,4.575159e-01,False,0.087232
4,5,4,25,2.049118e+03,2.142453e+03,2.142453e+03,True,0.000000
5,6,5,25,-3.997806e-01,-5.996891e-01,-3.997806e-01,False,0.120595
6,7,6,35,1.479290e+00,1.063850e+00,1.479290e+00,False,0.173936
7,8,8,45,9.964291e+00,9.961071e+00,9.964291e+00,False,0.132040


## 39. Round 6 Strategy

In [ ]:
round_6_strategy = {
    1: {
        "local_fraction": 0.15,
        "local_scale": 0.22,
        "min_distance": 0.06,
        "beta": 2.7,
    },

    2: {
        "local_fraction": 0.30,
        "local_scale": 0.18,
        "min_distance": 0.05,
        "beta": 2.3,
    },

    3: {
        "local_fraction": 0.15,
        "local_scale": 0.22,
        "min_distance": 0.07,
        "beta": 2.6,
    },

    4: {
        "local_fraction": 0.20,
        "local_scale": 0.18,
        "min_distance": 0.08,
        "beta": 2.4,
    },

    5: {
        "local_fraction": 0.70,
        "local_scale": 0.07,
        "min_distance": 0.05,
        "beta": 1.8,
    },

    6: {
        "local_fraction": 0.20,
        "local_scale": 0.20,
        "min_distance": 0.10,
        "beta": 2.6,
    },

    7: {
        "local_fraction": 0.20,
        "local_scale": 0.18,
        "min_distance": 0.12,
        "beta": 2.5,
    },

    8: {
        "local_fraction": 0.60,
        "local_scale": 0.09,
        "min_distance": 0.12,
        "beta": 1.9,
    },
}

print("Round 6 strategy loaded.")

Round 6 strategy loaded.


## 40. Generate Round 6 Candidate Sets

In [ ]:
round_6_candidate_sets = {}

for function_number in range(1, 9):
    X = datasets[function_number]["X"]
    y = datasets[function_number]["y"]

    strategy = round_6_strategy[function_number]

    result = generate_round_5_candidates(
        X=X,
        y=y,
        local_fraction=strategy["local_fraction"],
        local_scale=strategy["local_scale"],
        min_distance=strategy["min_distance"],
        beta=strategy["beta"],
        n_candidates=300_000,
        random_state=600 + function_number,
    )

    round_6_candidate_sets[function_number] = result

    print(f"Function {function_number}")
    print("Fitted kernel:", result["model"].kernel_)

    for name, candidate in result["candidates"].items():
        print(f"\n{name}")
        print("Query:", candidate["query"])
        print("Predicted mean:", candidate["mean"])
        print("Predicted std:", candidate["std"])
        print("EI:", candidate["ei"])
        print("UCB:", candidate["ucb"])
        print(
            "Closest existing distance:",
            candidate["distance"],
        )

    print("\n" + "-" * 60 + "\n")

Function 1
Fitted kernel: 1.03**2 * Matern(length_scale=[0.0203, 8.91e+03], nu=2.5) + WhiteKernel(noise_level=2.59e-10)

EI
Query: [0.14545063 0.20130716]
Predicted mean: -0.0002348037592046138
Predicted std: 0.0009306644463284593
EI: 1.639438285387483e-32
UCB: 0.0022779902458822266
Closest existing distance: 0.20735555222946409

UCB
Query: [0.95282112 0.59705074]
Predicted mean: -0.00021740859252740736
Predicted std: 0.0009275066673916334
EI: 1.325729117576873e-32
UCB: 0.002286859409430003
Closest existing distance: 0.07050153508920554

Uncertainty
Query: [0.19582695 0.556527  ]
Predicted mean: -0.0002403107165557468
Predicted std: 0.000930917003673552
EI: 1.5870156046333481e-32
UCB: 0.0022731651933628437
Closest existing distance: 0.19042699829707752

Mean
Query: [0.6943495  0.93399882]
Predicted mean: 0.0002728679113984412
Predicted std: 0.0005341852536998203
EI: 6.351235504724574e-79
UCB: 0.0017151680963879562
Closest existing distance: 0.07375592142024824

------------------------

/usr/local/lib/python3.13/dist-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 10000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


Function 2
Fitted kernel: 0.956**2 * Matern(length_scale=[0.0711, 1e+04], nu=2.5) + WhiteKernel(noise_level=0.0348)

EI
Query: [0.69068127 0.85343558]
Predicted mean: 0.5645533183111953
Predicted std: 0.0501570762081113
EI: 0.003245788369631873
UCB: 0.6799145935898513
Closest existing distance: 0.074099416751552

UCB
Query: [0.68782077 0.98519127]
Predicted mean: 0.5631034804539726
Predicted std: 0.05089327691380297
EI: 0.0032142425876847324
UCB: 0.6801580173557195
Closest existing distance: 0.0530318057219086

Uncertainty
Query: [0.24017119 0.00315415]
Predicted mean: 0.07364135398435534
Predicted std: 0.20131786204483254
EI: 0.0001999386782444842
UCB: 0.5366724366874702
Closest existing distance: 0.10474085065845967

Mean
Query: [0.69487332 0.76349275]
Predicted mean: 0.5653426926087887
Predicted std: 0.04945081623426126
EI: 0.0031990060733400483
UCB: 0.6790795699475896
Closest existing distance: 0.11746699229538998

------------------------------------------------------------



/usr/local/lib/python3.13/dist-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified upper bound 10000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 10000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


Function 3
Fitted kernel: 1.09**2 * Matern(length_scale=[1e+04, 1e+04, 0.0228], nu=2.5) + WhiteKernel(noise_level=6.49e-10)

EI
Query: [0.34602358 0.8082659  0.83270227]
Predicted mean: -0.07535941944768065
Predicted std: 0.07368865727907607
EI: 0.010786955199445022
UCB: 0.11623108947791713
Closest existing distance: 0.445371792999714

UCB
Query: [0.350887   0.22122331 0.4953136 ]
Predicted mean: -0.09257060837632172
Predicted std: 0.08355420711878439
EI: 0.009856235251980676
UCB: 0.12467033013251769
Closest existing distance: 0.21421094884936553

Uncertainty
Query: [0.2135022  0.82453602 0.71286826]
Predicted mean: -0.11212721909019137
Predicted std: 0.08571576556329998
EI: 0.0068948713363022
UCB: 0.11073377137438857
Closest existing distance: 0.47683171992737644

Mean
Query: [0.46225246 0.08706374 0.3338634 ]
Predicted mean: -0.0250033531281611
Predicted std: 0.015135849663612432
EI: 0.005954682620133699
UCB: 0.014349855997231223
Closest existing distance: 0.3208344478916671

-------

/usr/local/lib/python3.13/dist-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified upper bound 10000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 10000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


Function 5
Fitted kernel: 1.57**2 * Matern(length_scale=[1e+04, 1e+04, 0.686, 0.829], nu=2.5) + WhiteKernel(noise_level=0.0101)

EI
Query: [0.7216586  0.11900464 0.99979896 0.99993767]
Predicted mean: 2066.956076973388
Predicted std: 85.02324598907269
EI: 8.727141138147756
UCB: 2219.9979197537186
Closest existing distance: 0.13388981513495804

UCB
Query: [0.7216586  0.11900464 0.99979896 0.99993767]
Predicted mean: 2066.956076973388
Predicted std: 85.02324598907269
EI: 8.727141138147756
UCB: 2219.9979197537186
Closest existing distance: 0.13388981513495804

Uncertainty
Query: [0.9937658  0.5585934  0.99821538 0.00543346]
Predicted mean: 187.29002104779397
Predicted std: 397.7122159918802
EI: 3.323979031541436e-05
UCB: 903.1720098331783
Closest existing distance: 0.5466591697236943

Mean
Query: [0.7216586  0.11900464 0.99979896 0.99993767]
Predicted mean: 2066.956076973388
Predicted std: 85.02324598907269
EI: 8.727141138147756
UCB: 2219.9979197537186
Closest existing distance: 0.1338898

/usr/local/lib/python3.13/dist-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 7 of parameter k1__k2__length_scale is close to the specified upper bound 10000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/gaussian_process/kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-10. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


Function 8
Fitted kernel: 10.6**2 * Matern(length_scale=[6.67, 9.33, 5.31, 10.2, 16.4, 9.36, 6.81, 1e+04], nu=2.5) + WhiteKernel(noise_level=1e-10)

EI
Query: [0.02265708 0.1654596  0.10125101 0.19240635 0.99813641 0.40546468
 0.14200198 0.76587661]
Predicted mean: 9.949544132269283
Predicted std: 0.03445107351466351
EI: 0.0047715352675983605
UCB: 10.015001171947144
Closest existing distance: 0.17926805402929724

UCB
Query: [0.01807082 0.11742728 0.08865105 0.11106924 0.989454   0.35830825
 0.07647259 0.74353916]
Predicted mean: 9.92005901659336
Predicted std: 0.052536827753497994
EI: 0.004114793544714777
UCB: 10.019878989325006
Closest existing distance: 0.24735105927671425

Uncertainty
Query: [0.11143636 0.99214428 0.99659438 0.99319923 0.91578336 0.88268789
 0.94327997 0.81304187]
Predicted mean: 5.39617451971951
Predicted std: 0.277863827413724
EI: 4.5603442769123584e-63
UCB: 5.924115791805586
Closest existing distance: 1.0901675197860772

Mean
Query: [0.10343088 0.15230782 0.12183

## 41. Function 4 Round 6 Diversified Candidates

In [ ]:
X = datasets[4]["X"]
y = datasets[4]["y"]

f4_round_6_diversified = generate_round_5_candidates(
    X=X,
    y=y,
    local_fraction=0.10,
    local_scale=0.22,
    min_distance=0.15,
    beta=2.6,
    n_candidates=400_000,
    random_state=804,
)

print("Function 4 Round 6 diversified candidates")
print(
    "Fitted kernel:",
    f4_round_6_diversified["model"].kernel_,
)

for name, candidate in f4_round_6_diversified["candidates"].items():
    print(f"\n{name}")
    print("Query:", candidate["query"])
    print("Predicted mean:", candidate["mean"])
    print("Predicted std:", candidate["std"])
    print("EI:", candidate["ei"])
    print("UCB:", candidate["ucb"])
    print(
        "Closest existing distance:",
        candidate["distance"],
    )

Function 4 Round 6 diversified candidates
Fitted kernel: 2.82**2 * Matern(length_scale=[1.65, 1.47, 1.45, 1.5], nu=2.5) + WhiteKernel(noise_level=0.00119)

EI
Query: [0.40229669 0.30422833 0.49498487 0.45591935]
Predicted mean: -1.3491145975685832
Predicted std: 0.7321190373950587
EI: 0.0015544063786310843
UCB: 0.5543948996585695
Closest existing distance: 0.15010270538259113

UCB
Query: [0.40229669 0.30422833 0.49498487 0.45591935]
Predicted mean: -1.3491145975685832
Predicted std: 0.7321190373950587
EI: 0.0015544063786310843
UCB: 0.5543948996585695
Closest existing distance: 0.15010270538259113

Uncertainty
Query: [0.02931609 0.04227357 0.98914028 0.98214737]
Predicted mean: -28.132020762226563
Predicted std: 6.469034556857665
EI: 6.591934605829406e-06
UCB: -11.312530914396635
Closest existing distance: 0.539162604254524

Mean
Query: [0.48588173 0.45259558 0.50354116 0.43765512]
Predicted mean: -1.2467356634314193
Predicted std: 0.5380149683250706
EI: 0.00010487087138973225
UCB: 0.15

## 42. Round 6 Submission Values

In [ ]:
round_6_queries = {
    1: np.array([0.95282112, 0.59705074]),

    2: np.array([0.69487332, 0.76349275]),

    3: np.array([
        0.46225246,
        0.08706374,
        0.33386340,
    ]),

    4: np.array([
        0.39143742,
        0.35188293,
        0.43910248,
        0.43205701,
    ]),

    5: np.array([
        0.72165860,
        0.11900464,
        0.99979896,
        0.99993767,
    ]),

    6: np.array([
        0.45969702,
        0.31433917,
        0.65867730,
        0.75022432,
        0.00430888,
    ]),

    7: np.array([
        0.10346183,
        0.34177521,
        0.45337484,
        0.18001609,
        0.39393495,
        0.78007909,
    ]),

    8: np.array([
        0.10343088,
        0.15230782,
        0.12183862,
        0.16426330,
        0.98934843,
        0.49748774,
        0.22783769,
        0.67043082,
    ]),
}

round_6_submission_values = {}

print("ROUND 6 SUBMISSION VALUES\n")

for function_number in range(1, 9):
    submission_string = "-".join(
        f"{value:.6f}"
        for value in round_6_queries[function_number]
    )

    round_6_submission_values[function_number] = submission_string

    print(
        f"Function {function_number}: "
        f"{submission_string}"
    )

ROUND 6 SUBMISSION VALUES

Function 1: 0.952821-0.597051
Function 2: 0.694873-0.763493
Function 3: 0.462252-0.087064-0.333863
Function 4: 0.391437-0.351883-0.439102-0.432057
Function 5: 0.721659-0.119005-0.999799-0.999938
Function 6: 0.459697-0.314339-0.658677-0.750224-0.004309
Function 7: 0.103462-0.341775-0.453375-0.180016-0.393935-0.780079
Function 8: 0.103431-0.152308-0.121839-0.164263-0.989348-0.497488-0.227838-0.670431


## 43. Save Round 6 Queries

In [ ]:
import os
import json

week_6_dir = "/content/drive/MyDrive/Imperial_BBO/data/week_6"

os.makedirs(
    week_6_dir,
    exist_ok=True,
)

round_6_queries_to_save = {
    f"Function {function_number}": [
        round(float(value), 6)
        for value in round_6_queries[function_number]
    ]
    for function_number in range(1, 9)
}

save_path = os.path.join(
    week_6_dir,
    "submitted_queries.json",
)

with open(save_path, "w") as file:
    json.dump(
        round_6_queries_to_save,
        file,
        indent=4,
    )

print(f"Saved to: {save_path}")

Saved to: /content/drive/MyDrive/Imperial_BBO/data/week_6/submitted_queries.json


## 44. Round 6 Results

In [ ]:
round_6_results = {
    1: {
        "x": [0.952821, 0.597051],
        "y": 9.77293262952221e-75,
    },
    2: {
        "x": [0.694873, 0.763493],
        "y": 0.6122414108233222,
    },
    3: {
        "x": [0.462252, 0.087064, 0.333863],
        "y": -0.09665268796313876,
    },
    4: {
        "x": [0.391437, 0.351883, 0.439102, 0.432057],
        "y": 0.3163348226957905,
    },
    5: {
        "x": [0.721659, 0.119005, 0.999799, 0.999938],
        "y": 2259.061659721088,
    },
    6: {
        "x": [0.459697, 0.314339, 0.658677, 0.750224, 0.004309],
        "y": -0.28389722921895866,
    },
    7: {
        "x": [0.103462, 0.341775, 0.453375, 0.180016, 0.393935, 0.780079],
        "y": 2.105312487096124,
    },
    8: {
        "x": [0.103431, 0.152308, 0.121839, 0.164263, 0.989348, 0.497488, 0.227838, 0.670431],
        "y": 9.9795892869219,
    },
}

print("Round 6 results loaded.")

Round 6 results loaded.


## 45. Update Datasets with Round 6 Results

In [ ]:
for function_number in range(1, 9):
    result = round_6_results[function_number]

    datasets[function_number]["X"] = np.vstack([
        datasets[function_number]["X"],
        np.array(result["x"]).reshape(1, -1),
    ])

    datasets[function_number]["y"] = np.append(
        datasets[function_number]["y"],
        result["y"],
    )

print("Round 6 results added to datasets.")

for function_number in range(1, 9):
    print(
        f"Function {function_number}: "
        f"{len(datasets[function_number]['y'])} observations"
    )

Round 6 results added to datasets.
Function 1: 16 observations
Function 2: 16 observations
Function 3: 21 observations
Function 4: 36 observations
Function 5: 26 observations
Function 6: 26 observations
Function 7: 36 observations
Function 8: 46 observations


## 46. Round 6 Diagnostics

In [ ]:
import pandas as pd
import numpy as np

round_6_diagnostics = []

for function_number in range(1, 9):
    result = round_6_results[function_number]

    X = datasets[function_number]["X"]
    y = datasets[function_number]["y"]

    previous_y = y[:-1]
    previous_best = np.max(previous_y)

    round_6_output = result["y"]
    current_best = np.max(y)

    improved = round_6_output > previous_best

    best_index = np.argmax(y)
    current_best_x = X[best_index]

    distance_from_best = np.linalg.norm(
        np.array(result["x"]) - current_best_x
    )

    round_6_diagnostics.append({
        "Function": function_number,
        "Dimensions": X.shape[1],
        "Observations": len(y),
        "Previous Best": previous_best,
        "Round 6 Output": round_6_output,
        "Current Best": current_best,
        "Improved in Round 6": improved,
        "Distance from Current Best": distance_from_best,
    })

round_6_diagnostics_df = pd.DataFrame(round_6_diagnostics)

round_6_diagnostics_df

,Function,Dimensions,Observations,Previous Best,Round 6 Output,Current Best,Improved in Round 6,Distance from Current Best
0,1,2,16,7.710875e-16,9.772933e-75,7.710875e-16,False,0.260146
1,2,2,16,6.112052e-01,6.122414e-01,6.122414e-01,True,0.000000
2,3,3,21,-3.483531e-02,-9.665269e-02,-3.483531e-02,False,0.525443
3,4,4,36,4.575159e-01,3.163348e-01,4.575159e-01,False,0.084382
4,5,4,26,2.142453e+03,2.259062e+03,2.259062e+03,True,0.000000
5,6,5,26,-3.997806e-01,-2.838972e-01,-2.838972e-01,True,0.000000
6,7,6,36,1.479290e+00,2.105312e+00,2.105312e+00,True,0.000000
7,8,8,46,9.964291e+00,9.979589e+00,9.979589e+00,True,0.000000


## 47. Fit Gaussian Process Models for Round 7

In [ ]:
round_7_models = {}

for function_number in range(1, 9):
    X = datasets[function_number]["X"]
    y = datasets[function_number]["y"]

    dimensions = X.shape[1]

    kernel = (
        ConstantKernel(
            constant_value=1.0,
            constant_value_bounds=(1e-3, 1e3),
        )
        * Matern(
            length_scale=np.ones(dimensions),
            length_scale_bounds=(1e-4, 1e4),
            nu=2.5,
        )
        + WhiteKernel(
            noise_level=1e-6,
            noise_level_bounds=(1e-10, 1e-1),
        )
    )

    model = GaussianProcessRegressor(
        kernel=kernel,
        normalize_y=True,
        n_restarts_optimizer=10,
        random_state=42 + function_number,
    )

    model.fit(X, y)

    round_7_models[function_number] = model

    print(f"Function {function_number}")
    print("Fitted kernel:", model.kernel_)
    print()

Function 1
Fitted kernel: 1.05**2 * Matern(length_scale=[5.91e+03, 0.0312], nu=2.5) + WhiteKernel(noise_level=3.82e-08)

Function 2
Fitted kernel: 0.981**2 * Matern(length_scale=[0.0757, 228], nu=2.5) + WhiteKernel(noise_level=0.0315)

Function 3
Fitted kernel: 1.03**2 * Matern(length_scale=[356, 0.013, 2.43e+03], nu=2.5) + WhiteKernel(noise_level=1.06e-08)

Function 4
Fitted kernel: 2.97**2 * Matern(length_scale=[1.66, 1.59, 1.47, 1.56], nu=2.5) + WhiteKernel(noise_level=0.000949)



/usr/local/lib/python3.13/dist-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified upper bound 10000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 10000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


Function 5
Fitted kernel: 1.69**2 * Matern(length_scale=[1e+04, 1e+04, 0.927, 0.851], nu=2.5) + WhiteKernel(noise_level=0.0127)

Function 6
Fitted kernel: 1.47**2 * Matern(length_scale=[0.799, 0.795, 1.3, 0.965, 1.77], nu=2.5) + WhiteKernel(noise_level=1e-06)



/usr/local/lib/python3.13/dist-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 5 of parameter k1__k2__length_scale is close to the specified upper bound 10000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


Function 7
Fitted kernel: 0.786**2 * Matern(length_scale=[0.171, 406, 2.64e+03, 0.11, 0.148, 1e+04], nu=2.5) + WhiteKernel(noise_level=9.96e-07)

Function 8
Fitted kernel: 9.44**2 * Matern(length_scale=[6.34, 8.57, 4.95, 9.15, 15.9, 8.61, 6.5, 536], nu=2.5) + WhiteKernel(noise_level=4.58e-10)



## 48. Round 7 Candidate Generator

In [ ]:
from scipy.stats import norm
from scipy.spatial.distance import cdist

def standardized_expected_improvement(model, X_candidates, y_best):
    mean, std = model.predict(X_candidates, return_std=True)

    std = np.maximum(std, 1e-12)

    y_train_mean = model._y_train_mean
    y_train_std = model._y_train_std

    if np.ndim(y_train_std) > 0:
        y_train_std = float(np.ravel(y_train_std)[0])

    if y_train_std <= 1e-12:
        y_train_std = 1.0

    best_standardized = (y_best - y_train_mean) / y_train_std
    mean_standardized = (mean - y_train_mean) / y_train_std
    std_standardized = std / y_train_std

    improvement = mean_standardized - best_standardized

    z = improvement / np.maximum(std_standardized, 1e-12)

    ei = (
        improvement * norm.cdf(z)
        + std_standardized * norm.pdf(z)
    )

    return ei


def generate_round_7_candidates(
    function_number,
    model,
    X,
    y,
    random_state=42,
):
    rng = np.random.default_rng(
        random_state + function_number
    )

    dimensions = X.shape[1]

    best_index = np.argmax(y)
    best_x = X[best_index]
    best_y = y[best_index]

    settings = {
        1: {
            "local_fraction": 0.15,
            "radius": 0.30,
            "kappa": 3.0,
        },
        2: {
            "local_fraction": 0.55,
            "radius": 0.18,
            "kappa": 2.0,
        },
        3: {
            "local_fraction": 0.20,
            "radius": 0.35,
            "kappa": 3.0,
        },
        4: {
            "local_fraction": 0.50,
            "radius": 0.20,
            "kappa": 2.2,
        },
        5: {
            "local_fraction": 0.80,
            "radius": 0.12,
            "kappa": 1.5,
        },
        6: {
            "local_fraction": 0.70,
            "radius": 0.16,
            "kappa": 1.8,
        },
        7: {
            "local_fraction": 0.85,
            "radius": 0.12,
            "kappa": 1.5,
        },
        8: {
            "local_fraction": 0.70,
            "radius": 0.15,
            "kappa": 1.8,
        },
    }

    config = settings[function_number]

    n_candidates = 60000

    n_local = int(
        n_candidates * config["local_fraction"]
    )
    n_global = n_candidates - n_local

    global_candidates = rng.uniform(
        0.0001,
        0.9999,
        size=(n_global, dimensions),
    )

    local_candidates = best_x + rng.normal(
        0,
        config["radius"],
        size=(n_local, dimensions),
    )

    local_candidates = np.clip(
        local_candidates,
        0.0001,
        0.9999,
    )

    candidates = np.vstack([
        global_candidates,
        local_candidates,
    ])

    distances = cdist(
        candidates,
        X,
    ).min(axis=1)

    minimum_distance = {
        1: 0.08,
        2: 0.05,
        3: 0.10,
        4: 0.06,
        5: 0.04,
        6: 0.05,
        7: 0.05,
        8: 0.06,
    }[function_number]

    valid = distances >= minimum_distance

    candidates = candidates[valid]
    distances = distances[valid]

    mean, std = model.predict(
        candidates,
        return_std=True,
    )

    ei = standardized_expected_improvement(
        model,
        candidates,
        best_y,
    )

    mean_scale = np.std(mean)

    if mean_scale <= 1e-12:
        mean_scale = 1.0

    ucb = (
        mean
        + config["kappa"] * std
    )

    scores = {
        "EI": ei,
        "UCB": ucb,
        "Mean": mean,
        "Uncertainty": std,
    }

    results = {}

    for method, score in scores.items():
        index = np.argmax(score)

        results[method] = {
            "x": candidates[index],
            "mean": mean[index],
            "std": std[index],
            "score": score[index],
            "distance": distances[index],
        }

    return results

## 49. Generate Round 7 Candidates

In [ ]:
round_7_candidates = {}

for function_number in range(1, 9):
    X = datasets[function_number]["X"]
    y = datasets[function_number]["y"]
    model = round_7_models[function_number]

    candidates = generate_round_7_candidates(
        function_number=function_number,
        model=model,
        X=X,
        y=y,
        random_state=42,
    )

    round_7_candidates[function_number] = candidates

    print(f"\nFunction {function_number}")
    print("-" * 60)

    for method, result in candidates.items():
        x_string = ", ".join(
            f"{value:.6f}"
            for value in result["x"]
        )

        print(f"{method}")
        print(f"x: [{x_string}]")
        print(f"mean: {result['mean']:.6f}")
        print(f"std: {result['std']:.6f}")
        print(f"score: {result['score']:.6f}")
        print(f"distance: {result['distance']:.6f}")
        print()


Function 1
------------------------------------------------------------
EI
x: [0.724964, 0.452925]
mean: -0.000120
std: 0.000861
score: 0.328676
distance: 0.152735

UCB
x: [0.020765, 0.000100]
mean: -0.000200
std: 0.000906
score: 0.002516
distance: 0.302329

Mean
x: [0.584272, 0.744676]
mean: 0.000178
std: 0.000254
score: 0.000178
distance: 0.091231

Uncertainty
x: [0.020765, 0.000100]
mean: -0.000200
std: 0.000906
score: 0.000906
distance: 0.302329


Function 2
------------------------------------------------------------
EI
x: [0.691832, 0.978274]
mean: 0.576445
std: 0.047586
score: 0.025986
distance: 0.051698

UCB
x: [0.690016, 0.980263]
mean: 0.575790
std: 0.047971
score: 0.671731
distance: 0.052573

Mean
x: [0.694184, 0.825717]
mean: 0.576719
std: 0.047255
score: 0.576719
distance: 0.062228

Uncertainty
x: [0.240027, 0.869102]
mean: 0.060518
std: 0.206258
score: 0.206258
distance: 0.268934


Function 3
------------------------------------------------------------
EI
x: [0.791630, 0

## 50. Select Round 7 Queries

In [ ]:
round_7_selections = {
    1: "UCB",
    2: "Mean",
    3: "UCB",
    4: "UCB",
    5: "EI",
    6: "Mean",
    7: "Mean",
    8: "Mean",
}

round_7_queries = {}

for function_number in range(1, 9):
    method = round_7_selections[function_number]

    round_7_queries[function_number] = (
        round_7_candidates[function_number][method]["x"]
    )

    values = "-".join(
        f"{value:.6f}"
        for value in round_7_queries[function_number]
    )

    print(
        f"Function {function_number}: "
        f"{values}"
    )

Function 1: 0.020765-0.000100
Function 2: 0.694184-0.825717
Function 3: 0.786923-0.838462-0.220085
Function 4: 0.411722-0.411048-0.469445-0.434386
Function 5: 0.999900-0.000100-0.999900-0.999900
Function 6: 0.448596-0.273706-0.701605-0.756821-0.000100
Function 7: 0.132125-0.267293-0.378521-0.165728-0.386801-0.746167
Function 8: 0.035073-0.180868-0.136597-0.187581-0.999900-0.528293-0.264511-0.583930


## 51. Save Round 7 Queries

In [ ]:
import os
import json

week_7_dir = "/content/drive/MyDrive/Imperial_BBO/data/week_7"

os.makedirs(
    week_7_dir,
    exist_ok=True,
)

round_7_queries_to_save = {
    f"Function {function_number}": [
        round(float(value), 6)
        for value in round_7_queries[function_number]
    ]
    for function_number in range(1, 9)
}

save_path = os.path.join(
    week_7_dir,
    "submitted_queries.json",
)

with open(save_path, "w") as file:
    json.dump(
        round_7_queries_to_save,
        file,
        indent=4,
    )

print(f"Saved to: {save_path}")

Saved to: /content/drive/MyDrive/Imperial_BBO/data/week_7/submitted_queries.json


## 52. Round 7 Results

In [ ]:
round_7_results = {
    1: {
        "x": [0.020765, 0.000100],
        "y": 4.522177955217332e-236,
    },
    2: {
        "x": [0.694184, 0.825717],
        "y": 0.5890431360008267,
    },
    3: {
        "x": [0.786923, 0.838462, 0.220085],
        "y": -0.14527277287827178,
    },
    4: {
        "x": [0.411722, 0.411048, 0.469445, 0.434386],
        "y": -0.5258612720975253,
    },
    5: {
        "x": [0.999900, 0.000100, 0.999900, 0.999900],
        "y": 4436.361460610766,
    },
    6: {
        "x": [0.448596, 0.273706, 0.701605, 0.756821, 0.000100],
        "y": -0.2588092056883585,
    },
    7: {
        "x": [0.132125, 0.267293, 0.378521, 0.165728, 0.386801, 0.746167],
        "y": 2.318788583741406,
    },
    8: {
        "x": [0.035073, 0.180868, 0.136597, 0.187581, 0.999900, 0.528293, 0.264511, 0.583930],
        "y": 9.959943581549,
    },
}

print("Round 7 results loaded.")

Round 7 results loaded.


## 53. Update Datasets with Round 7 Results

In [ ]:
for function_number in range(1, 9):
    result = round_7_results[function_number]

    new_x = np.array(result["x"]).reshape(1, -1)
    new_y = np.array([result["y"]])

    datasets[function_number]["X"] = np.vstack([
        datasets[function_number]["X"],
        new_x,
    ])

    datasets[function_number]["y"] = np.concatenate([
        datasets[function_number]["y"],
        new_y,
    ])

print("Round 7 results added to datasets.")

for function_number in range(1, 9):
    print(
        f"Function {function_number}: "
        f"{len(datasets[function_number]['y'])} observations"
    )

Round 7 results added to datasets.
Function 1: 17 observations
Function 2: 17 observations
Function 3: 22 observations
Function 4: 37 observations
Function 5: 27 observations
Function 6: 27 observations
Function 7: 37 observations
Function 8: 47 observations


## 54. Round 7 Diagnostics

In [ ]:
for function_number in range(1, 9):
    X = datasets[function_number]["X"]
    y = datasets[function_number]["y"]

    current_result = round_7_results[function_number]
    current_y = current_result["y"]

    previous_y = y[:-1]
    previous_best = np.max(previous_y)

    current_best = np.max(y)
    improved = current_y > previous_best

    best_index = np.argmax(y)
    best_x = X[best_index]

    current_x = np.array(current_result["x"])
    distance_to_best = np.linalg.norm(current_x - best_x)

    print(f"Function {function_number}")
    print(f"  Observations: {len(y)}")
    print(f"  Previous best: {previous_best:.10g}")
    print(f"  Round 7 value: {current_y:.10g}")
    print(f"  Current best: {current_best:.10g}")
    print(f"  Improved: {improved}")
    print(f"  Distance to current best: {distance_to_best:.6f}")
    print()

Function 1
  Observations: 17
  Previous best: 7.710875115e-16
  Round 7 value: 4.522177955e-236
  Current best: 7.710875115e-16
  Improved: False
  Distance to current best: 1.020593

Function 2
  Observations: 17
  Previous best: 0.6122414108
  Round 7 value: 0.589043136
  Current best: 0.6122414108
  Improved: False
  Distance to current best: 0.062228

Function 3
  Observations: 22
  Previous best: -0.03483531335
  Round 7 value: -0.1452727729
  Current best: -0.03483531335
  Improved: False
  Distance to current best: 0.390549

Function 4
  Observations: 37
  Previous best: 0.4575158799
  Round 7 value: -0.5258612721
  Current best: 0.4575158799
  Improved: False
  Distance to current best: 0.070620

Function 5
  Observations: 27
  Previous best: 2259.06166
  Round 7 value: 4436.361461
  Current best: 4436.361461
  Improved: True
  Distance to current best: 0.000000

Function 6
  Observations: 27
  Previous best: -0.2838972292
  Round 7 value: -0.2588092057
  Current best: -0.2588

## 55. Fit Gaussian Process Models for Round 8/

In [ ]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, Matern, WhiteKernel

round_8_models = {}

for function_number in range(1, 9):
    X = datasets[function_number]["X"]
    y = datasets[function_number]["y"]

    dimensions = X.shape[1]

    kernel = (
        ConstantKernel(
            constant_value=1.0,
            constant_value_bounds=(1e-3, 1e3),
        )
        * Matern(
            length_scale=np.ones(dimensions),
            length_scale_bounds=(1e-4, 1e4),
            nu=2.5,
        )
        + WhiteKernel(
            noise_level=1e-6,
            noise_level_bounds=(1e-10, 1e-1),
        )
    )

    model = GaussianProcessRegressor(
        kernel=kernel,
        normalize_y=True,
        n_restarts_optimizer=15,
        random_state=100 + function_number,
    )

    model.fit(X, y)

    round_8_models[function_number] = model

    print(f"Function {function_number}")
    print("Fitted kernel:", model.kernel_)
    print()

Function 1
Fitted kernel: 1.05**2 * Matern(length_scale=[9e+03, 0.0308], nu=2.5) + WhiteKernel(noise_level=8.78e-08)



/usr/local/lib/python3.13/dist-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 10000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


Function 2
Fitted kernel: 1.02**2 * Matern(length_scale=[0.0795, 1e+04], nu=2.5) + WhiteKernel(noise_level=0.0255)

Function 3
Fitted kernel: 1.06**2 * Matern(length_scale=[380, 369, 0.0122], nu=2.5) + WhiteKernel(noise_level=6.43e-09)

Function 4
Fitted kernel: 3.16**2 * Matern(length_scale=[1.77, 1.64, 1.6, 1.64], nu=2.5) + WhiteKernel(noise_level=0.00137)

Function 5
Fitted kernel: 3.13**2 * Matern(length_scale=[0.933, 9.16, 0.938, 3.87], nu=2.5) + WhiteKernel(noise_level=3.66e-05)



/usr/local/lib/python3.13/dist-packages/sklearn/gaussian_process/kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-10. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


Function 6
Fitted kernel: 1.57**2 * Matern(length_scale=[0.894, 0.823, 1.41, 1.09, 2.06], nu=2.5) + WhiteKernel(noise_level=1e-10)



/usr/local/lib/python3.13/dist-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 10000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 2 of parameter k1__k2__length_scale is close to the specified upper bound 10000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 5 of parameter k1__k2__length_scale is close to the specified upper bound 10000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


Function 7
Fitted kernel: 0.731**2 * Matern(length_scale=[0.212, 1e+04, 1e+04, 0.127, 0.168, 1e+04], nu=2.5) + WhiteKernel(noise_level=2.31e-10)

Function 8
Fitted kernel: 11.9**2 * Matern(length_scale=[7.34, 10, 5.85, 10.2, 16.1, 10.1, 7.68, 555], nu=2.5) + WhiteKernel(noise_level=1.44e-10)



## 56. Round 7 Predictive Calibration Check

In [ ]:
round_7_calibration = {}

for function_number in range(1, 9):
    X = datasets[function_number]["X"]
    y = datasets[function_number]["y"]

    X_train = X[:-1]
    y_train = y[:-1]

    X_test = X[-1].reshape(1, -1)
    y_actual = y[-1]

    dimensions = X_train.shape[1]

    kernel = (
        ConstantKernel(
            constant_value=1.0,
            constant_value_bounds=(1e-3, 1e3),
        )
        * Matern(
            length_scale=np.ones(dimensions),
            length_scale_bounds=(1e-4, 1e4),
            nu=2.5,
        )
        + WhiteKernel(
            noise_level=1e-6,
            noise_level_bounds=(1e-10, 1e-1),
        )
    )

    model = GaussianProcessRegressor(
        kernel=kernel,
        normalize_y=True,
        n_restarts_optimizer=10,
        random_state=200 + function_number,
    )

    model.fit(X_train, y_train)

    predicted_mean, predicted_std = model.predict(
        X_test,
        return_std=True,
    )

    predicted_mean = float(predicted_mean[0])
    predicted_std = float(predicted_std[0])

    error = y_actual - predicted_mean

    if predicted_std > 1e-12:
        z_score = error / predicted_std
    else:
        z_score = np.nan

    round_7_calibration[function_number] = {
        "predicted_mean": predicted_mean,
        "predicted_std": predicted_std,
        "actual": y_actual,
        "error": error,
        "z_score": z_score,
    }

    print(f"Function {function_number}")
    print(f"  Predicted: {predicted_mean:.10g}")
    print(f"  Predicted std: {predicted_std:.10g}")
    print(f"  Actual: {y_actual:.10g}")
    print(f"  Error: {error:.10g}")
    print(f"  Standardised error: {z_score:.3f}")
    print()

/usr/local/lib/python3.13/dist-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified upper bound 10000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


Function 1
  Predicted: -0.0002004297504
  Predicted std: 0.0009056083817
  Actual: 4.522177955e-236
  Error: 0.0002004297504
  Standardised error: 0.221



/usr/local/lib/python3.13/dist-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 10000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


Function 2
  Predicted: 0.5767153521
  Predicted std: 0.04725201269
  Actual: 0.589043136
  Error: 0.0123277839
  Standardised error: 0.261

Function 3
  Predicted: -0.121708881
  Predicted std: 0.02146775064
  Actual: -0.1452727729
  Error: -0.02356389187
  Standardised error: -1.098

Function 4
  Predicted: 0.1206022064
  Predicted std: 0.3686986876
  Actual: -0.5258612721
  Error: -0.6464634785
  Standardised error: -1.753

Function 5
  Predicted: 2741.385858
  Predicted std: 151.9001737
  Actual: 4436.361461
  Error: 1694.975603
  Standardised error: 11.158

Function 6
  Predicted: -0.3200877304
  Predicted std: 0.09387635583
  Actual: -0.2588092057
  Error: 0.06127852471
  Standardised error: 0.653



/usr/local/lib/python3.13/dist-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 10000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 2 of parameter k1__k2__length_scale is close to the specified upper bound 10000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/gaussian_process/kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-10. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


Function 7
  Predicted: 1.811459914
  Predicted std: 0.1242445414
  Actual: 2.318788584
  Error: 0.5073286697
  Standardised error: 4.083

Function 8
  Predicted: 9.974094911
  Predicted std: 0.02804859568
  Actual: 9.959943582
  Error: -0.01415132933
  Standardised error: -0.505



/usr/local/lib/python3.13/dist-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 5 of parameter k1__k2__length_scale is close to the specified upper bound 10000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 7 of parameter k1__k2__length_scale is close to the specified upper bound 10000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


## 57. Round 8 Candidate Generator

In [ ]:
from scipy.stats import norm, qmc
from scipy.spatial.distance import cdist

def expected_improvement(model, X_candidates, y_best):
    mean, std = model.predict(X_candidates, return_std=True)

    std = np.maximum(std, 1e-12)

    improvement = mean - y_best
    z = improvement / std

    ei = (
        improvement * norm.cdf(z)
        + std * norm.pdf(z)
    )

    return ei


def generate_round_8_candidates(
    function_number,
    model,
    X,
    y,
    random_state=42,
):
    rng = np.random.default_rng(
        random_state + function_number
    )

    dimensions = X.shape[1]

    best_index = np.argmax(y)
    best_x = X[best_index]
    best_y = y[best_index]

    settings = {
        1: {
            "global_fraction": 0.50,
            "local_radius": 0.20,
            "kappa": 2.8,
        },
        2: {
            "global_fraction": 0.30,
            "local_radius": 0.10,
            "kappa": 1.8,
        },
        3: {
            "global_fraction": 0.70,
            "local_radius": 0.25,
            "kappa": 2.8,
        },
        4: {
            "global_fraction": 0.40,
            "local_radius": 0.08,
            "kappa": 2.2,
        },
        5: {
            "global_fraction": 0.05,
            "local_radius": 0.04,
            "kappa": 1.2,
        },
        6: {
            "global_fraction": 0.20,
            "local_radius": 0.10,
            "kappa": 1.5,
        },
        7: {
            "global_fraction": 0.15,
            "local_radius": 0.08,
            "kappa": 1.4,
        },
        8: {
            "global_fraction": 0.35,
            "local_radius": 0.10,
            "kappa": 1.8,
        },
    }

    config = settings[function_number]

    n_total = 65536

    n_global = int(
        n_total * config["global_fraction"]
    )

    n_local = n_total - n_global

    sampler = qmc.Sobol(
        d=dimensions,
        scramble=True,
        seed=random_state + function_number,
    )

    global_candidates = sampler.random(
        n=n_global
    )

    local_candidates = (
        best_x
        + rng.normal(
            0,
            config["local_radius"],
            size=(n_local, dimensions),
        )
    )

    local_candidates = np.clip(
        local_candidates,
        0.000001,
        0.999999,
    )

    candidate_sets = [
        global_candidates,
        local_candidates,
    ]

    if function_number == 5:
        n_boundary = 20000

        boundary_candidates = np.tile(
            best_x,
            (n_boundary, 1),
        )

        boundary_candidates += rng.normal(
            0,
            0.025,
            size=boundary_candidates.shape,
        )

        boundary_candidates = np.clip(
            boundary_candidates,
            0.000001,
            0.999999,
        )

        candidate_sets.append(
            boundary_candidates
        )

    candidates = np.vstack(
        candidate_sets
    )

    distances = cdist(
        candidates,
        X,
    ).min(axis=1)

    valid = distances > 1e-5

    candidates = candidates[valid]
    distances = distances[valid]

    mean, std = model.predict(
        candidates,
        return_std=True,
    )

    ei = expected_improvement(
        model,
        candidates,
        best_y,
    )

    ucb = (
        mean
        + config["kappa"] * std
    )

    scores = {
        "EI": ei,
        "UCB": ucb,
        "Mean": mean,
        "Uncertainty": std,
    }

    results = {}

    for method, score in scores.items():
        index = np.argmax(score)

        results[method] = {
            "x": candidates[index],
            "mean": mean[index],
            "std": std[index],
            "score": score[index],
            "distance": distances[index],
        }

    return results

## 58. Generate Round 8 Candidates

In [ ]:
round_8_candidates = {}

for function_number in range(1, 9):
    X = datasets[function_number]["X"]
    y = datasets[function_number]["y"]

    model = round_8_models[function_number]

    round_8_candidates[function_number] = (
        generate_round_8_candidates(
            function_number=function_number,
            model=model,
            X=X,
            y=y,
            random_state=800,
        )
    )

    print(f"Function {function_number}")

    for method, result in round_8_candidates[function_number].items():
        values = ", ".join(
            f"{value:.6f}"
            for value in result["x"]
        )

        print(
            f"  {method}: [{values}]"
        )
        print(
            f"    mean={result['mean']:.6f}, "
            f"std={result['std']:.6f}, "
            f"score={result['score']:.6f}, "
            f"distance={result['distance']:.6f}"
        )

    print()

Function 1
  EI: [0.564024, 0.453006]
    mean=-0.000115, std=0.000837, score=0.000280, distance=0.191258
  UCB: [0.076540, 0.942532]
    mean=-0.000154, std=0.000865, score=0.002269, distance=0.302042
  Mean: [0.029524, 0.744614]
    mean=0.000176, std=0.000250, score=0.000176, distance=0.290459
  Uncertainty: [0.823451, 0.941454]
    mean=-0.000155, std=0.000866, score=0.000866, distance=0.161471



/tmp/ipykernel_5596/3233374439.py:96: UserWarning: The balance properties of Sobol' points require n to be a power of 2.
  global_candidates = sampler.random(


Function 2
  EI: [0.691819, 0.625824]
    mean=0.579223, std=0.042602, score=0.005350, distance=0.006418
  UCB: [0.690499, 0.838771]
    mean=0.578810, std=0.042861, score=0.655960, distance=0.013564
  Mean: [0.693780, 0.829166]
    mean=0.579452, std=0.042339, score=0.579452, distance=0.003472
  Uncertainty: [0.239901, 0.702135]
    mean=0.046137, std=0.209818, score=0.209818, distance=0.199003



/tmp/ipykernel_5596/3233374439.py:96: UserWarning: The balance properties of Sobol' points require n to be a power of 2.
  global_candidates = sampler.random(


Function 3
  EI: [0.461000, 0.155883, 0.076288]
    mean=-0.053442, std=0.058426, score=0.015177, distance=0.266613
  UCB: [0.719237, 0.424878, 0.048479]
    mean=-0.088164, std=0.076390, score=0.125728, distance=0.224223
  Mean: [0.264381, 0.999999, 0.340606]
    mean=-0.034530, std=0.000978, score=-0.034530, distance=0.122875
  Uncertainty: [0.825500, 0.688135, 0.712863]
    mean=-0.109131, std=0.080229, score=0.080229, distance=0.233368



...

Function 6
  EI: [0.435974, 0.155305, 0.980959, 0.741320, 0.000001]
    mean=-0.320997, std=0.106607, score=0.018474, distance=0.304067
  UCB: [0.315066, 0.088161, 0.999999, 0.858979, 0.000001]
    mean=-0.423921, std=0.183622, score=-0.148488, distance=0.364431
  Mean: [0.437418, 0.243408, 0.755385, 0.771695, 0.000001]
    mean=-0.250749, std=0.011910, score=-0.250749, distance=0.064471
  Uncertainty: [0.043439, 0.016937, 0.177166, 0.034603, 0.121940]
    mean=-1.853542, std=0.548849, score=0.548849, distance=0.818037



/tmp/ipykernel_5596/3233374439.py:96: UserWarning: The balance properties of Sobol' points require n to be a power of 2.
  global_candidates = sampler.random(


Function 7
  EI: [0.159599, 0.261855, 0.382387, 0.156135, 0.374719, 0.602113]
    mean=2.381759, std=0.043423, score=0.064395, distance=0.147611
  UCB: [0.179141, 0.164710, 0.464734, 0.163371, 0.352783, 0.738366]
    mean=2.319243, std=0.116425, score=2.482238, distance=0.146253
  Mean: [0.159599, 0.261855, 0.382387, 0.156135, 0.374719, 0.602113]
    mean=2.381759, std=0.043423, score=2.381759, distance=0.147611
  Uncertainty: [0.990430, 0.291777, 0.115956, 0.525761, 0.980709, 0.790053]
    mean=0.377355, std=0.432414, score=0.432414, distance=0.571352



/tmp/ipykernel_5596/3233374439.py:96: UserWarning: The balance properties of Sobol' points require n to be a power of 2.
  global_candidates = sampler.random(


Function 8
  EI: [0.072463, 0.105056, 0.153306, 0.180922, 0.816113, 0.505056, 0.169492, 0.483786]
    mean=9.987245, std=0.021345, score=0.012886, distance=0.246665
  UCB: [0.027655, 0.157553, 0.178851, 0.155625, 0.826655, 0.478152, 0.099328, 0.549085]
    mean=9.963553, std=0.038027, score=10.032002, distance=0.253837
  Mean: [0.119777, 0.162664, 0.141891, 0.176185, 0.972008, 0.472671, 0.183211, 0.509322]
    mean=9.990336, std=0.009158, score=9.990336, distance=0.153987
  Uncertainty: [0.017805, 0.080791, 0.993815, 0.996435, 0.492926, 0.864269, 0.010321, 0.776084]
    mean=7.021905, std=0.244990, score=0.244990, distance=0.702334



## 59. Polish Round 8 Candidates

In [ ]:
from scipy.optimize import minimize

def polish_candidate(
    model,
    x_start,
    method,
    y_best,
    kappa,
):
    dimensions = len(x_start)

    def objective(x):
        x = np.asarray(x).reshape(1, -1)

        mean, std = model.predict(
            x,
            return_std=True,
        )

        mean = float(mean[0])
        std = max(float(std[0]), 1e-12)

        if method == "Mean":
            score = mean

        elif method == "UCB":
            score = mean + kappa * std

        elif method == "EI":
            improvement = mean - y_best
            z = improvement / std

            score = (
                improvement * norm.cdf(z)
                + std * norm.pdf(z)
            )

        else:
            raise ValueError("Unknown method")

        return -score

    result = minimize(
        objective,
        x0=x_start,
        method="L-BFGS-B",
        bounds=[(0.000001, 0.999999)] * dimensions,
        options={
            "maxiter": 500,
            "ftol": 1e-12,
        },
    )

    x = result.x

    mean, std = model.predict(
        x.reshape(1, -1),
        return_std=True,
    )

    return {
        "x": x,
        "mean": float(mean[0]),
        "std": float(std[0]),
        "success": result.success,
    }


round_8_kappa = {
    1: 2.8,
    2: 1.8,
    3: 2.8,
    4: 2.2,
    5: 1.2,
    6: 1.5,
    7: 1.4,
    8: 1.8,
}

round_8_polished = {}

for function_number in range(1, 9):
    model = round_8_models[function_number]
    y = datasets[function_number]["y"]
    y_best = np.max(y)

    round_8_polished[function_number] = {}

    print(f"Function {function_number}")

    for method in ["EI", "UCB", "Mean"]:
        x_start = (
            round_8_candidates[function_number][method]["x"]
        )

        result = polish_candidate(
            model=model,
            x_start=x_start,
            method=method,
            y_best=y_best,
            kappa=round_8_kappa[function_number],
        )

        round_8_polished[function_number][method] = result

        values = ", ".join(
            f"{value:.6f}"
            for value in result["x"]
        )

        print(
            f"  {method}: [{values}]"
        )
        print(
            f"    mean={result['mean']:.6f}, "
            f"std={result['std']:.6f}, "
            f"success={result['success']}"
        )

    print()

Function 1
  EI: [0.564024, 0.453006]
    mean=-0.000115, std=0.000837, success=True
  UCB: [0.076540, 0.942532]
    mean=-0.000154, std=0.000865, success=True
  Mean: [0.029524, 0.744614]
    mean=0.000176, std=0.000250, success=True

Function 2
  EI: [0.691826, 0.625824]
    mean=0.579225, std=0.042601, success=True
  UCB: [0.690496, 0.838771]
    mean=0.578809, std=0.042862, success=True
  Mean: [0.693780, 0.829166]
    mean=0.579452, std=0.042339, success=True

Function 3
  EI: [0.461000, 0.155883, 0.076278]
    mean=-0.053420, std=0.058405, success=True
  UCB: [0.719237, 0.424878, 0.048481]
    mean=-0.088161, std=0.076389, success=True
  Mean: [0.264381, 0.999999, 0.340586]
    mean=-0.034529, std=0.000938, success=True

Function 4
  EI: [0.366541, 0.393044, 0.418603, 0.428926]
    mean=0.160889, std=0.378806, success=True
  UCB: [0.362714, 0.388330, 0.417564, 0.430858]
    mean=0.151448, std=0.384009, success=False
  Mean: [0.370656, 0.398086, 0.419367, 0.427257]
    mean=0.1642

## 60. Function 5 Boundary Check

In [ ]:
f5_model = round_8_models[5]
f5_y_best = np.max(datasets[5]["y"])

x2_values = np.array([
    0.000001,
    0.001,
    0.005,
    0.01,
    0.025,
    0.05,
    0.10,
    0.15,
    0.25,
    0.50,
    0.75,
    0.999999,
])

f5_boundary_points = np.array([
    [0.999999, x2, 0.999999, 0.999999]
    for x2 in x2_values
])

mean, std = f5_model.predict(
    f5_boundary_points,
    return_std=True,
)

improvement = mean - f5_y_best
z = improvement / np.maximum(std, 1e-12)

ei = (
    improvement * norm.cdf(z)
    + std * norm.pdf(z)
)

ucb = mean + 1.2 * std

print(
    "x2        mean        std         EI          UCB"
)
print("-" * 60)

for i, x2 in enumerate(x2_values):
    print(
        f"{x2:0.6f}  "
        f"{mean[i]:10.3f}  "
        f"{std[i]:10.3f}  "
        f"{ei[i]:10.3f}  "
        f"{ucb[i]:10.3f}"
    )

x2        mean        std         EI          UCB
------------------------------------------------------------
0.000001    4438.198       9.194       4.659    4449.230
0.001000    4438.571       9.197       4.879    4449.608
0.005000    4440.065       9.303       5.854    4451.229
0.010000    4441.932       9.634       7.254    4453.492
0.025000    4447.522      11.703      12.225    4461.566
0.050000    4456.812      17.173      21.432    4477.419
0.100000    4475.283      30.424      40.369    4511.792
0.150000    4493.608      44.424      59.319    4546.917
0.250000    4529.810      72.852      96.889    4617.233
0.500000    4617.588     143.953     188.374    4790.331
0.750000    4701.279     214.440     276.100    4958.606
0.999999    4780.693     284.338     359.927    5121.899


## 61. Function 7 Local Refinement Check

In [ ]:
f7_model = round_8_models[7]

f7_best_index = np.argmax(
    datasets[7]["y"]
)

f7_best_x = datasets[7]["X"][
    f7_best_index
]

rng = np.random.default_rng(807)

radii = [
    0.02,
    0.04,
    0.06,
    0.08,
    0.12,
]

print("Current F7 best:")
print(
    np.round(f7_best_x, 6)
)
print()

for radius in radii:
    candidates = (
        f7_best_x
        + rng.normal(
            0,
            radius,
            size=(30000, 6),
        )
    )

    candidates = np.clip(
        candidates,
        0.000001,
        0.999999,
    )

    mean, std = f7_model.predict(
        candidates,
        return_std=True,
    )

    best_index = np.argmax(mean)

    best_candidate = candidates[
        best_index
    ]

    distance = np.linalg.norm(
        best_candidate - f7_best_x
    )

    values = ", ".join(
        f"{value:.6f}"
        for value in best_candidate
    )

    print(f"Radius {radius:.2f}")
    print(
        f"  Point: [{values}]"
    )
    print(
        f"  Mean: {mean[best_index]:.6f}"
    )
    print(
        f"  Std: {std[best_index]:.6f}"
    )
    print(
        f"  Distance: {distance:.6f}"
    )
    print()

Current F7 best:
[0.132125 0.267293 0.378521 0.165728 0.386801 0.746167]

Radius 0.02
  Point: [0.158216, 0.255147, 0.353046, 0.156217, 0.376682, 0.744626]
  Mean: 2.381943
  Std: 0.038655
  Distance: 0.040896

Radius 0.04
  Point: [0.160704, 0.297233, 0.359602, 0.153517, 0.374977, 0.789400]
  Mean: 2.381000
  Std: 0.045772
  Distance: 0.065032

Radius 0.06
  Point: [0.161958, 0.253259, 0.368259, 0.154350, 0.380229, 0.742814]
  Mean: 2.380727
  Std: 0.041416
  Distance: 0.037097

Radius 0.08
  Point: [0.160254, 0.262764, 0.396696, 0.157001, 0.377242, 0.615319]
  Mean: 2.381846
  Std: 0.040292
  Distance: 0.135760

Radius 0.12
  Point: [0.156449, 0.180591, 0.293473, 0.155928, 0.375053, 0.652440]
  Mean: 2.381384
  Std: 0.039535
  Distance: 0.156080



## 62. Round 8 Final Candidate Check

In [ ]:
round_8_shortlist = {
    1: np.array([
        0.029524,
        0.744614,
    ]),

    2: np.array([
        0.693780,
        0.763493,
    ]),

    3: np.array([
        0.461000,
        0.155883,
        0.076278,
    ]),

    4: np.array([
        0.366541,
        0.393044,
        0.418603,
        0.428926,
    ]),

    5: np.array([
        0.999999,
        0.000001,
        0.999999,
        0.999999,
    ]),

    6: np.array([
        0.434177,
        0.240282,
        0.760719,
        0.766188,
        0.000001,
    ]),

    7: np.array([
        0.158216,
        0.255147,
        0.353046,
        0.156217,
        0.376682,
        0.744626,
    ]),

    8: np.array([
        0.110172,
        0.142869,
        0.145572,
        0.150374,
        0.891367,
        0.490266,
        0.176693,
        0.670431,
    ]),
}

for function_number in range(1, 9):
    x = round_8_shortlist[function_number]
    model = round_8_models[function_number]
    X = datasets[function_number]["X"]
    y = datasets[function_number]["y"]

    mean, std = model.predict(
        x.reshape(1, -1),
        return_std=True,
    )

    distance = np.min(
        np.linalg.norm(
            X - x,
            axis=1,
        )
    )

    values = ", ".join(
        f"{value:.6f}"
        for value in x
    )

    print(f"Function {function_number}")
    print(f"  Point: [{values}]")
    print(f"  Current best: {np.max(y):.10g}")
    print(f"  Predicted mean: {mean[0]:.10g}")
    print(f"  Predicted std: {std[0]:.10g}")
    print(f"  Closest distance: {distance:.6f}")
    print()

Function 1
  Point: [0.029524, 0.744614]
  Current best: 7.710875115e-16
  Predicted mean: 0.0001755262714
  Predicted std: 0.0002497746615
  Closest distance: 0.290460

Function 2
  Point: [0.693780, 0.763493]
  Current best: 0.6122414108
  Predicted mean: 0.5794515275
  Predicted std: 0.04233925625
  Closest distance: 0.001093

Function 3
  Point: [0.461000, 0.155883, 0.076278]
  Current best: -0.03483531335
  Predicted mean: -0.05342037532
  Predicted std: 0.05840453888
  Closest distance: 0.266623

Function 4
  Point: [0.366541, 0.393044, 0.418603, 0.428926]
  Current best: 0.4575158799
  Predicted mean: 0.1608894676
  Predicted std: 0.3788061519
  Closest distance: 0.037692

Function 5
  Point: [0.999999, 0.000001, 0.999999, 0.999999]
  Current best: 4436.361461
  Predicted mean: 4438.197925
  Predicted std: 9.193684245
  Closest distance: 0.000198

Function 6
  Point: [0.434177, 0.240282, 0.760719, 0.766188, 0.000001]
  Current best: -0.2588092057
  Predicted mean: -0.2505338671


## 63. Select Round 8 Queries

In [ ]:
round_8_queries = {
    1: np.array([
        0.029524,
        0.744614,
    ]),

    2: np.array([
        0.693780,
        0.763493,
    ]),

    3: np.array([
        0.461000,
        0.155883,
        0.076278,
    ]),

    4: np.array([
        0.366541,
        0.393044,
        0.418603,
        0.428926,
    ]),

    5: np.array([
        0.999999,
        0.000001,
        0.999999,
        0.999999,
    ]),

    6: np.array([
        0.434177,
        0.240282,
        0.760719,
        0.766188,
        0.000001,
    ]),

    7: np.array([
        0.158216,
        0.255147,
        0.353046,
        0.156217,
        0.376682,
        0.744626,
    ]),

    8: np.array([
        0.110172,
        0.142869,
        0.145572,
        0.150374,
        0.891367,
        0.490266,
        0.176693,
        0.670431,
    ]),
}

print("ROUND 8 SUBMISSION VALUES\n")

for function_number in range(1, 9):
    values = "-".join(
        f"{value:.6f}"
        for value in round_8_queries[function_number]
    )

    print(
        f"Function {function_number}: "
        f"{values}"
    )

ROUND 8 SUBMISSION VALUES

Function 1: 0.029524-0.744614
Function 2: 0.693780-0.763493
Function 3: 0.461000-0.155883-0.076278
Function 4: 0.366541-0.393044-0.418603-0.428926
Function 5: 0.999999-0.000001-0.999999-0.999999
Function 6: 0.434177-0.240282-0.760719-0.766188-0.000001
Function 7: 0.158216-0.255147-0.353046-0.156217-0.376682-0.744626
Function 8: 0.110172-0.142869-0.145572-0.150374-0.891367-0.490266-0.176693-0.670431


## 64. Save Round 8 Queries

In [ ]:
import os
import json

week_8_dir = "/content/drive/MyDrive/Imperial_BBO/data/week_8"

os.makedirs(
    week_8_dir,
    exist_ok=True,
)

round_8_queries_to_save = {
    f"Function {function_number}": [
        round(float(value), 6)
        for value in round_8_queries[function_number]
    ]
    for function_number in range(1, 9)
}

save_path = os.path.join(
    week_8_dir,
    "submitted_queries.json",
)

with open(save_path, "w") as file:
    json.dump(
        round_8_queries_to_save,
        file,
        indent=4,
    )

print(f"Saved to: {save_path}")

Saved to: /content/drive/MyDrive/Imperial_BBO/data/week_8/submitted_queries.json


## 65. Load Round 8 Returned Results

In [ ]:
round_8_results = {
    1: {
        "x": [0.029524, 0.744614],
        "y": -4.622359582167358e-180,
    },
    2: {
        "x": [0.693780, 0.763493],
        "y": 0.6265602258902787,
    },
    3: {
        "x": [0.461000, 0.155883, 0.076278],
        "y": -0.052361033676792776,
    },
    4: {
        "x": [0.366541, 0.393044, 0.418603, 0.428926],
        "y": 0.5657943181247842,
    },
    5: {
        "x": [0.999999, 0.000001, 0.999999, 0.999999],
        "y": 4440.480873839292,
    },
    6: {
        "x": [0.434177, 0.240282, 0.760719, 0.766188, 0.000001],
        "y": -0.3446779436496158,
    },
    7: {
        "x": [0.158216, 0.255147, 0.353046, 0.156217, 0.376682, 0.744626],
        "y": 2.32905184218897,
    },
    8: {
        "x": [0.110172, 0.142869, 0.145572, 0.150374, 0.891367, 0.490266, 0.176693, 0.670431],
        "y": 9.9931634080684,
    },
}

print("Round 8 results loaded.")

Round 8 results loaded.


## 66. Update Datasets with Round 8 Results

In [ ]:
for function_number, result in round_8_results.items():
    x_new = np.array(result["x"]).reshape(1, -1)
    y_new = np.array([result["y"]])

    datasets[function_number]["X"] = np.vstack(
        [datasets[function_number]["X"], x_new]
    )

    datasets[function_number]["y"] = np.concatenate(
        [datasets[function_number]["y"], y_new]
    )

print("Round 8 results added to datasets.")

for function_number in range(1, 9):
    print(
        f"Function {function_number}: "
        f"{len(datasets[function_number]['y'])} observations"
    )

Round 8 results added to datasets.
Function 1: 18 observations
Function 2: 18 observations
Function 3: 23 observations
Function 4: 38 observations
Function 5: 28 observations
Function 6: 28 observations
Function 7: 38 observations
Function 8: 48 observations


## 67. Round 8 Diagnostics

In [ ]:
for function_number in range(1, 9):
    X = datasets[function_number]["X"]
    y = datasets[function_number]["y"]

    previous_y = y[:-1]
    round_8_y = y[-1]

    previous_best = np.max(previous_y)
    current_best = np.max(y)

    improved = round_8_y > previous_best

    x_round_8 = X[-1]
    previous_X = X[:-1]

    closest_distance = np.min(
        np.linalg.norm(previous_X - x_round_8, axis=1)
    )

    print(f"Function {function_number}")
    print(f"  Previous best: {previous_best:.10g}")
    print(f"  Round 8 result: {round_8_y:.10g}")
    print(f"  Current best: {current_best:.10g}")
    print(f"  Improved: {improved}")
    print(f"  Closest previous distance: {closest_distance:.6f}")
    print()

Function 1
  Previous best: 7.710875115e-16
  Round 8 result: -4.622359582e-180
  Current best: 7.710875115e-16
  Improved: False
  Closest previous distance: 0.290460

Function 2
  Previous best: 0.6122414108
  Round 8 result: 0.6265602259
  Current best: 0.6265602259
  Improved: True
  Closest previous distance: 0.001093

Function 3
  Previous best: -0.03483531335
  Round 8 result: -0.05236103368
  Current best: -0.03483531335
  Improved: False
  Closest previous distance: 0.266623

Function 4
  Previous best: 0.4575158799
  Round 8 result: 0.5657943181
  Current best: 0.5657943181
  Improved: True
  Closest previous distance: 0.037692

Function 5
  Previous best: 4436.361461
  Round 8 result: 4440.480874
  Current best: 4440.480874
  Improved: True
  Closest previous distance: 0.000198

Function 6
  Previous best: -0.2588092057
  Round 8 result: -0.3446779436
  Current best: -0.2588092057
  Improved: False
  Closest previous distance: 0.070052

Function 7
  Previous best: 2.31878858

## 68. Install and Import BoTorch for Round 9

In [ ]:
!pip -q install botorch gpytorch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.0/6.0 MB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 291.2/291.2 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 174.8/174.8 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.4/183.4 kB 13.3 MB/s eta 0:00:00


In [ ]:
import torch

from botorch.models import SingleTaskGP
from botorch.fit import fit_gpytorch_mll
from botorch.acquisition.analytic import LogExpectedImprovement
from botorch.optim import optimize_acqf

from gpytorch.mlls import ExactMarginalLogLikelihood

torch.set_default_dtype(torch.double)

print("BoTorch ready.")

BoTorch ready.


## 69. Fit BoTorch Gaussian Process Models for Round 9

In [ ]:
round_9_models = {}

for function_number in range(1, 9):
    X_np = datasets[function_number]["X"]
    y_np = datasets[function_number]["y"]

    train_X = torch.tensor(
        X_np,
        dtype=torch.double,
    )

    train_Y = torch.tensor(
        y_np,
        dtype=torch.double,
    ).unsqueeze(-1)

    model = SingleTaskGP(
        train_X,
        train_Y,
    )

    mll = ExactMarginalLogLikelihood(
        model.likelihood,
        model,
    )

    fit_gpytorch_mll(mll)

    round_9_models[function_number] = model

    print(
        f"Function {function_number}: "
        f"model fitted with {len(y_np)} observations"
    )

Function 1: model fitted with 18 observations
Function 2: model fitted with 18 observations
Function 3: model fitted with 23 observations
Function 4: model fitted with 38 observations
Function 5: model fitted with 28 observations
Function 6: model fitted with 28 observations
Function 7: model fitted with 38 observations
Function 8: model fitted with 48 observations


## 70. Round 8 Predictive Calibration Check with BoTorch

In [ ]:
round_8_botorch_calibration = {}

for function_number in range(1, 9):
    X_np = datasets[function_number]["X"]
    y_np = datasets[function_number]["y"]

    train_X = torch.tensor(
        X_np[:-1],
        dtype=torch.double,
    )

    train_Y = torch.tensor(
        y_np[:-1],
        dtype=torch.double,
    ).unsqueeze(-1)

    test_X = torch.tensor(
        X_np[-1],
        dtype=torch.double,
    ).unsqueeze(0)

    actual_y = y_np[-1]

    model = SingleTaskGP(
        train_X,
        train_Y,
    )

    mll = ExactMarginalLogLikelihood(
        model.likelihood,
        model,
    )

    fit_gpytorch_mll(mll)

    model.eval()

    with torch.no_grad():
        posterior = model.posterior(test_X)

        predicted_mean = posterior.mean.item()
        predicted_std = posterior.variance.sqrt().item()

    error = actual_y - predicted_mean

    z_score = (
        error / predicted_std
        if predicted_std > 0
        else np.nan
    )

    round_8_botorch_calibration[function_number] = {
        "predicted_mean": predicted_mean,
        "predicted_std": predicted_std,
        "actual": actual_y,
        "error": error,
        "z_score": z_score,
    }

    print(f"Function {function_number}")
    print(f"  Predicted mean: {predicted_mean:.10g}")
    print(f"  Predicted std: {predicted_std:.10g}")
    print(f"  Actual: {actual_y:.10g}")
    print(f"  Error: {error:.10g}")
    print(f"  Z-score: {z_score:.3f}")
    print()

Function 1
  Predicted mean: -0.0001948877782
  Predicted std: 0.0008745984318
  Actual: -4.622359582e-180
  Error: 0.0001948877782
  Z-score: 0.223

Function 2
  Predicted mean: 0.5856811478
  Predicted std: 0.01585512761
  Actual: 0.6265602259
  Error: 0.04087907812
  Z-score: 2.578

Function 3
  Predicted mean: -0.1281961443
  Predicted std: 0.03463549869
  Actual: -0.05236103368
  Error: 0.07583511066
  Z-score: 2.190

Function 4
  Predicted mean: 0.05270967149
  Predicted std: 0.1793808688
  Actual: 0.5657943181
  Error: 0.5130846466
  Z-score: 2.860

Function 5
  Predicted mean: 4425.361006
  Predicted std: 47.19008149
  Actual: 4440.480874
  Error: 15.11986769
  Z-score: 0.320

Function 6
  Predicted mean: -0.3081630692
  Predicted std: 0.04605163566
  Actual: -0.3446779436
  Error: -0.03651487441
  Z-score: -0.793

Function 7
  Predicted mean: 2.319958026
  Predicted std: 0.06053576324
  Actual: 2.329051842
  Error: 0.009093816165
  Z-score: 0.150

Function 8
  Predicted mean: 

## 71. Generate Global and Local LogEI Candidates for Round 9

In [ ]:
round_9_candidates = {}

trust_region_radius = {
    1: 0.30,
    2: 0.08,
    3: 0.30,
    4: 0.08,
    5: 0.02,
    6: 0.18,
    7: 0.06,
    8: 0.08,
}

for function_number in range(1, 9):
    model = round_9_models[function_number]

    X_np = datasets[function_number]["X"]
    y_np = datasets[function_number]["y"]

    best_index = np.argmax(y_np)
    incumbent = torch.tensor(
        X_np[best_index],
        dtype=torch.double,
    )

    best_f = float(np.max(y_np))

    acquisition = LogExpectedImprovement(
        model=model,
        best_f=best_f,
    )

    dimension = X_np.shape[1]

    global_bounds = torch.stack([
        torch.full(
            (dimension,),
            0.000001,
            dtype=torch.double,
        ),
        torch.full(
            (dimension,),
            0.999999,
            dtype=torch.double,
        ),
    ])

    radius = trust_region_radius[function_number]

    local_lower = torch.clamp(
        incumbent - radius,
        min=0.000001,
        max=0.999999,
    )

    local_upper = torch.clamp(
        incumbent + radius,
        min=0.000001,
        max=0.999999,
    )

    local_bounds = torch.stack([
        local_lower,
        local_upper,
    ])

    global_candidate, global_value = optimize_acqf(
        acq_function=acquisition,
        bounds=global_bounds,
        q=1,
        num_restarts=20,
        raw_samples=1024,
    )

    local_candidate, local_value = optimize_acqf(
        acq_function=acquisition,
        bounds=local_bounds,
        q=1,
        num_restarts=20,
        raw_samples=1024,
    )

    global_candidate = global_candidate.squeeze(0)
    local_candidate = local_candidate.squeeze(0)

    with torch.no_grad():
        global_posterior = model.posterior(
            global_candidate.unsqueeze(0)
        )

        local_posterior = model.posterior(
            local_candidate.unsqueeze(0)
        )

    global_mean = global_posterior.mean.item()
    global_std = global_posterior.variance.sqrt().item()

    local_mean = local_posterior.mean.item()
    local_std = local_posterior.variance.sqrt().item()

    global_np = global_candidate.detach().cpu().numpy()
    local_np = local_candidate.detach().cpu().numpy()

    global_distance = np.min(
        np.linalg.norm(
            X_np - global_np,
            axis=1,
        )
    )

    local_distance = np.min(
        np.linalg.norm(
            X_np - local_np,
            axis=1,
        )
    )

    round_9_candidates[function_number] = {
        "global": global_np,
        "local": local_np,
        "global_logei": global_value.item(),
        "local_logei": local_value.item(),
        "global_mean": global_mean,
        "global_std": global_std,
        "local_mean": local_mean,
        "local_std": local_std,
        "global_distance": global_distance,
        "local_distance": local_distance,
    }

    print(f"Function {function_number}")

    print(
        "  Global:",
        np.round(global_np, 6),
    )
    print(
        f"    Mean: {global_mean:.10g}"
    )
    print(
        f"    Std: {global_std:.10g}"
    )
    print(
        f"    LogEI: {global_value.item():.10g}"
    )
    print(
        f"    Closest distance: {global_distance:.6f}"
    )

    print(
        "  Local:",
        np.round(local_np, 6),
    )
    print(
        f"    Mean: {local_mean:.10g}"
    )
    print(
        f"    Std: {local_std:.10g}"
    )
    print(
        f"    LogEI: {local_value.item():.10g}"
    )
    print(
        f"    Closest distance: {local_distance:.6f}"
    )

    print()

Function 1
  Global: [0.732111 0.86184 ]
    Mean: 0.0001559741125
    Std: 0.0007047792939
    LogEI: -7.912847734
    Closest distance: 0.048699
  Local: [0.732111 0.86184 ]
    Mean: 0.0001559741624
    Std: 0.0007047792186
    LogEI: -7.912847734
    Closest distance: 0.048699

Function 2
  Global: [0.681299 0.999999]
    Mean: 0.5907930612
    Std: 0.02888337243
    LogEI: -6.50423229
    Closest distance: 0.057444
  Local: [0.678134 0.843493]
    Mean: 0.5826518253
    Std: 0.02697830561
    LogEI: -7.439761009
    Closest distance: 0.023950

Function 3
  Global: [0.315488 0.627886 0.044924]
    Mean: -0.01530108563
    Std: 0.02845217793
    LogEI: -3.742585611
    Closest distance: 0.191512
  Local: [0.315488 0.627886 0.044924]
    Mean: -0.0153010862
    Std: 0.02845217929
    LogEI: -3.742585611
    Closest distance: 0.191512

Function 4
  Global: [0.384409 0.376347 0.403855 0.445891]
    Mean: 0.09440079085
    Std: 0.1854237824
    LogEI: -8.029323089
    Closest distance: 

## 72. Targeted Round 9 Candidate Checks

In [ ]:
# Function 2: keep the successful x2 value and scan x1 nearby

f2_model = round_9_models[2]
f2_x2 = 0.763493

f2_x1_values = np.linspace(
    0.64,
    0.72,
    17,
)

print("FUNCTION 2")
print("x1        mean        std")

for x1 in f2_x1_values:
    x = torch.tensor(
        [[x1, f2_x2]],
        dtype=torch.double,
    )

    with torch.no_grad():
        posterior = f2_model.posterior(x)

    mean = posterior.mean.item()
    std = posterior.variance.sqrt().item()

    print(
        f"{x1:.6f}  "
        f"{mean:.9f}  "
        f"{std:.9f}"
    )


# Function 4: small perturbations around the new incumbent

f4_model = round_9_models[4]

f4_best_index = np.argmax(
    datasets[4]["y"]
)

f4_incumbent = datasets[4]["X"][
    f4_best_index
]

rng = np.random.default_rng(904)

f4_candidates = np.clip(
    f4_incumbent
    + rng.normal(
        0.0,
        0.025,
        size=(20000, 4),
    ),
    0.000001,
    0.999999,
)

f4_tensor = torch.tensor(
    f4_candidates,
    dtype=torch.double,
)

with torch.no_grad():
    posterior = f4_model.posterior(
        f4_tensor
    )

f4_means = (
    posterior.mean
    .squeeze(-1)
    .cpu()
    .numpy()
)

f4_stds = (
    posterior.variance
    .sqrt()
    .squeeze(-1)
    .cpu()
    .numpy()
)

f4_best_candidate_index = np.argmax(
    f4_means
)

print("\nFUNCTION 4")
print(
    "Incumbent:",
    np.round(
        f4_incumbent,
        6,
    ),
)

print(
    "Best local candidate:",
    np.round(
        f4_candidates[
            f4_best_candidate_index
        ],
        6,
    ),
)

print(
    "Mean:",
    f"{f4_means[f4_best_candidate_index]:.10g}",
)

print(
    "Std:",
    f"{f4_stds[f4_best_candidate_index]:.10g}",
)


# Function 5: scan x2 with the other three coordinates at the boundary

f5_model = round_9_models[5]

f5_x2_values = [
    0.000001,
    0.005,
    0.010,
    0.020,
    0.050,
    0.100,
    0.250,
    0.500,
    0.750,
    0.999999,
]

print("\nFUNCTION 5")
print("x2        mean          std")

for x2 in f5_x2_values:
    x = torch.tensor(
        [[
            0.999999,
            x2,
            0.999999,
            0.999999,
        ]],
        dtype=torch.double,
    )

    with torch.no_grad():
        posterior = f5_model.posterior(x)

    mean = posterior.mean.item()
    std = posterior.variance.sqrt().item()

    print(
        f"{x2:.6f}  "
        f"{mean:.6f}  "
        f"{std:.6f}"
    )


# Function 8: compare the global/local candidates while preserving incumbent x8

f8_model = round_9_models[8]

f8_best_index = np.argmax(
    datasets[8]["y"]
)

f8_incumbent = datasets[8]["X"][
    f8_best_index
]

f8_incumbent_x8 = (
    f8_incumbent[-1]
)

f8_global_fixed = (
    round_9_candidates[8]["global"].copy()
)

f8_local_fixed = (
    round_9_candidates[8]["local"].copy()
)

f8_global_fixed[-1] = (
    f8_incumbent_x8
)

f8_local_fixed[-1] = (
    f8_incumbent_x8
)

print("\nFUNCTION 8")

for name, candidate in [
    ("Global with incumbent x8", f8_global_fixed),
    ("Local with incumbent x8", f8_local_fixed),
]:
    x = torch.tensor(
        candidate,
        dtype=torch.double,
    ).unsqueeze(0)

    with torch.no_grad():
        posterior = f8_model.posterior(x)

    mean = posterior.mean.item()
    std = posterior.variance.sqrt().item()

    print(name)
    print(
        "  Point:",
        np.round(
            candidate,
            6,
        ),
    )
    print(
        f"  Mean: {mean:.10g}"
    )
    print(
        f"  Std: {std:.10g}"
    )

FUNCTION 2
x1        mean        std
0.640000  0.414651254  0.060745039
0.645000  0.445385666  0.057287382
0.650000  0.473992552  0.053144871
0.655000  0.500117333  0.048472896
0.660000  0.523449297  0.043432814
0.665000  0.543725773  0.038190868
0.670000  0.560735077  0.032920417
0.675000  0.574318255  0.027809245
0.680000  0.584369747  0.023074959
0.685000  0.590837078  0.018990652
0.690000  0.593719756  0.015907180
0.695000  0.593067544  0.014197482
0.700000  0.588978283  0.014017073
0.705000  0.581595413  0.015095823
0.710000  0.571105340  0.016949434
0.715000  0.557734714  0.019205296
0.720000  0.541747675  0.021685836

FUNCTION 4
Incumbent: [0.366541 0.393044 0.418603 0.428926]
Best local candidate: [0.381816 0.398119 0.412318 0.431892]
Mean: 0.1468644093
Std: 0.1582362599

FUNCTION 5
x2        mean          std
0.000001  4434.875962  30.703447
0.005000  4436.708450  30.774777
0.010000  4438.533316  30.971234
0.020000  4442.159009  31.728534
0.050000  4452.842549  36.532882
0.100

## 73. Final Round 9 Candidate Refinement Checks

In [ ]:
# Function 2: scan x2 while keeping x1 near the successful value

f2_model = round_9_models[2]

f2_x2_values = np.linspace(
    0.70,
    0.999999,
    16,
)

print("FUNCTION 2")
print("x2        mean        std")

for x2 in f2_x2_values:
    x = torch.tensor(
        [[0.693780, x2]],
        dtype=torch.double,
    )

    with torch.no_grad():
        posterior = f2_model.posterior(x)

    mean = posterior.mean.item()
    std = posterior.variance.sqrt().item()

    print(
        f"{x2:.6f}  "
        f"{mean:.9f}  "
        f"{std:.9f}"
    )


# Functions 1, 3, 6 and 7:
# compare current incumbent, global LogEI and local LogEI candidates

for function_number in [1, 3, 6, 7]:
    model = round_9_models[function_number]

    X_np = datasets[function_number]["X"]
    y_np = datasets[function_number]["y"]

    best_index = np.argmax(y_np)
    incumbent = X_np[best_index]

    global_candidate = round_9_candidates[
        function_number
    ]["global"]

    local_candidate = round_9_candidates[
        function_number
    ]["local"]

    print(f"\nFUNCTION {function_number}")

    for name, candidate in [
        ("Incumbent", incumbent),
        ("Global LogEI", global_candidate),
        ("Local LogEI", local_candidate),
    ]:
        x = torch.tensor(
            candidate,
            dtype=torch.double,
        ).unsqueeze(0)

        with torch.no_grad():
            posterior = model.posterior(x)

        mean = posterior.mean.item()
        std = posterior.variance.sqrt().item()

        distance = np.min(
            np.linalg.norm(
                X_np - candidate,
                axis=1,
            )
        )

        print(name)
        print(
            "  Point:",
            np.round(candidate, 6),
        )
        print(
            f"  Mean: {mean:.10g}"
        )
        print(
            f"  Std: {std:.10g}"
        )
        print(
            f"  Closest distance: {distance:.6f}"
        )

FUNCTION 2
x2        mean        std
0.700000  0.591818241  0.013669676
0.720000  0.592377654  0.013878444
0.740000  0.592923769  0.014128646
0.760000  0.593456537  0.014418324
0.780000  0.593975906  0.014745357
0.800000  0.594481828  0.015107526
0.820000  0.594974256  0.015502578
0.840000  0.595453143  0.015928278
0.859999  0.595918444  0.016382449
0.879999  0.596370114  0.016863005
0.899999  0.596808112  0.017367969
0.919999  0.597232396  0.017895488
0.939999  0.597642925  0.018443839
0.959999  0.598039660  0.019011432
0.979999  0.598422563  0.019596803
0.999999  0.598791598  0.020198618

FUNCTION 1
Incumbent
  Point: [0.731024 0.733   ]
  Mean: -2.946473963e-06
  Std: 6.936121221e-05
  Closest distance: 0.000000
Global LogEI
  Point: [0.732111 0.86184 ]
  Mean: 0.0001559741125
  Std: 0.0007047792939
  Closest distance: 0.048699
Local LogEI
  Point: [0.732111 0.86184 ]
  Mean: 0.0001559741624
  Std: 0.0007047792186
  Closest distance: 0.048699

FUNCTION 3
Incumbent
  Point: [0.492581

## 74. Final F4 and F5 Risk Check

In [ ]:
# Function 4: directional empirical candidates around the Round 8 winner

f4_candidates = np.array([
    [0.366541, 0.393044, 0.418603, 0.428926],
    [0.356541, 0.403044, 0.408603, 0.428926],
    [0.346541, 0.413044, 0.398603, 0.428926],
    [0.361541, 0.398044, 0.413603, 0.428926],
    [0.371541, 0.388044, 0.423603, 0.428926],
])

print("FUNCTION 4")

for candidate in f4_candidates:
    x = torch.tensor(
        candidate,
        dtype=torch.double,
    ).unsqueeze(0)

    with torch.no_grad():
        posterior = round_9_models[4].posterior(x)

    mean = posterior.mean.item()
    std = posterior.variance.sqrt().item()

    print(
        np.round(candidate, 6),
        f"mean={mean:.10g}",
        f"std={std:.10g}",
    )


# Function 5: conservative x2 steps

f5_candidates = np.array([
    [0.999999, 0.000001, 0.999999, 0.999999],
    [0.999999, 0.025000, 0.999999, 0.999999],
    [0.999999, 0.050000, 0.999999, 0.999999],
    [0.999999, 0.075000, 0.999999, 0.999999],
    [0.999999, 0.100000, 0.999999, 0.999999],
])

print("\nFUNCTION 5")

for candidate in f5_candidates:
    x = torch.tensor(
        candidate,
        dtype=torch.double,
    ).unsqueeze(0)

    with torch.no_grad():
        posterior = round_9_models[5].posterior(x)

    mean = posterior.mean.item()
    std = posterior.variance.sqrt().item()

    print(
        np.round(candidate, 6),
        f"mean={mean:.10g}",
        f"std={std:.10g}",
    )

FUNCTION 4
[0.366541 0.393044 0.418603 0.428926] mean=0.132841797 std=0.1643225251
[0.356541 0.403044 0.408603 0.428926] mean=0.1099726405 std=0.1664468964
[0.346541 0.413044 0.398603 0.428926] mean=0.04944711845 std=0.1768886127
[0.361541 0.398044 0.413603 0.428926] mean=0.1261248059 std=0.1642762617
[0.371541 0.388044 0.423603 0.428926] mean=0.1301105459 std=0.1666062732

FUNCTION 5
[9.99999e-01 1.00000e-06 9.99999e-01 9.99999e-01] mean=4434.875962 std=30.70344723
[0.999999 0.025    0.999999 0.999999] mean=4443.959802 std=32.28071689
[0.999999 0.05     0.999999 0.999999] mean=4452.842549 std=36.53288194
[0.999999 0.075    0.999999 0.999999] mean=4461.521724 std=42.66165298
[0.999999 0.1      0.999999 0.999999] mean=4469.995261 std=49.97574425


## 75. Round 9 Final Candidate Check

In [ ]:
round_9_shortlist = {
    1: np.array([
        0.732111,
        0.861840,
    ]),

    2: np.array([
        0.693780,
        0.999999,
    ]),

    3: np.array([
        0.315488,
        0.627886,
        0.044924,
    ]),

    4: np.array([
        0.371541,
        0.388044,
        0.423603,
        0.428926,
    ]),

    5: np.array([
        0.999999,
        0.100000,
        0.999999,
        0.999999,
    ]),

    6: np.array([
        0.475080,
        0.386463,
        0.693187,
        0.684341,
        0.000001,
    ]),

    7: np.array([
        0.179550,
        0.187389,
        0.656015,
        0.218688,
        0.429470,
        0.783615,
    ]),

    8: np.array([
        0.161692,
        0.158351,
        0.172309,
        0.145397,
        0.849666,
        0.530329,
        0.209785,
        0.670431,
    ]),
}

for function_number in range(1, 9):
    candidate = round_9_shortlist[
        function_number
    ]

    model = round_9_models[
        function_number
    ]

    X_np = datasets[
        function_number
    ]["X"]

    y_np = datasets[
        function_number
    ]["y"]

    x = torch.tensor(
        candidate,
        dtype=torch.double,
    ).unsqueeze(0)

    with torch.no_grad():
        posterior = model.posterior(x)

    mean = posterior.mean.item()
    std = posterior.variance.sqrt().item()

    closest_distance = np.min(
        np.linalg.norm(
            X_np - candidate,
            axis=1,
        )
    )

    values = ", ".join(
        f"{value:.6f}"
        for value in candidate
    )

    print(
        f"Function {function_number}"
    )
    print(
        f"  Point: [{values}]"
    )
    print(
        f"  Current best: {np.max(y_np):.10g}"
    )
    print(
        f"  Predicted mean: {mean:.10g}"
    )
    print(
        f"  Predicted std: {std:.10g}"
    )
    print(
        f"  Closest distance: {closest_distance:.6f}"
    )
    print()

Function 1
  Point: [0.732111, 0.861840]
  Current best: 7.710875115e-16
  Predicted mean: 0.0001559751184
  Predicted std: 0.0007047777757
  Closest distance: 0.048699

Function 2
  Point: [0.693780, 0.999999]
  Current best: 0.6265602259
  Predicted mean: 0.5987915984
  Predicted std: 0.02019861796
  Closest distance: 0.044963

Function 3
  Point: [0.315488, 0.627886, 0.044924]
  Current best: -0.03483531335
  Predicted mean: -0.01530109164
  Predicted std: 0.02845219229
  Closest distance: 0.191511

Function 4
  Point: [0.371541, 0.388044, 0.423603, 0.428926]
  Current best: 0.5657943181
  Predicted mean: 0.1301105459
  Predicted std: 0.1666062732
  Closest distance: 0.008660

Function 5
  Point: [0.999999, 0.100000, 0.999999, 0.999999]
  Current best: 4440.480874
  Predicted mean: 4469.995261
  Predicted std: 49.97574425
  Closest distance: 0.099900

Function 6
  Point: [0.475080, 0.386463, 0.693187, 0.684341, 0.000001]
  Current best: -0.2588092057
  Predicted mean: -0.2722007843


## 76. Select and Save Round 9 Queries

In [ ]:
import os
import json

round_9_queries = {
    function_number: round_9_shortlist[function_number]
    for function_number in range(1, 9)
}

week_9_dir = "/content/drive/MyDrive/Imperial_BBO/data/Week_9"

round_9_queries_to_save = {
    f"Function {function_number}": [
        round(float(value), 6)
        for value in round_9_queries[function_number]
    ]
    for function_number in range(1, 9)
}

save_path = os.path.join(
    week_9_dir,
    "submitted_queries.json",
)

with open(save_path, "w") as file:
    json.dump(
        round_9_queries_to_save,
        file,
        indent=4,
    )

print(f"Saved to: {save_path}")
print("\nROUND 9 SUBMISSION VALUES\n")

for function_number in range(1, 9):
    values = "-".join(
        f"{value:.6f}"
        for value in round_9_queries[function_number]
    )

    print(
        f"Function {function_number}: "
        f"{values}"
    )

Saved to: /content/drive/MyDrive/Imperial_BBO/data/Week_9/submitted_queries.json

ROUND 9 SUBMISSION VALUES

Function 1: 0.732111-0.861840
Function 2: 0.693780-0.999999
Function 3: 0.315488-0.627886-0.044924
Function 4: 0.371541-0.388044-0.423603-0.428926
Function 5: 0.999999-0.100000-0.999999-0.999999
Function 6: 0.475080-0.386463-0.693187-0.684341-0.000001
Function 7: 0.179550-0.187389-0.656015-0.218688-0.429470-0.783615
Function 8: 0.161692-0.158351-0.172309-0.145397-0.849666-0.530329-0.209785-0.670431


## 77. Load Round 9 Returned Results

In [ ]:
round_9_results = {
    1: {
        "x": [0.732111, 0.861840],
        "y": 5.70089734000831e-46,
    },
    2: {
        "x": [0.693780, 0.999999],
        "y": 0.6301453244913451,
    },
    3: {
        "x": [0.315488, 0.627886, 0.044924],
        "y": -0.04033029445506345,
    },
    4: {
        "x": [0.371541, 0.388044, 0.423603, 0.428926],
        "y": 0.5148533165592686,
    },
    5: {
        "x": [0.999999, 0.100000, 0.999999, 0.999999],
        "y": 4441.147510279543,
    },
    6: {
        "x": [0.475080, 0.386463, 0.693187, 0.684341, 0.000001],
        "y": -0.30027805326318,
    },
    7: {
        "x": [0.179550, 0.187389, 0.656015, 0.218688, 0.429470, 0.783615],
        "y": 2.144084424164344,
    },
    8: {
        "x": [0.161692, 0.158351, 0.172309, 0.145397, 0.849666, 0.530329, 0.209785, 0.670431],
        "y": 9.9840863639739,
    },
}

print("Round 9 results loaded.")

Round 9 results loaded.


## 78. Update Datasets with Round 9 Results

In [ ]:
for function_number, result in round_9_results.items():
    x_new = np.array(result["x"]).reshape(1, -1)
    y_new = np.array([result["y"]])

    datasets[function_number]["X"] = np.vstack(
        [datasets[function_number]["X"], x_new]
    )

    datasets[function_number]["y"] = np.concatenate(
        [datasets[function_number]["y"], y_new]
    )

print("Round 9 results added to datasets.")

for function_number in range(1, 9):
    print(
        f"Function {function_number}: "
        f"{len(datasets[function_number]['y'])} observations"
    )

Round 9 results added to datasets.
Function 1: 19 observations
Function 2: 19 observations
Function 3: 24 observations
Function 4: 39 observations
Function 5: 29 observations
Function 6: 29 observations
Function 7: 39 observations
Function 8: 49 observations


## 79. Round 9 Diagnostics

In [ ]:
for function_number in range(1, 9):
    X = datasets[function_number]["X"]
    y = datasets[function_number]["y"]

    previous_X = X[:-1]
    previous_y = y[:-1]

    round_9_x = X[-1]
    round_9_y = y[-1]

    previous_best = np.max(previous_y)
    current_best = np.max(y)

    improved = round_9_y > previous_best

    closest_distance = np.min(
        np.linalg.norm(
            previous_X - round_9_x,
            axis=1,
        )
    )

    print(f"Function {function_number}")
    print(f"  Previous best: {previous_best:.10g}")
    print(f"  Round 9 result: {round_9_y:.10g}")
    print(f"  Current best: {current_best:.10g}")
    print(f"  Improved: {improved}")
    print(f"  Closest previous distance: {closest_distance:.6f}")
    print()

Function 1
  Previous best: 7.710875115e-16
  Round 9 result: 5.70089734e-46
  Current best: 7.710875115e-16
  Improved: False
  Closest previous distance: 0.048699

Function 2
  Previous best: 0.6265602259
  Round 9 result: 0.6301453245
  Current best: 0.6301453245
  Improved: True
  Closest previous distance: 0.044963

Function 3
  Previous best: -0.03483531335
  Round 9 result: -0.04033029446
  Current best: -0.03483531335
  Improved: False
  Closest previous distance: 0.191511

Function 4
  Previous best: 0.5657943181
  Round 9 result: 0.5148533166
  Current best: 0.5657943181
  Improved: False
  Closest previous distance: 0.008660

Function 5
  Previous best: 4440.480874
  Round 9 result: 4441.14751
  Current best: 4441.14751
  Improved: True
  Closest previous distance: 0.099900

Function 6
  Previous best: -0.2588092057
  Round 9 result: -0.3002780533
  Current best: -0.2588092057
  Improved: False
  Closest previous distance: 0.104826

Function 7
  Previous best: 2.329051842
  

## 80. Multi-Surrogate Validation Setup for Round 10

In [ ]:
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, Matern, WhiteKernel
from scipy.interpolate import RBFInterpolator
from scipy.stats import spearmanr


def fit_gp(X, y, random_state=0):
    dimension = X.shape[1]

    kernel = (
        ConstantKernel(
            1.0,
            (1e-3, 1e3),
        )
        * Matern(
            length_scale=np.ones(dimension),
            length_scale_bounds=(1e-4, 1e4),
            nu=2.5,
        )
        + WhiteKernel(
            noise_level=1e-8,
            noise_level_bounds=(1e-10, 1e-1),
        )
    )

    model = GaussianProcessRegressor(
        kernel=kernel,
        normalize_y=True,
        n_restarts_optimizer=5,
        random_state=random_state,
    )

    model.fit(X, y)

    return model


def predict_gp(X_train, y_train, X_test, random_state=0):
    model = fit_gp(
        X_train,
        y_train,
        random_state=random_state,
    )

    mean, std = model.predict(
        X_test,
        return_std=True,
    )

    return mean, std


def predict_rbf(X_train, y_train, X_test):
    y_mean = np.mean(y_train)
    y_std = np.std(y_train)

    if y_std < 1e-12:
        y_std = 1.0

    y_scaled = (
        y_train - y_mean
    ) / y_std

    model = RBFInterpolator(
        X_train,
        y_scaled,
        kernel="thin_plate_spline",
        smoothing=1e-8,
    )

    prediction_scaled = model(
        X_test
    )

    prediction = (
        prediction_scaled * y_std
        + y_mean
    )

    return prediction


def predict_extra_trees(
    X_train,
    y_train,
    X_test,
    random_state=0,
):
    model = ExtraTreesRegressor(
        n_estimators=500,
        min_samples_leaf=1,
        max_features=1.0,
        random_state=random_state,
        n_jobs=-1,
    )

    model.fit(
        X_train,
        y_train,
    )

    prediction = model.predict(
        X_test
    )

    return prediction


print("Round 10 surrogate validation tools ready.")

Round 10 surrogate validation tools ready.


## 81. Rolling Historical Surrogate Validation

In [ ]:
initial_observation_counts = {
    1: 10,
    2: 10,
    3: 15,
    4: 30,
    5: 20,
    6: 20,
    7: 30,
    8: 40,
}

round_10_validation = {}

for function_number in range(1, 9):
    X = datasets[function_number]["X"]
    y = datasets[function_number]["y"]

    start_index = initial_observation_counts[
        function_number
    ]

    actual_values = []
    gp_predictions = []
    gp_stds = []
    rbf_predictions = []
    et_predictions = []

    print(f"Validating Function {function_number}...")

    for test_index in range(
        start_index,
        len(y),
    ):
        X_train = X[:test_index]
        y_train = y[:test_index]

        X_test = X[
            test_index:test_index + 1
        ]

        actual = y[test_index]

        gp_mean, gp_std = predict_gp(
            X_train,
            y_train,
            X_test,
            random_state=1000
            + function_number
            + test_index,
        )

        rbf_prediction = predict_rbf(
            X_train,
            y_train,
            X_test,
        )

        et_prediction = predict_extra_trees(
            X_train,
            y_train,
            X_test,
            random_state=2000
            + function_number
            + test_index,
        )

        actual_values.append(
            actual
        )

        gp_predictions.append(
            gp_mean[0]
        )

        gp_stds.append(
            gp_std[0]
        )

        rbf_predictions.append(
            rbf_prediction[0]
        )

        et_predictions.append(
            et_prediction[0]
        )

    actual_values = np.array(
        actual_values
    )

    gp_predictions = np.array(
        gp_predictions
    )

    gp_stds = np.array(
        gp_stds
    )

    rbf_predictions = np.array(
        rbf_predictions
    )

    et_predictions = np.array(
        et_predictions
    )

    scale = np.std(
        actual_values
    )

    if scale < 1e-12:
        scale = 1.0

    def calculate_metrics(
        predictions
    ):
        errors = (
            actual_values
            - predictions
        )

        mae = np.mean(
            np.abs(errors)
        )

        rmse = np.sqrt(
            np.mean(
                errors ** 2
            )
        )

        nrmse = rmse / scale

        spearman = spearmanr(
            actual_values,
            predictions,
        ).statistic

        return {
            "MAE": mae,
            "RMSE": rmse,
            "nRMSE": nrmse,
            "Spearman": spearman,
        }

    gp_metrics = calculate_metrics(
        gp_predictions
    )

    rbf_metrics = calculate_metrics(
        rbf_predictions
    )

    et_metrics = calculate_metrics(
        et_predictions
    )

    gp_z_scores = (
        actual_values
        - gp_predictions
    ) / np.maximum(
        gp_stds,
        1e-12,
    )

    gp_95_coverage = np.mean(
        np.abs(
            gp_z_scores
        ) <= 1.96
    )

    round_10_validation[
        function_number
    ] = {
        "actual": actual_values,
        "GP": gp_predictions,
        "GP_std": gp_stds,
        "RBF": rbf_predictions,
        "ExtraTrees": et_predictions,
        "GP_metrics": gp_metrics,
        "RBF_metrics": rbf_metrics,
        "ExtraTrees_metrics": et_metrics,
        "GP_z_scores": gp_z_scores,
        "GP_95_coverage": gp_95_coverage,
    }

    print(
        f"  GP          "
        f"nRMSE={gp_metrics['nRMSE']:.3f}  "
        f"Spearman={gp_metrics['Spearman']:.3f}"
    )

    print(
        f"  RBF         "
        f"nRMSE={rbf_metrics['nRMSE']:.3f}  "
        f"Spearman={rbf_metrics['Spearman']:.3f}"
    )

    print(
        f"  ExtraTrees  "
        f"nRMSE={et_metrics['nRMSE']:.3f}  "
        f"Spearman={et_metrics['Spearman']:.3f}"
    )

    print(
        f"  GP 95% coverage: "
        f"{gp_95_coverage:.3f}"
    )

    print(
        f"  GP max |z|: "
        f"{np.max(np.abs(gp_z_scores)):.3f}"
    )

    print()

Validating Function 1...


/usr/local/lib/python3.13/dist-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 10000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 10000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/gaussian_process/kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-10. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/gaussian_process/kernels.py:452: Conve

  GP          nRMSE=0.000  Spearman=-0.117
  RBF         nRMSE=0.001  Spearman=-0.033
  ExtraTrees  nRMSE=0.000  Spearman=-0.150
  GP 95% coverage: 1.000
  GP max |z|: 0.703

Validating Function 2...


/usr/local/lib/python3.13/dist-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 10000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 10000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 10000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/gaussian_process/kernels.py:452

  GP          nRMSE=0.627  Spearman=0.567
  RBF         nRMSE=1.288  Spearman=-0.033
  ExtraTrees  nRMSE=0.601  Spearman=0.767
  GP 95% coverage: 0.778
  GP max |z|: 4.882

Validating Function 3...


...

  GP          nRMSE=0.909  Spearman=0.750
  RBF         nRMSE=0.887  Spearman=0.717
  ExtraTrees  nRMSE=1.238  Spearman=0.450
  GP 95% coverage: 0.778
  GP max |z|: 7.052

Validating Function 7...


/usr/local/lib/python3.13/dist-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 10000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 2 of parameter k1__k2__length_scale is close to the specified upper bound 10000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 3 of parameter k1__k2__length_scale is close to the specified upper bound 10000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/gaussian_process/kernels.py:452

  GP          nRMSE=0.663  Spearman=0.817
  RBF         nRMSE=0.448  Spearman=0.883
  ExtraTrees  nRMSE=0.681  Spearman=0.817
  GP 95% coverage: 0.556
  GP max |z|: 7.841

Validating Function 8...


/usr/local/lib/python3.13/dist-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 5 of parameter k1__k2__length_scale is close to the specified upper bound 10000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 7 of parameter k1__k2__length_scale is close to the specified upper bound 10000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 5 of parameter k1__k2__length_scale is close to the specified upper bound 10000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/gaussian_process/kernels.py:452

  GP          nRMSE=2.122  Spearman=0.033
  RBF         nRMSE=7.259  Spearman=0.067
  ExtraTrees  nRMSE=11.967  Spearman=0.667
  GP 95% coverage: 0.889
  GP max |z|: 3.036



## 82. Leave-One-Out Surrogate Validation

In [ ]:
round_10_loo_validation = {}

for function_number in range(1, 9):
    X = datasets[function_number]["X"]
    y = datasets[function_number]["y"]

    actual_values = []
    gp_predictions = []
    rbf_predictions = []
    et_predictions = []

    print(f"LOO Function {function_number}...")

    for test_index in range(len(y)):
        train_mask = np.ones(
            len(y),
            dtype=bool,
        )
        train_mask[test_index] = False

        X_train = X[train_mask]
        y_train = y[train_mask]

        X_test = X[
            test_index:test_index + 1
        ]

        actual = y[test_index]

        gp_mean, _ = predict_gp(
            X_train,
            y_train,
            X_test,
            random_state=3000
            + function_number
            + test_index,
        )

        rbf_prediction = predict_rbf(
            X_train,
            y_train,
            X_test,
        )

        et_prediction = predict_extra_trees(
            X_train,
            y_train,
            X_test,
            random_state=4000
            + function_number
            + test_index,
        )

        actual_values.append(actual)
        gp_predictions.append(gp_mean[0])
        rbf_predictions.append(rbf_prediction[0])
        et_predictions.append(et_prediction[0])

    actual_values = np.array(actual_values)
    gp_predictions = np.array(gp_predictions)
    rbf_predictions = np.array(rbf_predictions)
    et_predictions = np.array(et_predictions)

    scale = np.std(actual_values)

    if scale < 1e-12:
        scale = 1.0

    def loo_metrics(predictions):
        errors = actual_values - predictions

        rmse = np.sqrt(
            np.mean(errors ** 2)
        )

        nrmse = rmse / scale

        spearman = spearmanr(
            actual_values,
            predictions,
        ).statistic

        return {
            "nRMSE": nrmse,
            "Spearman": spearman,
        }

    gp_metrics = loo_metrics(
        gp_predictions
    )

    rbf_metrics = loo_metrics(
        rbf_predictions
    )

    et_metrics = loo_metrics(
        et_predictions
    )

    round_10_loo_validation[
        function_number
    ] = {
        "GP": gp_metrics,
        "RBF": rbf_metrics,
        "ExtraTrees": et_metrics,
    }

    print(
        f"  GP          "
        f"nRMSE={gp_metrics['nRMSE']:.3f}  "
        f"Spearman={gp_metrics['Spearman']:.3f}"
    )

    print(
        f"  RBF         "
        f"nRMSE={rbf_metrics['nRMSE']:.3f}  "
        f"Spearman={rbf_metrics['Spearman']:.3f}"
    )

    print(
        f"  ExtraTrees  "
        f"nRMSE={et_metrics['nRMSE']:.3f}  "
        f"Spearman={et_metrics['Spearman']:.3f}"
    )

    print()

LOO Function 1...


/usr/local/lib/python3.13/dist-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified upper bound 10000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/gaussian_process/kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-10. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/gaussian_process/kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-10. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceW

  GP          nRMSE=1.207  Spearman=-0.358
  RBF         nRMSE=1.518  Spearman=-0.200
  ExtraTrees  nRMSE=1.191  Spearman=-0.614

LOO Function 2...


/usr/local/lib/python3.13/dist-packages/sklearn/gaussian_process/kernels.py:442: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified lower bound 0.0001. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/gaussian_process/kernels.py:442: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified lower bound 0.0001. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/gaussian_process/kernels.py:442: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified lower bound 0.0001. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


  GP          nRMSE=0.815  Spearman=0.607
  RBF         nRMSE=0.978  Spearman=0.479
  ExtraTrees  nRMSE=0.578  Spearman=0.861

LOO Function 3...


...

  GP          nRMSE=0.258  Spearman=0.813
  RBF         nRMSE=0.133  Spearman=0.850
  ExtraTrees  nRMSE=0.178  Spearman=0.867

LOO Function 6...
  GP          nRMSE=0.553  Spearman=0.835
  RBF         nRMSE=0.347  Spearman=0.952
  ExtraTrees  nRMSE=0.490  Spearman=0.854

LOO Function 7...


/usr/local/lib/python3.13/dist-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 2 of parameter k1__k2__length_scale is close to the specified upper bound 10000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 5 of parameter k1__k2__length_scale is close to the specified upper bound 10000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/gaussian_process/kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-10. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/gaussian_process/kernels.py:452: Conve

  GP          nRMSE=0.374  Spearman=0.787
  RBF         nRMSE=0.379  Spearman=0.780
  ExtraTrees  nRMSE=0.397  Spearman=0.680

LOO Function 8...


/usr/local/lib/python3.13/dist-packages/sklearn/gaussian_process/kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-10. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 5 of parameter k1__k2__length_scale is close to the specified upper bound 10000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 7 of parameter k1__k2__length_scale is close to the specified upper bound 10000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/gaussian_process/kernels.py:442: Conve

  GP          nRMSE=0.091  Spearman=0.994
  RBF         nRMSE=0.177  Spearman=0.977
  ExtraTrees  nRMSE=0.336  Spearman=0.939



## 83. Set Round 10 Surrogate Weights and Fit Final Models

In [ ]:
round_10_strategy = {
    1: {
        "weights": {
            "GP": 0.33,
            "RBF": 0.33,
            "ExtraTrees": 0.34,
        },
        "exploration_weight": 0.75,
        "mode": "global_exploration",
    },

    2: {
        "weights": {
            "GP": 0.25,
            "RBF": 0.10,
            "ExtraTrees": 0.65,
        },
        "exploration_weight": 0.10,
        "mode": "local_exploitation",
    },

    3: {
        "weights": {
            "GP": 0.20,
            "RBF": 0.40,
            "ExtraTrees": 0.40,
        },
        "exploration_weight": 0.55,
        "mode": "global_exploration",
    },

    4: {
        "weights": {
            "GP": 0.45,
            "RBF": 0.45,
            "ExtraTrees": 0.10,
        },
        "exploration_weight": 0.10,
        "mode": "local_empirical",
    },

    5: {
        "weights": {
            "GP": 0.15,
            "RBF": 0.50,
            "ExtraTrees": 0.35,
        },
        "exploration_weight": 0.00,
        "mode": "boundary_search",
    },

    6: {
        "weights": {
            "GP": 0.30,
            "RBF": 0.60,
            "ExtraTrees": 0.10,
        },
        "exploration_weight": 0.25,
        "mode": "rbf_local_global",
    },

    7: {
        "weights": {
            "GP": 0.30,
            "RBF": 0.60,
            "ExtraTrees": 0.10,
        },
        "exploration_weight": 0.15,
        "mode": "rbf_local",
    },

    8: {
        "weights": {
            "GP": 0.55,
            "RBF": 0.30,
            "ExtraTrees": 0.15,
        },
        "exploration_weight": 0.05,
        "mode": "fine_local",
    },
}


def fit_rbf_full(X, y):
    y_mean = np.mean(y)
    y_std = np.std(y)

    if y_std < 1e-12:
        y_std = 1.0

    y_scaled = (
        y - y_mean
    ) / y_std

    model = RBFInterpolator(
        X,
        y_scaled,
        kernel="thin_plate_spline",
        smoothing=1e-8,
    )

    return {
        "model": model,
        "y_mean": y_mean,
        "y_std": y_std,
    }


round_10_models = {}

for function_number in range(1, 9):
    X = datasets[function_number]["X"]
    y = datasets[function_number]["y"]

    gp_model = fit_gp(
        X,
        y,
        random_state=5000 + function_number,
    )

    rbf_model = fit_rbf_full(
        X,
        y,
    )

    et_model = ExtraTreesRegressor(
        n_estimators=1000,
        min_samples_leaf=1,
        max_features=1.0,
        random_state=6000 + function_number,
        n_jobs=-1,
    )

    et_model.fit(
        X,
        y,
    )

    round_10_models[function_number] = {
        "GP": gp_model,
        "RBF": rbf_model,
        "ExtraTrees": et_model,
    }

    strategy = round_10_strategy[
        function_number
    ]

    print(
        f"Function {function_number}: "
        f"{strategy['mode']}"
    )

    print(
        f"  Weights: "
        f"{strategy['weights']}"
    )

    print(
        f"  Exploration weight: "
        f"{strategy['exploration_weight']:.2f}"
    )

    print()

/usr/local/lib/python3.13/dist-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified upper bound 10000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


Function 1: global_exploration
  Weights: {'GP': 0.33, 'RBF': 0.33, 'ExtraTrees': 0.34}
  Exploration weight: 0.75

Function 2: local_exploitation
  Weights: {'GP': 0.25, 'RBF': 0.1, 'ExtraTrees': 0.65}
  Exploration weight: 0.10



/usr/local/lib/python3.13/dist-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified upper bound 10000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 2 of parameter k1__k2__length_scale is close to the specified upper bound 10000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


Function 3: global_exploration
  Weights: {'GP': 0.2, 'RBF': 0.4, 'ExtraTrees': 0.4}
  Exploration weight: 0.55

Function 4: local_empirical
  Weights: {'GP': 0.45, 'RBF': 0.45, 'ExtraTrees': 0.1}
  Exploration weight: 0.10

Function 5: boundary_search
  Weights: {'GP': 0.15, 'RBF': 0.5, 'ExtraTrees': 0.35}
  Exploration weight: 0.00

Function 6: rbf_local_global
  Weights: {'GP': 0.3, 'RBF': 0.6, 'ExtraTrees': 0.1}
  Exploration weight: 0.25



/usr/local/lib/python3.13/dist-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 2 of parameter k1__k2__length_scale is close to the specified upper bound 10000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


Function 7: rbf_local
  Weights: {'GP': 0.3, 'RBF': 0.6, 'ExtraTrees': 0.1}
  Exploration weight: 0.15

Function 8: fine_local
  Weights: {'GP': 0.55, 'RBF': 0.3, 'ExtraTrees': 0.15}
  Exploration weight: 0.05



## 84. Generate Round 10 Candidate Pools

In [ ]:
from scipy.stats import qmc

round_10_candidate_pools = {}

local_radius = {
    1: 0.25,
    2: 0.08,
    3: 0.25,
    4: 0.05,
    5: 0.00,
    6: 0.15,
    7: 0.08,
    8: 0.05,
}

pool_mix = {
    1: {"global": 0.85, "local": 0.15},
    2: {"global": 0.15, "local": 0.85},
    3: {"global": 0.75, "local": 0.25},
    4: {"global": 0.15, "local": 0.85},
    5: {"global": 0.00, "local": 1.00},
    6: {"global": 0.35, "local": 0.65},
    7: {"global": 0.20, "local": 0.80},
    8: {"global": 0.10, "local": 0.90},
}

n_total = 65536

for function_number in range(1, 9):
    X = datasets[function_number]["X"]
    y = datasets[function_number]["y"]

    dimension = X.shape[1]

    best_index = np.argmax(y)
    incumbent = X[best_index].copy()

    if function_number == 5:
        x2_grid = np.linspace(
            0.000001,
            0.999999,
            5001,
        )

        candidates = np.column_stack([
            np.full_like(x2_grid, 0.999999),
            x2_grid,
            np.full_like(x2_grid, 0.999999),
            np.full_like(x2_grid, 0.999999),
        ])

        sources = np.array(
            ["boundary"] * len(candidates)
        )

    else:
        n_global = int(
            n_total
            * pool_mix[function_number]["global"]
        )

        n_local = (
            n_total - n_global
        )

        global_sampler = qmc.Sobol(
            d=dimension,
            scramble=True,
            seed=7000 + function_number,
        )

        global_candidates = global_sampler.random(
            n=n_global
        )

        rng = np.random.default_rng(
            8000 + function_number
        )

        radius = local_radius[
            function_number
        ]

        local_candidates = (
            incumbent
            + rng.uniform(
                -radius,
                radius,
                size=(n_local, dimension),
            )
        )

        local_candidates = np.clip(
            local_candidates,
            0.000001,
            0.999999,
        )

        candidates = np.vstack([
            global_candidates,
            local_candidates,
        ])

        sources = np.concatenate([
            np.array(
                ["global"] * len(global_candidates)
            ),
            np.array(
                ["local"] * len(local_candidates)
            ),
        ])

    distances = np.min(
        np.linalg.norm(
            candidates[:, None, :]
            - X[None, :, :],
            axis=2,
        ),
        axis=1,
    )

    keep = distances > 1e-6

    candidates = candidates[keep]
    sources = sources[keep]
    distances = distances[keep]

    round_10_candidate_pools[
        function_number
    ] = {
        "X": candidates,
        "source": sources,
        "distance": distances,
        "incumbent": incumbent,
    }

    print(
        f"Function {function_number}: "
        f"{len(candidates)} candidates"
    )

    print(
        f"  Incumbent: "
        f"{np.round(incumbent, 6)}"
    )

    if function_number == 5:
        print(
            "  Pool: boundary line search"
        )
    else:
        unique, counts = np.unique(
            sources,
            return_counts=True,
        )

        source_counts = dict(
            zip(unique, counts)
        )

        print(
            f"  Sources: {source_counts}"
        )

    print()

Function 1: 65536 candidates
  Incumbent: [0.731024 0.733   ]
  Sources: {np.str_('global'): np.int64(55705), np.str_('local'): np.int64(9831)}



/tmp/ipykernel_5596/2236193390.py:72: UserWarning: The balance properties of Sobol' points require n to be a power of 2.
  global_candidates = global_sampler.random(


Function 2: 65536 candidates
  Incumbent: [0.69378  0.999999]
  Sources: {np.str_('global'): np.int64(9830), np.str_('local'): np.int64(55706)}

Function 3: 65536 candidates
  Incumbent: [0.492581 0.611593 0.340176]
  Sources: {np.str_('global'): np.int64(49152), np.str_('local'): np.int64(16384)}

Function 4: 65536 candidates
  Incumbent: [0.366541 0.393044 0.418603 0.428926]
  Sources: {np.str_('global'): np.int64(9830), np.str_('local'): np.int64(55706)}

Function 5: 4999 candidates
  Incumbent: [0.999999 0.1      0.999999 0.999999]
  Pool: boundary line search

Function 6: 65536 candidates
  Incumbent: [4.48596e-01 2.73706e-01 7.01605e-01 7.56821e-01 1.00000e-04]
  Sources: {np.str_('global'): np.int64(22937), np.str_('local'): np.int64(42599)}

Function 7: 65536 candidates
  Incumbent: [0.158216 0.255147 0.353046 0.156217 0.376682 0.744626]
  Sources: {np.str_('global'): np.int64(13107), np.str_('local'): np.int64(52429)}

Function 8: 65536 candidates
  Incumbent: [0.110172 0.1428

## 85. Score Round 10 Candidates with the Multi-Surrogate Portfolio

In [ ]:
from scipy.stats import rankdata

round_10_scored_candidates = {}

for function_number in range(1, 9):
    pool = round_10_candidate_pools[
        function_number
    ]

    candidates = pool["X"]
    sources = pool["source"]
    distances = pool["distance"]

    models = round_10_models[
        function_number
    ]

    strategy = round_10_strategy[
        function_number
    ]

    weights = strategy["weights"]
    exploration_weight = strategy[
        "exploration_weight"
    ]

    gp_mean, gp_std = models[
        "GP"
    ].predict(
        candidates,
        return_std=True,
    )

    rbf_info = models["RBF"]

    rbf_scaled = rbf_info[
        "model"
    ](
        candidates
    )

    rbf_prediction = (
        rbf_scaled
        * rbf_info["y_std"]
        + rbf_info["y_mean"]
    )

    et_prediction = models[
        "ExtraTrees"
    ].predict(
        candidates
    )

    n_candidates = len(
        candidates
    )

    gp_rank = rankdata(
        gp_mean,
        method="average",
    ) / n_candidates

    rbf_rank = rankdata(
        rbf_prediction,
        method="average",
    ) / n_candidates

    et_rank = rankdata(
        et_prediction,
        method="average",
    ) / n_candidates

    distance_rank = rankdata(
        distances,
        method="average",
    ) / n_candidates

    model_score = (
        weights["GP"] * gp_rank
        + weights["RBF"] * rbf_rank
        + weights["ExtraTrees"] * et_rank
    )

    final_score = (
        model_score
        + exploration_weight
        * distance_rank
    ) / (
        1.0
        + exploration_weight
    )

    round_10_scored_candidates[
        function_number
    ] = {
        "X": candidates,
        "source": sources,
        "distance": distances,
        "GP_mean": gp_mean,
        "GP_std": gp_std,
        "RBF": rbf_prediction,
        "ExtraTrees": et_prediction,
        "GP_rank": gp_rank,
        "RBF_rank": rbf_rank,
        "ExtraTrees_rank": et_rank,
        "distance_rank": distance_rank,
        "model_score": model_score,
        "final_score": final_score,
    }

    best_index = np.argmax(
        final_score
    )

    print(
        f"Function {function_number}"
    )

    print(
        f"  Best candidate: "
        f"{np.round(candidates[best_index], 6)}"
    )

    print(
        f"  Source: "
        f"{sources[best_index]}"
    )

    print(
        f"  Final score: "
        f"{final_score[best_index]:.6f}"
    )

    print(
        f"  GP: "
        f"{gp_mean[best_index]:.10g}"
    )

    print(
        f"  RBF: "
        f"{rbf_prediction[best_index]:.10g}"
    )

    print(
        f"  ExtraTrees: "
        f"{et_prediction[best_index]:.10g}"
    )

    print(
        f"  Distance: "
        f"{distances[best_index]:.6f}"
    )

    print()

Function 1
  Best candidate: [0.997845 0.874938]
  Source: global
  Final score: 0.946304
  GP: 1.485279124e-06
  RBF: 0.0006527502197
  ExtraTrees: 6.428024165e-17
  Distance: 0.266056

Function 2
  Best candidate: [0.693481 0.963088]
  Source: local
  Final score: 0.924941
  GP: 0.5932155748
  RBF: 0.624276821
  ExtraTrees: 0.6214263279
  Distance: 0.036912

Function 3
  Best candidate: [0.388324 0.861945 0.04099 ]
  Source: global
  Final score: 0.860177
  GP: -0.05577412555
  RBF: -0.03641320099
  ExtraTrees: -0.05684993914
  Distance: 0.245161

Function 4
  Best candidate: [0.360033 0.396079 0.413445 0.431524]
  Source: local
  Final score: 0.908943
  GP: 0.3100468913
  RBF: 0.5872160483
  ExtraTrees: 0.5156527285
  Distance: 0.009215

Function 5
  Best candidate: [0.999999 0.094601 0.999999 0.999999]
  Source: boundary
  Final score: 0.862372
  GP: 4441.113725
  RBF: 4441.906655
  ExtraTrees: 4441.113512
  Distance: 0.005399

Function 6
  Best candidate: [4.20294e-01 3.68998e-01 

## 86. Inspect Model-Specific and Exploratory Round 10 Candidates

In [ ]:
round_10_candidate_shortlist = {}

for function_number in range(1, 9):
    scored = round_10_scored_candidates[
        function_number
    ]

    X = scored["X"]

    candidate_indices = {
        "consensus": np.argmax(
            scored["final_score"]
        ),
        "GP": np.argmax(
            scored["GP_mean"]
        ),
        "RBF": np.argmax(
            scored["RBF"]
        ),
        "ExtraTrees": np.argmax(
            scored["ExtraTrees"]
        ),
        "exploratory": np.argmax(
            scored["distance"]
        ),
    }

    round_10_candidate_shortlist[
        function_number
    ] = candidate_indices

    current_best = np.max(
        datasets[function_number]["y"]
    )

    print("=" * 70)
    print(
        f"Function {function_number}"
    )
    print(
        f"Current best: {current_best:.10g}"
    )
    print()

    seen = set()

    for label, index in candidate_indices.items():
        point_tuple = tuple(
            np.round(
                X[index],
                10,
            )
        )

        if point_tuple in seen:
            continue

        seen.add(
            point_tuple
        )

        print(
            f"{label.upper()}"
        )

        print(
            f"  X: "
            f"{np.round(X[index], 6)}"
        )

        print(
            f"  Source: "
            f"{scored['source'][index]}"
        )

        print(
            f"  GP: "
            f"{scored['GP_mean'][index]:.10g}"
        )

        print(
            f"  GP std: "
            f"{scored['GP_std'][index]:.10g}"
        )

        print(
            f"  RBF: "
            f"{scored['RBF'][index]:.10g}"
        )

        print(
            f"  ExtraTrees: "
            f"{scored['ExtraTrees'][index]:.10g}"
        )

        print(
            f"  Final score: "
            f"{scored['final_score'][index]:.6f}"
        )

        print(
            f"  Distance: "
            f"{scored['distance'][index]:.6f}"
        )

        print()

Function 1
Current best: 7.710875115e-16

CONSENSUS
  X: [0.997845 0.874938]
  Source: global
  GP: 1.485279124e-06
  GP std: 5.445140238e-05
  RBF: 0.0006527502197
  ExtraTrees: 6.428024165e-17
  Final score: 0.946304
  Distance: 0.266056

GP
  X: [0.424366 0.609939]
  Source: global
  GP: 5.474553065e-05
  GP std: 0.0001011954136
  RBF: 0.0004523272239
  ExtraTrees: 2.320668476e-19
  Final score: 0.538936
  Distance: 0.042634

RBF
  X: [0.838247 0.771569]
  Source: local
  GP: 2.371964771e-05
  GP std: 0.0002066282271
  RBF: 0.0007761312991
  ExtraTrees: 1.536997721e-16
  Final score: 0.789789
  Distance: 0.113949

EXTRATREES
  X: [0.73473  0.733607]
  Source: global
  GP: 8.767894844e-06
  GP std: 8.176240028e-06
  RBF: 8.547501471e-05
  ExtraTrees: 3.119159139e-16
  Final score: 0.495980
  Distance: 0.003756

EXPLORATORY
  X: [9.8445e-01 7.1500e-04]
  Source: global
  GP: 5.064807528e-07
  GP std: 2.039173209e-05
  RBF: 7.55369635e-05
  ExtraTrees: 1.600807326e-17
  Final score: 0.

## 87. Targeted Local Checks Before Final Round 10 Selection

In [ ]:
targeted_checks = {}

def evaluate_points(function_number, points):
    points = np.array(points)

    models = round_10_models[
        function_number
    ]

    gp_mean, gp_std = models[
        "GP"
    ].predict(
        points,
        return_std=True,
    )

    rbf_info = models["RBF"]

    rbf_prediction = (
        rbf_info["model"](points)
        * rbf_info["y_std"]
        + rbf_info["y_mean"]
    )

    et_prediction = models[
        "ExtraTrees"
    ].predict(points)

    X_existing = datasets[
        function_number
    ]["X"]

    distances = np.min(
        np.linalg.norm(
            points[:, None, :]
            - X_existing[None, :, :],
            axis=2,
        ),
        axis=1,
    )

    return {
        "X": points,
        "GP": gp_mean,
        "GP_std": gp_std,
        "RBF": rbf_prediction,
        "ExtraTrees": et_prediction,
        "distance": distances,
    }


# Function 2: stay near x2 = 1
f2_x1 = np.linspace(
    0.688,
    0.699,
    45,
)

f2_points = np.column_stack([
    f2_x1,
    np.full_like(
        f2_x1,
        0.999999,
    ),
])

targeted_checks[2] = evaluate_points(
    2,
    f2_points,
)


# Function 5: fine x2 profile near successful region
f5_x2 = np.linspace(
    0.07,
    0.13,
    121,
)

f5_points = np.column_stack([
    np.full_like(f5_x2, 0.999999),
    f5_x2,
    np.full_like(f5_x2, 0.999999),
    np.full_like(f5_x2, 0.999999),
])

targeted_checks[5] = evaluate_points(
    5,
    f5_points,
)


# Function 6: move from incumbent toward RBF candidate
f6_incumbent = round_10_candidate_pools[
    6
]["incumbent"]

f6_target = np.array([
    0.437026,
    0.287057,
    0.688289,
    0.760720,
    0.000001,
])

f6_alpha = np.linspace(
    0.1,
    1.0,
    19,
)

f6_points = np.array([
    f6_incumbent
    + alpha
    * (
        f6_target
        - f6_incumbent
    )
    for alpha in f6_alpha
])

targeted_checks[6] = evaluate_points(
    6,
    f6_points,
)


# Function 7: cautious path toward RBF candidate
f7_incumbent = round_10_candidate_pools[
    7
]["incumbent"]

f7_target = np.array([
    0.175992,
    0.192875,
    0.420729,
    0.167299,
    0.386135,
    0.763854,
])

f7_alpha = np.linspace(
    0.1,
    1.0,
    19,
)

f7_points = np.array([
    f7_incumbent
    + alpha
    * (
        f7_target
        - f7_incumbent
    )
    for alpha in f7_alpha
])

targeted_checks[7] = evaluate_points(
    7,
    f7_points,
)


# Function 8: cautious path toward GP candidate
f8_incumbent = round_10_candidate_pools[
    8
]["incumbent"]

f8_target = np.array([
    0.114846,
    0.142684,
    0.155842,
    0.155654,
    0.854051,
    0.495220,
    0.180066,
    0.637573,
])

f8_alpha = np.linspace(
    0.1,
    1.0,
    19,
)

f8_points = np.array([
    f8_incumbent
    + alpha
    * (
        f8_target
        - f8_incumbent
    )
    for alpha in f8_alpha
])

targeted_checks[8] = evaluate_points(
    8,
    f8_points,
)


for function_number in [
    2,
    5,
    6,
    7,
    8,
]:
    result = targeted_checks[
        function_number
    ]

    weights = round_10_strategy[
        function_number
    ]["weights"]

    n = len(
        result["X"]
    )

    gp_rank = rankdata(
        result["GP"]
    ) / n

    rbf_rank = rankdata(
        result["RBF"]
    ) / n

    et_rank = rankdata(
        result["ExtraTrees"]
    ) / n

    score = (
        weights["GP"] * gp_rank
        + weights["RBF"] * rbf_rank
        + weights["ExtraTrees"] * et_rank
    )

    best_index = np.argmax(
        score
    )

    print(
        f"Function {function_number}"
    )

    print(
        f"  Best targeted X: "
        f"{np.round(result['X'][best_index], 6)}"
    )

    print(
        f"  GP: "
        f"{result['GP'][best_index]:.10g}"
    )

    print(
        f"  RBF: "
        f"{result['RBF'][best_index]:.10g}"
    )

    print(
        f"  ExtraTrees: "
        f"{result['ExtraTrees'][best_index]:.10g}"
    )

    print(
        f"  Distance: "
        f"{result['distance'][best_index]:.6f}"
    )

    if function_number in [
        6,
        7,
        8,
    ]:
        print(
            f"  Alpha: "
            f"{[f6_alpha, f7_alpha, f8_alpha][[6,7,8].index(function_number)][best_index]:.2f}"
        )

    print()

Function 2
  Best targeted X: [0.69375  0.999999]
  GP: 0.5931789794
  RBF: 0.6302107702
  ExtraTrees: 0.6301453245
  Distance: 0.000030

Function 5
  Best targeted X: [0.999999 0.0945   0.999999 0.999999]
  GP: 4441.112065
  RBF: 4441.90608
  ExtraTrees: 4441.112845
  Distance: 0.005500

Function 6
  Best targeted X: [4.42811e-01 2.80382e-01 6.94947e-01 7.58770e-01 5.00000e-05]
  GP: -0.2898560381
  RBF: -0.2566074237
  ExtraTrees: -0.275149759
  Distance: 0.011232
  Alpha: 0.50

Function 7
  Best targeted X: [0.175992 0.192875 0.420729 0.167299 0.386135 0.763854]
  GP: 2.371863778
  RBF: 2.417473448
  ExtraTrees: 2.218948658
  Distance: 0.096730
  Alpha: 1.00

Function 8
  Best targeted X: [0.114846 0.142684 0.155842 0.155654 0.854051 0.49522  0.180066 0.637573]
  GP: 9.994582326
  RBF: 10.00850523
  ExtraTrees: 9.984100574
  Distance: 0.051607
  Alpha: 1.00



## 88. Final Safety Checks for Functions 2 and 4

In [ ]:
# Function 2:
# Search along x2 = 1, but require a meaningful move
# away from every point already evaluated.

f2_x1 = np.linspace(
    0.675,
    0.710,
    281,
)

f2_points = np.column_stack([
    f2_x1,
    np.full_like(
        f2_x1,
        0.999999,
    ),
])

f2_check = evaluate_points(
    2,
    f2_points,
)

f2_valid = (
    f2_check["distance"]
    >= 0.002
)

f2_weights = round_10_strategy[
    2
]["weights"]

n_f2 = np.sum(f2_valid)

f2_gp_rank = rankdata(
    f2_check["GP"][f2_valid]
) / n_f2

f2_rbf_rank = rankdata(
    f2_check["RBF"][f2_valid]
) / n_f2

f2_et_rank = rankdata(
    f2_check["ExtraTrees"][f2_valid]
) / n_f2

f2_score = (
    f2_weights["GP"]
    * f2_gp_rank
    + f2_weights["RBF"]
    * f2_rbf_rank
    + f2_weights["ExtraTrees"]
    * f2_et_rank
)

f2_valid_indices = np.where(
    f2_valid
)[0]

f2_best_index = f2_valid_indices[
    np.argmax(f2_score)
]


# Function 4:
# Round 9 moved away from the incumbent and got worse.
# Test the opposite direction plus the strongest
# local RBF/consensus directions.

f4_incumbent = round_10_candidate_pools[
    4
]["incumbent"]

f4_round9 = np.array([
    0.371541,
    0.388044,
    0.423603,
    0.428926,
])

f4_mirror = (
    2 * f4_incumbent
    - f4_round9
)

f4_consensus = np.array([
    0.360033,
    0.396079,
    0.413445,
    0.431524,
])

f4_rbf_target = np.array([
    0.364461,
    0.402571,
    0.411537,
    0.429537,
])

f4_targets = [
    f4_mirror,
    f4_consensus,
    f4_rbf_target,
]

f4_alpha = np.linspace(
    0.25,
    1.25,
    41,
)

f4_points = []

for target in f4_targets:
    for alpha in f4_alpha:
        point = (
            f4_incumbent
            + alpha
            * (
                target
                - f4_incumbent
            )
        )

        point = np.clip(
            point,
            0.000001,
            0.999999,
        )

        f4_points.append(
            point
        )

f4_points = np.array(
    f4_points
)

f4_check = evaluate_points(
    4,
    f4_points,
)

f4_weights = round_10_strategy[
    4
]["weights"]

n_f4 = len(
    f4_points
)

f4_gp_rank = rankdata(
    f4_check["GP"]
) / n_f4

f4_rbf_rank = rankdata(
    f4_check["RBF"]
) / n_f4

f4_et_rank = rankdata(
    f4_check["ExtraTrees"]
) / n_f4

f4_score = (
    f4_weights["GP"]
    * f4_gp_rank
    + f4_weights["RBF"]
    * f4_rbf_rank
    + f4_weights["ExtraTrees"]
    * f4_et_rank
)

f4_best_index = np.argmax(
    f4_score
)


for function_number, result, index in [
    (
        2,
        f2_check,
        f2_best_index,
    ),
    (
        4,
        f4_check,
        f4_best_index,
    ),
]:
    print(
        f"Function {function_number}"
    )

    print(
        f"  X: "
        f"{np.round(result['X'][index], 6)}"
    )

    print(
        f"  GP: "
        f"{result['GP'][index]:.10g}"
    )

    print(
        f"  RBF: "
        f"{result['RBF'][index]:.10g}"
    )

    print(
        f"  ExtraTrees: "
        f"{result['ExtraTrees'][index]:.10g}"
    )

    print(
        f"  Distance: "
        f"{result['distance'][index]:.6f}"
    )

    print()

Function 2
  X: [0.69175  0.999999]
  GP: 0.5931902951
  RBF: 0.6341842214
  ExtraTrees: 0.6235881761
  Distance: 0.002030

Function 4
  X: [0.362916 0.396669 0.414978 0.428926]
  GP: 0.3107523709
  RBF: 0.5876172597
  ExtraTrees: 0.535162312
  Distance: 0.006279



## 89. Select and Save Round 10 Queries

In [ ]:
round_10_queries = {
    1: [0.997845, 0.874938],

    2: [0.691750, 0.999999],

    3: [0.388324, 0.861945, 0.040990],

    4: [0.362916, 0.396669, 0.414978, 0.428926],

    5: [0.999999, 0.094500, 0.999999, 0.999999],

    6: [0.442811, 0.280382, 0.694947, 0.758770, 0.000050],

    7: [0.175992, 0.192875, 0.420729, 0.167299, 0.386135, 0.763854],

    8: [0.114846, 0.142684, 0.155842, 0.155654, 0.854051, 0.495220, 0.180066, 0.637573],
}

import json
import os

save_path = (
    "/content/drive/MyDrive/Imperial_BBO/"
    "data/Week_10/submitted_queries.json"
)

os.makedirs(
    os.path.dirname(save_path),
    exist_ok=True,
)

with open(
    save_path,
    "w",
) as f:
    json.dump(
        round_10_queries,
        f,
        indent=2,
    )

print(
    f"Saved Round 10 queries to:\n{save_path}"
)

print("\nPortal submission strings:\n")

for function_number in range(1, 9):
    query_string = "-".join(
        f"{value:.6f}"
        for value in round_10_queries[
            function_number
        ]
    )

    print(
        f"F{function_number}: "
        f"{query_string}"
    )

Saved Round 10 queries to:
/content/drive/MyDrive/Imperial_BBO/data/Week_10/submitted_queries.json

Portal submission strings:

F1: 0.997845-0.874938
F2: 0.691750-0.999999
F3: 0.388324-0.861945-0.040990
F4: 0.362916-0.396669-0.414978-0.428926
F5: 0.999999-0.094500-0.999999-0.999999
F6: 0.442811-0.280382-0.694947-0.758770-0.000050
F7: 0.175992-0.192875-0.420729-0.167299-0.386135-0.763854
F8: 0.114846-0.142684-0.155842-0.155654-0.854051-0.495220-0.180066-0.637573


## 90. Load Round 10 Returned Results

In [ ]:
round_10_results = {
    1: {
        "x": [0.997845, 0.874938],
        "y": 1.1873318444435154e-137,
    },
    2: {
        "x": [0.691750, 0.999999],
        "y": 0.5440923875179878,
    },
    3: {
        "x": [0.388324, 0.861945, 0.040990],
        "y": -0.06512206680415766,
    },
    4: {
        "x": [0.362916, 0.396669, 0.414978, 0.428926],
        "y": 0.5906583773384715,
    },
    5: {
        "x": [0.999999, 0.094500, 0.999999, 0.999999],
        "y": 4441.036783450597,
    },
    6: {
        "x": [0.442811, 0.280382, 0.694947, 0.758770, 0.000050],
        "y": -0.28514244789545373,
    },
    7: {
        "x": [0.175992, 0.192875, 0.420729, 0.167299, 0.386135, 0.763854],
        "y": 2.463725853893674,
    },
    8: {
        "x": [0.114846, 0.142684, 0.155842, 0.155654, 0.854051, 0.495220, 0.180066, 0.637573],
        "y": 9.9950507686586,
    },
}

print("Round 10 results loaded.")

Round 10 results loaded.


## 91. Update Datasets with Round 10 Results

In [ ]:
for function_number, result in round_10_results.items():
    x_new = np.array(result["x"]).reshape(1, -1)
    y_new = np.array([result["y"]])

    datasets[function_number]["X"] = np.vstack(
        [datasets[function_number]["X"], x_new]
    )

    datasets[function_number]["y"] = np.concatenate(
        [datasets[function_number]["y"], y_new]
    )

print("Round 10 results added to datasets.")

for function_number in range(1, 9):
    print(
        f"Function {function_number}: "
        f"{len(datasets[function_number]['y'])} observations"
    )

Round 10 results added to datasets.
Function 1: 20 observations
Function 2: 20 observations
Function 3: 25 observations
Function 4: 40 observations
Function 5: 30 observations
Function 6: 30 observations
Function 7: 40 observations
Function 8: 50 observations


## 92. Round 10 Performance Diagnostics

In [ ]:
round_10_diagnostics = {}

for function_number in range(1, 9):
    X = datasets[function_number]["X"]
    y = datasets[function_number]["y"]

    round_10_x = X[-1]
    round_10_y = y[-1]

    previous_best = np.max(y[:-1])
    current_best = np.max(y)

    improved = round_10_y > previous_best

    improvement = (
        round_10_y - previous_best
        if improved
        else 0.0
    )

    previous_X = X[:-1]

    closest_distance = np.min(
        np.linalg.norm(
            previous_X - round_10_x,
            axis=1,
        )
    )

    round_10_diagnostics[
        function_number
    ] = {
        "previous_best": previous_best,
        "round_10_result": round_10_y,
        "current_best": current_best,
        "improved": improved,
        "improvement": improvement,
        "distance": closest_distance,
    }

    print(f"Function {function_number}")
    print(
        f"  Previous best: "
        f"{previous_best:.10g}"
    )
    print(
        f"  Round 10 result: "
        f"{round_10_y:.10g}"
    )
    print(
        f"  Current best: "
        f"{current_best:.10g}"
    )
    print(
        f"  Improved: "
        f"{improved}"
    )
    print(
        f"  Closest previous distance: "
        f"{closest_distance:.6f}"
    )
    print()

Function 1
  Previous best: 7.710875115e-16
  Round 10 result: 1.187331844e-137
  Current best: 7.710875115e-16
  Improved: False
  Closest previous distance: 0.266057

Function 2
  Previous best: 0.6301453245
  Round 10 result: 0.5440923875
  Current best: 0.6301453245
  Improved: False
  Closest previous distance: 0.002030

Function 3
  Previous best: -0.03483531335
  Round 10 result: -0.0651220668
  Current best: -0.03483531335
  Improved: False
  Closest previous distance: 0.245162

Function 4
  Previous best: 0.5657943181
  Round 10 result: 0.5906583773
  Current best: 0.5906583773
  Improved: True
  Closest previous distance: 0.006279

Function 5
  Previous best: 4441.14751
  Round 10 result: 4441.036783
  Current best: 4441.14751
  Improved: False
  Closest previous distance: 0.005500

Function 6
  Previous best: -0.2588092057
  Round 10 result: -0.2851424479
  Current best: -0.2588092057
  Improved: False
  Closest previous distance: 0.011232

Function 7
  Previous best: 2.3290

## 93. Identify Input-Space Clusters for Round 11

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

round_11_clusters = {}

for function_number in range(1, 9):
    X = datasets[function_number]["X"]
    y = datasets[function_number]["y"]

    best_k = None
    best_score = -np.inf
    best_labels = None
    best_model = None

    max_k = min(
        6,
        len(X) - 1,
    )

    for k in range(
        2,
        max_k + 1,
    ):
        model = KMeans(
            n_clusters=k,
            n_init=50,
            random_state=9000 + function_number,
        )

        labels = model.fit_predict(X)

        score = silhouette_score(
            X,
            labels,
        )

        if score > best_score:
            best_score = score
            best_k = k
            best_labels = labels
            best_model = model

    cluster_info = []

    for cluster_id in range(best_k):
        mask = (
            best_labels
            == cluster_id
        )

        cluster_y = y[mask]
        cluster_X = X[mask]

        info = {
            "cluster": cluster_id,
            "size": np.sum(mask),
            "mean_y": np.mean(cluster_y),
            "max_y": np.max(cluster_y),
            "centroid": np.mean(
                cluster_X,
                axis=0,
            ),
        }

        cluster_info.append(
            info
        )

    best_observation_index = np.argmax(y)
    best_cluster = best_labels[
        best_observation_index
    ]

    round_11_clusters[
        function_number
    ] = {
        "k": best_k,
        "silhouette": best_score,
        "labels": best_labels,
        "model": best_model,
        "cluster_info": cluster_info,
        "best_cluster": best_cluster,
    }

    print("=" * 70)
    print(
        f"Function {function_number}"
    )
    print(
        f"  Selected clusters: {best_k}"
    )
    print(
        f"  Silhouette score: {best_score:.3f}"
    )
    print(
        f"  Best-value cluster: {best_cluster}"
    )
    print()

    for info in cluster_info:
        marker = (
            " <-- contains current best"
            if info["cluster"]
            == best_cluster
            else ""
        )

        print(
            f"  Cluster {info['cluster']}{marker}"
        )
        print(
            f"    Size: {info['size']}"
        )
        print(
            f"    Mean y: {info['mean_y']:.10g}"
        )
        print(
            f"    Max y: {info['max_y']:.10g}"
        )
        print(
            f"    Centroid: "
            f"{np.round(info['centroid'], 6)}"
        )
        print()

Function 1
  Selected clusters: 2
  Silhouette score: 0.450
  Best-value cluster: 0

  Cluster 0 <-- contains current best
    Size: 14
    Mean y: -0.0002575759032
    Max y: 7.710875115e-16
    Centroid: [0.720683 0.679949]

  Cluster 1
    Size: 6
    Mean y: -3.598748393e-55
    Max y: 3.60677119e-81
    Centroid: [0.228359 0.239652]

Function 2
  Selected clusters: 4
  Silhouette score: 0.560
  Best-value cluster: 1

  Cluster 0
    Size: 4
    Mean y: 0.04355807067
    Max y: 0.2149645101
    Centroid: [0.319436 0.220506]

  Cluster 1 <-- contains current best
    Size: 12
    Mean y: 0.4654050722
    Max y: 0.6301453245
    Centroid: [0.695469 0.821309]

  Cluster 2
    Size: 3
    Mean y: 0.401386665
    Max y: 0.5702046356
    Centroid: [0.775566 0.132137]

  Cluster 3
    Size: 1
    Mean y: 0.02921568688
    Max y: 0.02921568688
    Centroid: [1.00000e-06 9.90398e-01]

Function 3
  Selected clusters: 5
  Silhouette score: 0.349
  Best-value cluster: 1

  Cluster 0
    Size: 

## 94. Analyse the Best-Performing Region Within Each Cluster

In [ ]:
round_11_elite_regions = {}

for function_number in range(1, 9):
    X = datasets[function_number]["X"]
    y = datasets[function_number]["y"]

    cluster_result = round_11_clusters[
        function_number
    ]

    labels = cluster_result["labels"]
    best_cluster = cluster_result[
        "best_cluster"
    ]

    cluster_mask = (
        labels == best_cluster
    )

    cluster_X = X[cluster_mask]
    cluster_y = y[cluster_mask]

    order = np.argsort(
        cluster_y
    )[::-1]

    cluster_X_sorted = cluster_X[
        order
    ]

    cluster_y_sorted = cluster_y[
        order
    ]

    elite_count = max(
        3,
        int(
            np.ceil(
                0.30
                * len(cluster_y_sorted)
            )
        ),
    )

    elite_count = min(
        elite_count,
        len(cluster_y_sorted),
    )

    elite_X = cluster_X_sorted[
        :elite_count
    ]

    elite_y = cluster_y_sorted[
        :elite_count
    ]

    elite_centroid = np.mean(
        elite_X,
        axis=0,
    )

    elite_spread = np.std(
        elite_X,
        axis=0,
    )

    best_x = cluster_X_sorted[0]
    best_y = cluster_y_sorted[0]

    elite_distances = np.linalg.norm(
        elite_X
        - elite_centroid,
        axis=1,
    )

    elite_radius = np.max(
        elite_distances
    )

    round_11_elite_regions[
        function_number
    ] = {
        "cluster": best_cluster,
        "elite_X": elite_X,
        "elite_y": elite_y,
        "centroid": elite_centroid,
        "spread": elite_spread,
        "radius": elite_radius,
        "best_x": best_x,
        "best_y": best_y,
    }

    print("=" * 70)
    print(
        f"Function {function_number}"
    )

    print(
        f"  Elite points used: "
        f"{elite_count}"
    )

    print(
        f"  Current best X: "
        f"{np.round(best_x, 6)}"
    )

    print(
        f"  Current best y: "
        f"{best_y:.10g}"
    )

    print(
        f"  Elite centroid: "
        f"{np.round(elite_centroid, 6)}"
    )

    print(
        f"  Elite spread: "
        f"{np.round(elite_spread, 6)}"
    )

    print(
        f"  Elite radius: "
        f"{elite_radius:.6f}"
    )

    print(
        "  Elite observations:"
    )

    for x_point, y_value in zip(
        elite_X,
        elite_y,
    ):
        print(
            f"    y={y_value:.10g}  "
            f"x={np.round(x_point, 6)}"
        )

    print()

Function 1
  Elite points used: 5
  Current best X: [0.731024 0.733   ]
  Current best y: 7.710875115e-16
  Elite centroid: [0.637444 0.790646]
  Elite spread: [0.103048 0.101309]
  Elite radius: 0.243475
  Elite observations:
    y=7.710875115e-16  x=[0.731024 0.733   ]
    y=7.580643356e-19  x=[0.466336 0.617435]
    y=2.535001154e-40  x=[0.683418 0.861057]
    y=5.70089734e-46  x=[0.732111 0.86184 ]
    y=1.033078238e-46  x=[0.574329 0.879898]

Function 2
  Elite points used: 4
  Current best X: [0.69378  0.999999]
  Current best y: 0.6301453245
  Elite centroid: [0.696267 0.863387]
  Elite spread: [0.003704 0.103213]
  Elite radius: 0.136634
  Elite observations:
    y=0.6301453245  x=[0.69378  0.999999]
    y=0.6265602259  x=[0.69378  0.763493]
    y=0.6122414108  x=[0.694873 0.763493]
    y=0.6112052158  x=[0.702637 0.926564]

Function 3
  Elite points used: 3
  Current best X: [0.492581 0.611593 0.340176]
  Current best y: -0.03483531335
  Elite centroid: [0.469389 0.654872 0.15

## 95. Set Up Round 11 Cluster-Aware Surrogate Models

In [ ]:
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, Matern, WhiteKernel
from scipy.interpolate import RBFInterpolator


def fit_gp(X, y, random_state=0):
    dimension = X.shape[1]

    kernel = (
        ConstantKernel(
            1.0,
            (1e-3, 1e3),
        )
        * Matern(
            length_scale=np.ones(dimension),
            length_scale_bounds=(1e-4, 1e4),
            nu=2.5,
        )
        + WhiteKernel(
            noise_level=1e-8,
            noise_level_bounds=(1e-10, 1e-1),
        )
    )

    model = GaussianProcessRegressor(
        kernel=kernel,
        normalize_y=True,
        n_restarts_optimizer=5,
        random_state=random_state,
    )

    model.fit(X, y)

    return model


def fit_rbf(X, y):
    y_mean = np.mean(y)
    y_std = np.std(y)

    if y_std < 1e-12:
        y_std = 1.0

    y_scaled = (
        y - y_mean
    ) / y_std

    model = RBFInterpolator(
        X,
        y_scaled,
        kernel="thin_plate_spline",
        smoothing=1e-8,
    )

    return {
        "model": model,
        "y_mean": y_mean,
        "y_std": y_std,
    }


round_11_strategy = {
    1: {
        "weights": {
            "GP": 0.33,
            "RBF": 0.33,
            "ExtraTrees": 0.34,
        },
        "exploration": 0.70,
        "cluster_weight": 0.10,
        "mode": "global_exploration",
    },

    2: {
        "weights": {
            "GP": 0.20,
            "RBF": 0.25,
            "ExtraTrees": 0.55,
        },
        "exploration": 0.05,
        "cluster_weight": 0.35,
        "mode": "narrow_boundary",
    },

    3: {
        "weights": {
            "GP": 0.20,
            "RBF": 0.40,
            "ExtraTrees": 0.40,
        },
        "exploration": 0.50,
        "cluster_weight": 0.15,
        "mode": "broad_exploration",
    },

    4: {
        "weights": {
            "GP": 0.35,
            "RBF": 0.55,
            "ExtraTrees": 0.10,
        },
        "exploration": 0.05,
        "cluster_weight": 0.55,
        "mode": "elite_local",
    },

    5: {
        "weights": {
            "GP": 0.15,
            "RBF": 0.55,
            "ExtraTrees": 0.30,
        },
        "exploration": 0.00,
        "cluster_weight": 0.45,
        "mode": "fine_boundary",
    },

    6: {
        "weights": {
            "GP": 0.30,
            "RBF": 0.60,
            "ExtraTrees": 0.10,
        },
        "exploration": 0.15,
        "cluster_weight": 0.35,
        "mode": "incumbent_local",
    },

    7: {
        "weights": {
            "GP": 0.25,
            "RBF": 0.65,
            "ExtraTrees": 0.10,
        },
        "exploration": 0.05,
        "cluster_weight": 0.55,
        "mode": "winning_cluster",
    },

    8: {
        "weights": {
            "GP": 0.55,
            "RBF": 0.35,
            "ExtraTrees": 0.10,
        },
        "exploration": 0.02,
        "cluster_weight": 0.60,
        "mode": "fine_cluster",
    },
}


round_11_models = {}

for function_number in range(1, 9):
    X = datasets[function_number]["X"]
    y = datasets[function_number]["y"]

    gp_model = fit_gp(
        X,
        y,
        random_state=11000 + function_number,
    )

    rbf_model = fit_rbf(
        X,
        y,
    )

    et_model = ExtraTreesRegressor(
        n_estimators=1000,
        min_samples_leaf=1,
        max_features=1.0,
        random_state=12000 + function_number,
        n_jobs=-1,
    )

    et_model.fit(
        X,
        y,
    )

    round_11_models[function_number] = {
        "GP": gp_model,
        "RBF": rbf_model,
        "ExtraTrees": et_model,
    }

    strategy = round_11_strategy[
        function_number
    ]

    print(
        f"Function {function_number}: "
        f"{strategy['mode']}"
    )

    print(
        f"  Weights: "
        f"{strategy['weights']}"
    )

    print(
        f"  Exploration: "
        f"{strategy['exploration']:.2f}"
    )

    print(
        f"  Cluster weight: "
        f"{strategy['cluster_weight']:.2f}"
    )

    print()

Function 1: global_exploration
  Weights: {'GP': 0.33, 'RBF': 0.33, 'ExtraTrees': 0.34}
  Exploration: 0.70
  Cluster weight: 0.10



/usr/local/lib/python3.13/dist-packages/sklearn/gaussian_process/kernels.py:442: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified lower bound 0.0001. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


Function 2: narrow_boundary
  Weights: {'GP': 0.2, 'RBF': 0.25, 'ExtraTrees': 0.55}
  Exploration: 0.05
  Cluster weight: 0.35



/usr/local/lib/python3.13/dist-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified upper bound 10000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


Function 3: broad_exploration
  Weights: {'GP': 0.2, 'RBF': 0.4, 'ExtraTrees': 0.4}
  Exploration: 0.50
  Cluster weight: 0.15

Function 4: elite_local
  Weights: {'GP': 0.35, 'RBF': 0.55, 'ExtraTrees': 0.1}
  Exploration: 0.05
  Cluster weight: 0.55



/usr/local/lib/python3.13/dist-packages/sklearn/gaussian_process/_gpr.py:660: ConvergenceWarning: lbfgs failed to converge (status=2):
ABNORMAL: .

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  _check_optimize_result("lbfgs", opt_res)


Function 5: fine_boundary
  Weights: {'GP': 0.15, 'RBF': 0.55, 'ExtraTrees': 0.3}
  Exploration: 0.00
  Cluster weight: 0.45

Function 6: incumbent_local
  Weights: {'GP': 0.3, 'RBF': 0.6, 'ExtraTrees': 0.1}
  Exploration: 0.15
  Cluster weight: 0.35

Function 7: winning_cluster
  Weights: {'GP': 0.25, 'RBF': 0.65, 'ExtraTrees': 0.1}
  Exploration: 0.05
  Cluster weight: 0.55



/usr/local/lib/python3.13/dist-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 5 of parameter k1__k2__length_scale is close to the specified upper bound 10000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 7 of parameter k1__k2__length_scale is close to the specified upper bound 10000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


Function 8: fine_cluster
  Weights: {'GP': 0.55, 'RBF': 0.35, 'ExtraTrees': 0.1}
  Exploration: 0.02
  Cluster weight: 0.60



## 96. Generate Round 11 Cluster-Aware Candidate Pools

In [ ]:
from scipy.stats import qmc
from sklearn.neighbors import NearestNeighbors

round_11_candidate_pools = {}

rng = np.random.default_rng(13000)


def clean_candidates(candidates, sources):
    candidates = np.clip(
        candidates,
        0.000001,
        0.999999,
    )

    candidates = np.round(
        candidates,
        6,
    )

    candidates = np.clip(
        candidates,
        0.000001,
        0.999999,
    )

    _, unique_indices = np.unique(
        candidates,
        axis=0,
        return_index=True,
    )

    unique_indices = np.sort(
        unique_indices
    )

    return (
        candidates[unique_indices],
        sources[unique_indices],
    )


for function_number in range(1, 9):
    X = datasets[function_number]["X"]
    y = datasets[function_number]["y"]

    dimension = X.shape[1]

    elite = round_11_elite_regions[
        function_number
    ]

    incumbent = elite["best_x"]
    elite_centroid = elite["centroid"]
    elite_spread = elite["spread"]

    candidate_blocks = []
    source_blocks = []

    # F1: mostly global exploration
    if function_number == 1:
        sampler = qmc.Sobol(
            d=dimension,
            scramble=True,
            seed=13001,
        )

        global_candidates = sampler.random_base2(
            m=16
        )

        local_scale = np.maximum(
            elite_spread * 0.60,
            0.03,
        )

        local_candidates = (
            incumbent
            + rng.normal(
                0.0,
                local_scale,
                size=(8192, dimension),
            )
        )

        candidate_blocks = [
            global_candidates,
            local_candidates,
        ]

        source_blocks = [
            np.array(
                ["global"] * len(global_candidates)
            ),
            np.array(
                ["elite_local"] * len(local_candidates)
            ),
        ]

    # F2: microscopic search around the sharp x1 optimum,
    # keeping x2 at the empirically successful upper boundary
    elif function_number == 2:
        x1_grid = np.arange(
            0.692500,
            0.695001,
            0.000001,
        )

        candidates = np.column_stack([
            x1_grid,
            np.full_like(
                x1_grid,
                0.999999,
            ),
        ])

        candidate_blocks = [
            candidates
        ]

        source_blocks = [
            np.array(
                ["boundary_refinement"]
                * len(candidates)
            )
        ]

    # F3: broad exploration with some elite-region sampling
    elif function_number == 3:
        sampler = qmc.Sobol(
            d=dimension,
            scramble=True,
            seed=13003,
        )

        global_candidates = sampler.random_base2(
            m=16
        )

        local_scale = np.maximum(
            elite_spread * 0.75,
            0.04,
        )

        local_candidates = (
            elite_centroid
            + rng.normal(
                0.0,
                local_scale,
                size=(16384, dimension),
            )
        )

        candidate_blocks = [
            global_candidates,
            local_candidates,
        ]

        source_blocks = [
            np.array(
                ["global"] * len(global_candidates)
            ),
            np.array(
                ["elite_local"] * len(local_candidates)
            ),
        ]

    # F4: very tight exploitation around the new winning point
    elif function_number == 4:
        incumbent_scale = np.maximum(
            elite_spread * 0.45,
            0.0025,
        )

        centroid_scale = np.maximum(
            elite_spread * 0.60,
            0.0035,
        )

        incumbent_candidates = (
            incumbent
            + rng.normal(
                0.0,
                incumbent_scale,
                size=(49152, dimension),
            )
        )

        centroid_candidates = (
            elite_centroid
            + rng.normal(
                0.0,
                centroid_scale,
                size=(12288, dimension),
            )
        )

        sampler = qmc.Sobol(
            d=dimension,
            scramble=True,
            seed=13004,
        )

        global_candidates = sampler.random_base2(
            m=12
        )

        candidate_blocks = [
            incumbent_candidates,
            centroid_candidates,
            global_candidates,
        ]

        source_blocks = [
            np.array(
                ["incumbent_local"]
                * len(incumbent_candidates)
            ),
            np.array(
                ["elite_centroid"]
                * len(centroid_candidates)
            ),
            np.array(
                ["global"]
                * len(global_candidates)
            ),
        ]

    # F5: extremely fine one-dimensional refinement near x2 = 0.10
    elif function_number == 5:
        x2_grid = np.arange(
            0.097000,
            0.103001,
            0.000001,
        )

        candidates = np.column_stack([
            np.full_like(
                x2_grid,
                0.999999,
            ),
            x2_grid,
            np.full_like(
                x2_grid,
                0.999999,
            ),
            np.full_like(
                x2_grid,
                0.999999,
            ),
        ])

        candidate_blocks = [
            candidates
        ]

        source_blocks = [
            np.array(
                ["boundary_refinement"]
                * len(candidates)
            )
        ]

    # F6: stay centred on the empirical best,
    # but retain some cluster/global insurance
    elif function_number == 6:
        incumbent_scale = np.maximum(
            elite_spread * 0.45,
            0.008,
        )

        centroid_scale = np.maximum(
            elite_spread * 0.55,
            0.012,
        )

        incumbent_candidates = (
            incumbent
            + rng.normal(
                0.0,
                incumbent_scale,
                size=(40960, dimension),
            )
        )

        centroid_candidates = (
            elite_centroid
            + rng.normal(
                0.0,
                centroid_scale,
                size=(16384, dimension),
            )
        )

        sampler = qmc.Sobol(
            d=dimension,
            scramble=True,
            seed=13006,
        )

        global_candidates = sampler.random_base2(
            m=13
        )

        candidate_blocks = [
            incumbent_candidates,
            centroid_candidates,
            global_candidates,
        ]

        source_blocks = [
            np.array(
                ["incumbent_local"]
                * len(incumbent_candidates)
            ),
            np.array(
                ["elite_centroid"]
                * len(centroid_candidates)
            ),
            np.array(
                ["global"]
                * len(global_candidates)
            ),
        ]

    # F7: exploit the newly discovered winning basin
    elif function_number == 7:
        incumbent_scale = np.maximum(
            elite_spread * 0.40,
            0.004,
        )

        centroid_scale = np.maximum(
            elite_spread * 0.50,
            0.006,
        )

        incumbent_candidates = (
            incumbent
            + rng.normal(
                0.0,
                incumbent_scale,
                size=(49152, dimension),
            )
        )

        centroid_candidates = (
            elite_centroid
            + rng.normal(
                0.0,
                centroid_scale,
                size=(12288, dimension),
            )
        )

        direction = (
            incumbent
            - elite_centroid
        )

        alpha = np.linspace(
            0.05,
            0.75,
            4096,
        )

        directional_candidates = (
            incumbent[None, :]
            + alpha[:, None]
            * direction[None, :]
        )

        candidate_blocks = [
            incumbent_candidates,
            centroid_candidates,
            directional_candidates,
        ]

        source_blocks = [
            np.array(
                ["incumbent_local"]
                * len(incumbent_candidates)
            ),
            np.array(
                ["elite_centroid"]
                * len(centroid_candidates)
            ),
            np.array(
                ["directional"]
                * len(directional_candidates)
            ),
        ]

    # F8: very fine refinement in the high-performing cluster
    else:
        incumbent_scale = np.maximum(
            elite_spread * 0.28,
            0.0025,
        )

        centroid_scale = np.maximum(
            elite_spread * 0.35,
            0.0035,
        )

        incumbent_candidates = (
            incumbent
            + rng.normal(
                0.0,
                incumbent_scale,
                size=(57344, dimension),
            )
        )

        centroid_candidates = (
            elite_centroid
            + rng.normal(
                0.0,
                centroid_scale,
                size=(8192, dimension),
            )
        )

        direction = (
            incumbent
            - elite_centroid
        )

        alpha = np.linspace(
            0.05,
            0.50,
            4096,
        )

        directional_candidates = (
            incumbent[None, :]
            + alpha[:, None]
            * direction[None, :]
        )

        candidate_blocks = [
            incumbent_candidates,
            centroid_candidates,
            directional_candidates,
        ]

        source_blocks = [
            np.array(
                ["incumbent_local"]
                * len(incumbent_candidates)
            ),
            np.array(
                ["elite_centroid"]
                * len(centroid_candidates)
            ),
            np.array(
                ["directional"]
                * len(directional_candidates)
            ),
        ]

    candidates = np.vstack(
        candidate_blocks
    )

    sources = np.concatenate(
        source_blocks
    )

    candidates, sources = clean_candidates(
        candidates,
        sources,
    )

    nearest_model = NearestNeighbors(
        n_neighbors=1
    )

    nearest_model.fit(X)

    observed_distance = nearest_model.kneighbors(
        candidates,
        return_distance=True,
    )[0][:, 0]

    keep = observed_distance > 0.000001

    candidates = candidates[keep]
    sources = sources[keep]
    observed_distance = observed_distance[keep]

    cluster_model = round_11_clusters[
        function_number
    ]["model"]

    predicted_cluster = cluster_model.predict(
        candidates
    )

    best_cluster = round_11_clusters[
        function_number
    ]["best_cluster"]

    cluster_match = (
        predicted_cluster
        == best_cluster
    )

    elite_distance = np.linalg.norm(
        candidates
        - elite_centroid,
        axis=1,
    )

    round_11_candidate_pools[
        function_number
    ] = {
        "X": candidates,
        "source": sources,
        "observed_distance": observed_distance,
        "elite_distance": elite_distance,
        "predicted_cluster": predicted_cluster,
        "cluster_match": cluster_match,
        "incumbent": incumbent,
        "elite_centroid": elite_centroid,
    }

    print(
        f"Function {function_number}: "
        f"{len(candidates)} candidates"
    )

    print(
        f"  Current best: "
        f"{np.round(incumbent, 6)}"
    )

    unique, counts = np.unique(
        sources,
        return_counts=True,
    )

    print(
        f"  Sources: "
        f"{dict(zip(unique, counts))}"
    )

    print(
        f"  In winning cluster: "
        f"{np.mean(cluster_match):.3f}"
    )

    print(
        f"  Median distance from observed data: "
        f"{np.median(observed_distance):.6f}"
    )

    print()

Function 1: 73728 candidates
  Current best: [0.731024 0.733   ]
  Sources: {np.str_('elite_local'): np.int64(8192), np.str_('global'): np.int64(65536)}
  In winning cluster: 0.610
  Median distance from observed data: 0.102727

Function 2: 2499 candidates
  Current best: [0.69378  0.999999]
  Sources: {np.str_('boundary_refinement'): np.int64(2499)}
  In winning cluster: 1.000
  Median distance from observed data: 0.000626

Function 3: 81920 candidates
  Current best: [0.492581 0.611593 0.340176]
  Sources: {np.str_('elite_local'): np.int64(16384), np.str_('global'): np.int64(65536)}
  In winning cluster: 0.392
  Median distance from observed data: 0.180029

Function 4: 65536 candidates
  Current best: [0.362916 0.396669 0.414978 0.428926]
  Sources: {np.str_('elite_centroid'): np.int64(12288), np.str_('global'): np.int64(4096), np.str_('incumbent_local'): np.int64(49152)}
  In winning cluster: 0.954
  Median distance from observed data: 0.009965

Function 5: 6000 candidates
  Current

## 97. Score Round 11 Cluster-Aware Candidates

In [ ]:
from scipy.stats import rankdata

round_11_scored_candidates = {}

for function_number in range(1, 9):
    pool = round_11_candidate_pools[
        function_number
    ]

    candidates = pool["X"]

    models = round_11_models[
        function_number
    ]

    strategy = round_11_strategy[
        function_number
    ]

    weights = strategy["weights"]
    exploration_weight = strategy[
        "exploration"
    ]
    cluster_weight = strategy[
        "cluster_weight"
    ]

    gp_mean, gp_std = models[
        "GP"
    ].predict(
        candidates,
        return_std=True,
    )

    rbf_info = models["RBF"]

    rbf_prediction = (
        rbf_info["model"](candidates)
        * rbf_info["y_std"]
        + rbf_info["y_mean"]
    )

    et_prediction = models[
        "ExtraTrees"
    ].predict(
        candidates
    )

    n_candidates = len(
        candidates
    )

    gp_rank = rankdata(
        gp_mean,
        method="average",
    ) / n_candidates

    rbf_rank = rankdata(
        rbf_prediction,
        method="average",
    ) / n_candidates

    et_rank = rankdata(
        et_prediction,
        method="average",
    ) / n_candidates

    exploration_rank = rankdata(
        pool["observed_distance"],
        method="average",
    ) / n_candidates

    elite_X = round_11_elite_regions[
        function_number
    ]["elite_X"]

    nearest_elite_distance = np.min(
        np.linalg.norm(
            candidates[:, None, :]
            - elite_X[None, :, :],
            axis=2,
        ),
        axis=1,
    )

    elite_proximity_rank = rankdata(
        -nearest_elite_distance,
        method="average",
    ) / n_candidates

    cluster_match = pool[
        "cluster_match"
    ].astype(float)

    cluster_score = (
        0.40 * cluster_match
        + 0.60 * elite_proximity_rank
    )

    model_score = (
        weights["GP"] * gp_rank
        + weights["RBF"] * rbf_rank
        + weights["ExtraTrees"] * et_rank
    )

    final_score = (
        model_score
        + exploration_weight
        * exploration_rank
        + cluster_weight
        * cluster_score
    ) / (
        1.0
        + exploration_weight
        + cluster_weight
    )

    round_11_scored_candidates[
        function_number
    ] = {
        "X": candidates,
        "source": pool["source"],
        "GP": gp_mean,
        "GP_std": gp_std,
        "RBF": rbf_prediction,
        "ExtraTrees": et_prediction,
        "observed_distance": pool[
            "observed_distance"
        ],
        "nearest_elite_distance": nearest_elite_distance,
        "cluster_match": pool[
            "cluster_match"
        ],
        "model_score": model_score,
        "final_score": final_score,
    }

    best_index = np.argmax(
        final_score
    )

    print("=" * 70)
    print(
        f"Function {function_number}"
    )

    print(
        f"  Best candidate: "
        f"{np.round(candidates[best_index], 6)}"
    )

    print(
        f"  Source: "
        f"{pool['source'][best_index]}"
    )

    print(
        f"  Winning cluster: "
        f"{pool['cluster_match'][best_index]}"
    )

    print(
        f"  Final score: "
        f"{final_score[best_index]:.6f}"
    )

    print(
        f"  GP: "
        f"{gp_mean[best_index]:.10g}"
    )

    print(
        f"  RBF: "
        f"{rbf_prediction[best_index]:.10g}"
    )

    print(
        f"  ExtraTrees: "
        f"{et_prediction[best_index]:.10g}"
    )

    print(
        f"  Distance from tested data: "
        f"{pool['observed_distance'][best_index]:.6f}"
    )

    print(
        f"  Distance from nearest elite point: "
        f"{nearest_elite_distance[best_index]:.6f}"
    )

    print()

Function 1
  Best candidate: [0.886206 0.737631]
  Source: global
  Winning cluster: True
  Final score: 0.880418
  GP: 3.38007407e-05
  RBF: 0.0005240458502
  ExtraTrees: 1.737881317e-16
  Distance from tested data: 0.155251
  Distance from nearest elite point: 0.155251

Function 2
  Best candidate: [0.693781 0.999999]
  Source: boundary_refinement
  Winning cluster: True
  Final score: 0.803686
  GP: 0.5748250828
  RBF: 0.6301536628
  ExtraTrees: 0.6301042223
  Distance from tested data: 0.000001
  Distance from nearest elite point: 0.000001

Function 3
  Best candidate: [0.480004 0.613149 0.058672]
  Source: elite_local
  Winning cluster: True
  Final score: 0.793438
  GP: -0.02834003317
  RBF: -0.04008553653
  ExtraTrees: -0.04270508238
  Distance from tested data: 0.164373
  Distance from nearest elite point: 0.164373

Function 4
  Best candidate: [0.361979 0.397791 0.411567 0.42992 ]
  Source: incumbent_local
  Winning cluster: True
  Final score: 0.917469
  GP: 0.6103869196
  RB

## 98. Final Targeted Safety Checks for Round 11

In [ ]:
def evaluate_round_11_points(function_number, points):
    points = np.asarray(points)

    models = round_11_models[
        function_number
    ]

    gp_mean, gp_std = models[
        "GP"
    ].predict(
        points,
        return_std=True,
    )

    rbf_info = models["RBF"]

    rbf_prediction = (
        rbf_info["model"](points)
        * rbf_info["y_std"]
        + rbf_info["y_mean"]
    )

    et_prediction = models[
        "ExtraTrees"
    ].predict(points)

    X_existing = datasets[
        function_number
    ]["X"]

    distance = np.min(
        np.linalg.norm(
            points[:, None, :]
            - X_existing[None, :, :],
            axis=2,
        ),
        axis=1,
    )

    return {
        "X": points,
        "GP": gp_mean,
        "GP_std": gp_std,
        "RBF": rbf_prediction,
        "ExtraTrees": et_prediction,
        "distance": distance,
    }


# --------------------------------------------------
# Function 2
# Two controlled scans:
# A) keep x1 at the known optimum and move x2 slightly
# B) keep x2 at the upper boundary and move x1 slightly
# --------------------------------------------------

f2_x2 = np.linspace(
    0.985,
    0.999,
    141,
)

f2_scan_x2 = np.column_stack([
    np.full_like(
        f2_x2,
        0.693780,
    ),
    f2_x2,
])

f2_x1 = np.linspace(
    0.6925,
    0.6950,
    251,
)

f2_scan_x1 = np.column_stack([
    f2_x1,
    np.full_like(
        f2_x1,
        0.999999,
    ),
])

f2_points = np.vstack([
    f2_scan_x2,
    f2_scan_x1,
])

f2_check = evaluate_round_11_points(
    2,
    f2_points,
)

f2_valid = (
    f2_check["distance"]
    >= 0.0005
)

f2_weights = round_11_strategy[
    2
]["weights"]

f2_n = np.sum(
    f2_valid
)

f2_gp_rank = rankdata(
    f2_check["GP"][f2_valid]
) / f2_n

f2_rbf_rank = rankdata(
    f2_check["RBF"][f2_valid]
) / f2_n

f2_et_rank = rankdata(
    f2_check["ExtraTrees"][f2_valid]
) / f2_n

f2_score = (
    f2_weights["GP"] * f2_gp_rank
    + f2_weights["RBF"] * f2_rbf_rank
    + f2_weights["ExtraTrees"] * f2_et_rank
)

f2_valid_indices = np.where(
    f2_valid
)[0]

f2_best_index = f2_valid_indices[
    np.argmax(f2_score)
]


# --------------------------------------------------
# Function 5
# Explore immediately on both sides of x2 = 0.10,
# but reject near-duplicates of tested values.
# --------------------------------------------------

f5_x2 = np.linspace(
    0.080,
    0.130,
    501,
)

f5_points = np.column_stack([
    np.full_like(
        f5_x2,
        0.999999,
    ),
    f5_x2,
    np.full_like(
        f5_x2,
        0.999999,
    ),
    np.full_like(
        f5_x2,
        0.999999,
    ),
])

f5_check = evaluate_round_11_points(
    5,
    f5_points,
)

f5_valid = (
    f5_check["distance"]
    >= 0.001
)

f5_weights = round_11_strategy[
    5
]["weights"]

f5_n = np.sum(
    f5_valid
)

f5_gp_rank = rankdata(
    f5_check["GP"][f5_valid]
) / f5_n

f5_rbf_rank = rankdata(
    f5_check["RBF"][f5_valid]
) / f5_n

f5_et_rank = rankdata(
    f5_check["ExtraTrees"][f5_valid]
) / f5_n

f5_score = (
    f5_weights["GP"] * f5_gp_rank
    + f5_weights["RBF"] * f5_rbf_rank
    + f5_weights["ExtraTrees"] * f5_et_rank
)

f5_valid_indices = np.where(
    f5_valid
)[0]

f5_best_index = f5_valid_indices[
    np.argmax(f5_score)
]


# --------------------------------------------------
# Function 6
# Test how far to move from the empirical incumbent
# toward the portfolio candidate.
# --------------------------------------------------

f6_incumbent = round_11_elite_regions[
    6
]["best_x"]

f6_target = np.array([
    0.472284,
    0.266578,
    0.704289,
    0.752963,
    0.000876,
])

f6_alpha = np.linspace(
    0.10,
    1.00,
    19,
)

f6_points = np.array([
    f6_incumbent
    + alpha
    * (
        f6_target
        - f6_incumbent
    )
    for alpha in f6_alpha
])

f6_check = evaluate_round_11_points(
    6,
    f6_points,
)

f6_weights = round_11_strategy[
    6
]["weights"]

f6_n = len(
    f6_points
)

f6_gp_rank = rankdata(
    f6_check["GP"]
) / f6_n

f6_rbf_rank = rankdata(
    f6_check["RBF"]
) / f6_n

f6_et_rank = rankdata(
    f6_check["ExtraTrees"]
) / f6_n

f6_score = (
    f6_weights["GP"] * f6_gp_rank
    + f6_weights["RBF"] * f6_rbf_rank
    + f6_weights["ExtraTrees"] * f6_et_rank
)

f6_best_index = np.argmax(
    f6_score
)


for function_number, result, index in [
    (2, f2_check, f2_best_index),
    (5, f5_check, f5_best_index),
    (6, f6_check, f6_best_index),
]:
    print(
        f"Function {function_number}"
    )

    print(
        f"  X: "
        f"{np.round(result['X'][index], 6)}"
    )

    print(
        f"  GP: "
        f"{result['GP'][index]:.10g}"
    )

    print(
        f"  RBF: "
        f"{result['RBF'][index]:.10g}"
    )

    print(
        f"  ExtraTrees: "
        f"{result['ExtraTrees'][index]:.10g}"
    )

    print(
        f"  Distance: "
        f"{result['distance'][index]:.6f}"
    )

    if function_number == 6:
        print(
            f"  Alpha: "
            f"{f6_alpha[index]:.2f}"
        )

    print()

Function 2
  X: [0.69378 0.999  ]
  GP: 0.3496234492
  RBF: 0.6290372499
  ExtraTrees: 0.6292677232
  Distance: 0.000999

Function 5
  X: [0.999999 0.0979   0.999999 0.999999]
  GP: 4441.115973
  RBF: 4441.322758
  ExtraTrees: 4441.106652
  Distance: 0.002100

Function 6
  X: [0.472284 0.266578 0.704289 0.752963 0.000876]
  GP: -0.1878379914
  RBF: -0.2420103447
  ExtraTrees: -0.2756918361
  Distance: 0.025192
  Alpha: 1.00



## 99. Final Directional Checks for Functions 2 and 5

In [ ]:
# Function 2:
# Keep x2 at the successful upper boundary and make only
# microscopic changes to x1 around the incumbent.

f2_x1 = np.linspace(
    0.693200,
    0.694400,
    241,
)

f2_points = np.column_stack([
    f2_x1,
    np.full_like(
        f2_x1,
        0.999999,
    ),
])

f2_check_final = evaluate_round_11_points(
    2,
    f2_points,
)

f2_valid = (
    f2_check_final["distance"]
    >= 0.00010
)

f2_weights = round_11_strategy[2]["weights"]

valid_indices = np.where(
    f2_valid
)[0]

n_valid = len(valid_indices)

gp_rank = rankdata(
    f2_check_final["GP"][f2_valid]
) / n_valid

rbf_rank = rankdata(
    f2_check_final["RBF"][f2_valid]
) / n_valid

et_rank = rankdata(
    f2_check_final["ExtraTrees"][f2_valid]
) / n_valid

f2_score = (
    f2_weights["GP"] * gp_rank
    + f2_weights["RBF"] * rbf_rank
    + f2_weights["ExtraTrees"] * et_rank
)

f2_best_index = valid_indices[
    np.argmax(f2_score)
]


# Function 5:
# Direct observations improved from x2=.0945 to x2=.1000,
# so search only forward from the current incumbent.

f5_x2 = np.linspace(
    0.101000,
    0.150000,
    491,
)

f5_points = np.column_stack([
    np.full_like(
        f5_x2,
        0.999999,
    ),
    f5_x2,
    np.full_like(
        f5_x2,
        0.999999,
    ),
    np.full_like(
        f5_x2,
        0.999999,
    ),
])

f5_check_final = evaluate_round_11_points(
    5,
    f5_points,
)

f5_weights = round_11_strategy[5]["weights"]

n_f5 = len(
    f5_points
)

gp_rank = rankdata(
    f5_check_final["GP"]
) / n_f5

rbf_rank = rankdata(
    f5_check_final["RBF"]
) / n_f5

et_rank = rankdata(
    f5_check_final["ExtraTrees"]
) / n_f5

f5_score = (
    f5_weights["GP"] * gp_rank
    + f5_weights["RBF"] * rbf_rank
    + f5_weights["ExtraTrees"] * et_rank
)

f5_best_index = np.argmax(
    f5_score
)


for function_number, result, index in [
    (2, f2_check_final, f2_best_index),
    (5, f5_check_final, f5_best_index),
]:
    print(f"Function {function_number}")

    print(
        f"  X: "
        f"{np.round(result['X'][index], 6)}"
    )

    print(
        f"  GP: "
        f"{result['GP'][index]:.10g}"
    )

    print(
        f"  RBF: "
        f"{result['RBF'][index]:.10g}"
    )

    print(
        f"  ExtraTrees: "
        f"{result['ExtraTrees'][index]:.10g}"
    )

    print(
        f"  Distance: "
        f"{result['distance'][index]:.6f}"
    )

    print()

Function 2
  X: [0.69388  0.999999]
  GP: 0.5746905935
  RBF: 0.6335856625
  ExtraTrees: 0.627435802
  Distance: 0.000100

Function 5
  X: [0.999999 0.101    0.999999 0.999999]
  GP: 4441.166619
  RBF: 4440.887014
  ExtraTrees: 4427.821779
  Distance: 0.001000



## 100. Select and Save Round 11 Queries

In [ ]:
round_11_queries = {
    1: [0.886206, 0.737631],

    2: [0.693880, 0.999999],

    3: [0.480004, 0.613149, 0.058672],

    4: [0.361979, 0.397791, 0.411567, 0.429920],

    5: [0.999999, 0.101000, 0.999999, 0.999999],

    6: [0.472284, 0.266578, 0.704289, 0.752963, 0.000876],

    7: [0.177518, 0.189461, 0.428307, 0.168344, 0.385409, 0.766045],

    8: [0.113828, 0.146678, 0.155712, 0.156473, 0.852144, 0.491184, 0.173507, 0.639532],
}

import json
import os

save_path = (
    "/content/drive/MyDrive/Imperial_BBO/"
    "data/Week_11/submitted_queries.json"
)

os.makedirs(
    os.path.dirname(save_path),
    exist_ok=True,
)

with open(
    save_path,
    "w",
) as f:
    json.dump(
        round_11_queries,
        f,
        indent=2,
    )

print(
    f"Saved Round 11 queries to:\n{save_path}"
)

print("\nPortal submission strings:\n")

for function_number in range(1, 9):
    query_string = "-".join(
        f"{value:.6f}"
        for value in round_11_queries[
            function_number
        ]
    )

    print(
        f"F{function_number}: "
        f"{query_string}"
    )

Saved Round 11 queries to:
/content/drive/MyDrive/Imperial_BBO/data/Week_11/submitted_queries.json

Portal submission strings:

F1: 0.886206-0.737631
F2: 0.693880-0.999999
F3: 0.480004-0.613149-0.058672
F4: 0.361979-0.397791-0.411567-0.429920
F5: 0.999999-0.101000-0.999999-0.999999
F6: 0.472284-0.266578-0.704289-0.752963-0.000876
F7: 0.177518-0.189461-0.428307-0.168344-0.385409-0.766045
F8: 0.113828-0.146678-0.155712-0.156473-0.852144-0.491184-0.173507-0.639532


## 101. Load Round 11 Returned Results

In [ ]:
round_11_results = {
    1: {
        "x": [0.886206, 0.737631],
        "y": 1.716244923782533e-55,
    },
    2: {
        "x": [0.693880, 0.999999],
        "y": 0.44674177079508415,
    },
    3: {
        "x": [0.480004, 0.613149, 0.058672],
        "y": -0.02894866193240312,
    },
    4: {
        "x": [0.361979, 0.397791, 0.411567, 0.429920],
        "y": 0.5714926753098521,
    },
    5: {
        "x": [0.999999, 0.101000, 0.999999, 0.999999],
        "y": 4441.169340842286,
    },
    6: {
        "x": [0.472284, 0.266578, 0.704289, 0.752963, 0.000876],
        "y": -0.31723688601379013,
    },
    7: {
        "x": [0.177518, 0.189461, 0.428307, 0.168344, 0.385409, 0.766045],
        "y": 2.4757340151292033,
    },
    8: {
        "x": [0.113828, 0.146678, 0.155712, 0.156473, 0.852144, 0.491184, 0.173507, 0.639532],
        "y": 9.9945840603626,
    },
}

print("Round 11 results loaded.")

Round 11 results loaded.


## 102. Update Datasets with Round 11 Results

In [ ]:
for function_number, result in round_11_results.items():
    x_new = np.array(result["x"]).reshape(1, -1)
    y_new = np.array([result["y"]])

    datasets[function_number]["X"] = np.vstack(
        [datasets[function_number]["X"], x_new]
    )

    datasets[function_number]["y"] = np.concatenate(
        [datasets[function_number]["y"], y_new]
    )

print("Round 11 results added to datasets.")

for function_number in range(1, 9):
    print(
        f"Function {function_number}: "
        f"{len(datasets[function_number]['y'])} observations"
    )

Round 11 results added to datasets.
Function 1: 21 observations
Function 2: 21 observations
Function 3: 26 observations
Function 4: 41 observations
Function 5: 31 observations
Function 6: 31 observations
Function 7: 41 observations
Function 8: 51 observations


## 103. Round 11 Performance and Trust-Region Diagnostics

In [ ]:
round_12_diagnostics = {}

for function_number in range(1, 9):
    X = datasets[function_number]["X"]
    y = datasets[function_number]["y"]

    round_11_x = X[-1]
    round_11_y = y[-1]

    round_10_x = X[-2]
    round_10_y = y[-2]

    previous_best = np.max(y[:-1])
    current_best_index = np.argmax(y)
    current_best = y[current_best_index]
    current_best_x = X[current_best_index]

    round_11_improved = (
        round_11_y > previous_best
    )

    best_before_round_10 = np.max(
        y[:-2]
    )

    round_10_improved = (
        round_10_y > best_before_round_10
    )

    recent_pattern = (
        "success_success"
        if round_10_improved and round_11_improved
        else "success_failure"
        if round_10_improved and not round_11_improved
        else "failure_success"
        if not round_10_improved and round_11_improved
        else "failure_failure"
    )

    round_12_diagnostics[
        function_number
    ] = {
        "current_best": current_best,
        "current_best_x": current_best_x,
        "round_10_y": round_10_y,
        "round_11_y": round_11_y,
        "round_10_improved": round_10_improved,
        "round_11_improved": round_11_improved,
        "recent_pattern": recent_pattern,
    }

    print("=" * 70)
    print(
        f"Function {function_number}"
    )

    print(
        f"  Current best y: "
        f"{current_best:.10g}"
    )

    print(
        f"  Current best X: "
        f"{np.round(current_best_x, 6)}"
    )

    print(
        f"  Round 10 result: "
        f"{round_10_y:.10g}"
    )

    print(
        f"  Round 11 result: "
        f"{round_11_y:.10g}"
    )

    print(
        f"  Round 10 improved: "
        f"{round_10_improved}"
    )

    print(
        f"  Round 11 improved: "
        f"{round_11_improved}"
    )

    print(
        f"  Recent pattern: "
        f"{recent_pattern}"
    )

    print()

Function 1
  Current best y: 7.710875115e-16
  Current best X: [0.731024 0.733   ]
  Round 10 result: 1.187331844e-137
  Round 11 result: 1.716244924e-55
  Round 10 improved: False
  Round 11 improved: False
  Recent pattern: failure_failure

Function 2
  Current best y: 0.6301453245
  Current best X: [0.69378  0.999999]
  Round 10 result: 0.5440923875
  Round 11 result: 0.4467417708
  Round 10 improved: False
  Round 11 improved: False
  Recent pattern: failure_failure

Function 3
  Current best y: -0.02894866193
  Current best X: [0.480004 0.613149 0.058672]
  Round 10 result: -0.0651220668
  Round 11 result: -0.02894866193
  Round 10 improved: False
  Round 11 improved: True
  Recent pattern: failure_success

Function 4
  Current best y: 0.5906583773
  Current best X: [0.362916 0.396669 0.414978 0.428926]
  Round 10 result: 0.5906583773
  Round 11 result: 0.5714926753
  Round 10 improved: True
  Round 11 improved: False
  Recent pattern: success_failure

Function 5
  Current best y:

## 104. Define Adaptive Round 12 Search Strategies

In [ ]:
round_12_strategy = {
    1: {
        "mode": "rank_local_sparse",
        "radius": 0.050,
        "coordinate_probability": 0.50,
        "directional": False,
        "notes": "Two recent failures; abandon broad global search.",
    },

    2: {
        "mode": "exact_micro_grid",
        "radius": None,
        "coordinate_probability": None,
        "directional": False,
        "notes": "Extremely sharp six-decimal peak around x1=0.693780.",
    },

    3: {
        "mode": "expanded_local_quadratic",
        "radius": 0.100,
        "coordinate_probability": 1.00,
        "directional": True,
        "notes": "Recent success; exploit new basin with local models.",
    },

    4: {
        "mode": "shrunk_local_mirror",
        "radius": 0.015,
        "coordinate_probability": 0.50,
        "directional": True,
        "notes": "Round 10 succeeded and Round 11 failed; shrink and test opposite direction.",
    },

    5: {
        "mode": "one_dimensional_forward",
        "radius": None,
        "coordinate_probability": None,
        "directional": True,
        "notes": "Treat as 1D in x2 with x1=x3=x4 fixed at 0.999999.",
    },

    6: {
        "mode": "dycors_sparse_local",
        "radius": 0.040,
        "coordinate_probability": 0.35,
        "directional": False,
        "notes": "Two recent failures; hard shrink around empirical incumbent.",
    },

    7: {
        "mode": "directional_trust_region",
        "radius": 0.060,
        "coordinate_probability": 0.40,
        "directional": True,
        "notes": "Two consecutive successes; continue basin with cautious expansion.",
    },

    8: {
        "mode": "shrunk_sparse_mirror",
        "radius": 0.025,
        "coordinate_probability": 0.30,
        "directional": True,
        "notes": "Round 10 succeeded and Round 11 slipped; return to Round 10 incumbent and shrink.",
    },
}


for function_number in range(1, 9):
    strategy = round_12_strategy[
        function_number
    ]

    diagnostic = round_12_diagnostics[
        function_number
    ]

    print("=" * 70)
    print(
        f"Function {function_number}"
    )

    print(
        f"  Recent pattern: "
        f"{diagnostic['recent_pattern']}"
    )

    print(
        f"  Mode: "
        f"{strategy['mode']}"
    )

    print(
        f"  Radius: "
        f"{strategy['radius']}"
    )

    print(
        f"  Coordinate probability: "
        f"{strategy['coordinate_probability']}"
    )

    print(
        f"  Directional search: "
        f"{strategy['directional']}"
    )

    print(
        f"  Notes: "
        f"{strategy['notes']}"
    )

    print()

Function 1
  Recent pattern: failure_failure
  Mode: rank_local_sparse
  Radius: 0.05
  Coordinate probability: 0.5
  Directional search: False
  Notes: Two recent failures; abandon broad global search.

Function 2
  Recent pattern: failure_failure
  Mode: exact_micro_grid
  Radius: None
  Coordinate probability: None
  Directional search: False
  Notes: Extremely sharp six-decimal peak around x1=0.693780.

Function 3
  Recent pattern: failure_success
  Mode: expanded_local_quadratic
  Radius: 0.1
  Coordinate probability: 1.0
  Directional search: True
  Notes: Recent success; exploit new basin with local models.

Function 4
  Recent pattern: success_failure
  Mode: shrunk_local_mirror
  Radius: 0.015
  Coordinate probability: 0.5
  Directional search: True
  Notes: Round 10 succeeded and Round 11 failed; shrink and test opposite direction.

Function 5
  Recent pattern: failure_success
  Mode: one_dimensional_forward
  Radius: None
  Coordinate probability: None
  Directional search: 

## 105. Validate Surrogates for Recent Ranking and Improvement Direction

In [ ]:
import warnings

from scipy.interpolate import RBFInterpolator
from scipy.stats import spearmanr

from sklearn.ensemble import ExtraTreesRegressor
from sklearn.exceptions import ConvergenceWarning
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, Matern, WhiteKernel
from sklearn.linear_model import Ridge
from sklearn.preprocessing import PolynomialFeatures


warnings.filterwarnings(
    "ignore",
    category=ConvergenceWarning,
)


def fit_predict_gp(
    X_train,
    y_train,
    X_test,
    random_state=0,
):
    dimension = X_train.shape[1]

    kernel = (
        ConstantKernel(
            1.0,
            (1e-3, 1e3),
        )
        * Matern(
            length_scale=np.ones(dimension),
            length_scale_bounds=(1e-4, 1e4),
            nu=2.5,
        )
        + WhiteKernel(
            noise_level=1e-8,
            noise_level_bounds=(1e-10, 1e-1),
        )
    )

    model = GaussianProcessRegressor(
        kernel=kernel,
        normalize_y=True,
        n_restarts_optimizer=3,
        random_state=random_state,
    )

    model.fit(
        X_train,
        y_train,
    )

    return model.predict(
        X_test
    )[0]


def fit_predict_rbf(
    X_train,
    y_train,
    X_test,
):
    y_mean = np.mean(
        y_train
    )

    y_std = np.std(
        y_train
    )

    if y_std < 1e-12:
        y_std = 1.0

    y_scaled = (
        y_train - y_mean
    ) / y_std

    model = RBFInterpolator(
        X_train,
        y_scaled,
        kernel="thin_plate_spline",
        smoothing=1e-8,
    )

    prediction_scaled = model(
        X_test
    )[0]

    return (
        prediction_scaled * y_std
        + y_mean
    )


def fit_predict_extra_trees(
    X_train,
    y_train,
    X_test,
    random_state=0,
):
    model = ExtraTreesRegressor(
        n_estimators=500,
        min_samples_leaf=1,
        max_features=1.0,
        random_state=random_state,
        n_jobs=-1,
    )

    model.fit(
        X_train,
        y_train,
    )

    return model.predict(
        X_test
    )[0]


def fit_predict_local_quadratic(
    X_train,
    y_train,
    X_test,
):
    dimension = X_train.shape[1]

    distances = np.linalg.norm(
        X_train - X_test[0],
        axis=1,
    )

    local_count = min(
        len(X_train),
        max(
            12,
            3 * (dimension + 1),
        ),
    )

    local_indices = np.argsort(
        distances
    )[:local_count]

    X_local = X_train[
        local_indices
    ]

    y_local = y_train[
        local_indices
    ]

    y_mean = np.mean(
        y_local
    )

    y_std = np.std(
        y_local
    )

    if y_std < 1e-12:
        y_std = 1.0

    y_scaled = (
        y_local - y_mean
    ) / y_std

    polynomial = PolynomialFeatures(
        degree=2,
        include_bias=False,
    )

    X_poly = polynomial.fit_transform(
        X_local
    )

    X_test_poly = polynomial.transform(
        X_test
    )

    model = Ridge(
        alpha=1e-3,
    )

    model.fit(
        X_poly,
        y_scaled,
    )

    prediction_scaled = model.predict(
        X_test_poly
    )[0]

    return (
        prediction_scaled * y_std
        + y_mean
    )


def balanced_direction_accuracy(
    actual,
    predicted,
):
    actual = np.asarray(
        actual,
        dtype=bool,
    )

    predicted = np.asarray(
        predicted,
        dtype=bool,
    )

    classes = np.unique(
        actual
    )

    if len(classes) == 1:
        return np.mean(
            actual == predicted
        )

    true_positive_rate = np.mean(
        predicted[actual]
    )

    true_negative_rate = np.mean(
        ~predicted[~actual]
    )

    return 0.5 * (
        true_positive_rate
        + true_negative_rate
    )


recent_validation_rounds = 6

round_12_validation = {}


# Fixed F1 transformation scale using only data
# available before the recent validation window.
f1_y = datasets[1]["y"]

f1_reference = f1_y[
    :-recent_validation_rounds
]

f1_nonzero = np.abs(
    f1_reference[
        f1_reference != 0
    ]
)

if len(f1_nonzero) > 0:
    f1_scale = np.median(
        f1_nonzero
    )
else:
    f1_scale = 1e-12

f1_scale = max(
    f1_scale,
    1e-300,
)


def transform_target(
    function_number,
    y,
):
    y = np.asarray(y)

    if function_number == 1:
        return (
            np.sign(y)
            * np.log1p(
                np.abs(y)
                / f1_scale
            )
        )

    return y.copy()


for function_number in range(1, 9):
    X = datasets[
        function_number
    ]["X"]

    y_raw = datasets[
        function_number
    ]["y"]

    test_start = (
        len(y_raw)
        - recent_validation_rounds
    )

    actual_values = []

    predictions = {
        "GP": [],
        "RBF": [],
        "ExtraTrees": [],
    }

    if X.shape[1] <= 4:
        predictions[
            "Quadratic"
        ] = []

    actual_improvement = []

    predicted_improvement = {
        name: []
        for name in predictions
    }

    for test_index in range(
        test_start,
        len(y_raw),
    ):
        X_train = X[
            :test_index
        ]

        y_train_raw = y_raw[
            :test_index
        ]

        X_test = X[
            test_index:test_index + 1
        ]

        y_test_raw = y_raw[
            test_index
        ]

        y_train = transform_target(
            function_number,
            y_train_raw,
        )

        y_test = transform_target(
            function_number,
            [y_test_raw],
        )[0]

        historical_best_raw = np.max(
            y_train_raw
        )

        historical_best = transform_target(
            function_number,
            [historical_best_raw],
        )[0]

        model_predictions = {}

        model_predictions["GP"] = (
            fit_predict_gp(
                X_train,
                y_train,
                X_test,
                random_state=15000
                + function_number
                + test_index,
            )
        )

        model_predictions["RBF"] = (
            fit_predict_rbf(
                X_train,
                y_train,
                X_test,
            )
        )

        model_predictions[
            "ExtraTrees"
        ] = fit_predict_extra_trees(
            X_train,
            y_train,
            X_test,
            random_state=16000
            + function_number
            + test_index,
        )

        if "Quadratic" in predictions:
            model_predictions[
                "Quadratic"
            ] = fit_predict_local_quadratic(
                X_train,
                y_train,
                X_test,
            )

        actual_values.append(
            y_test
        )

        actual_improvement.append(
            y_test_raw
            > historical_best_raw
        )

        for name in predictions:
            prediction = model_predictions[
                name
            ]

            predictions[
                name
            ].append(
                prediction
            )

            predicted_improvement[
                name
            ].append(
                prediction
                > historical_best
            )

    actual_values = np.asarray(
        actual_values
    )

    actual_scale = np.std(
        actual_values
    )

    if actual_scale < 1e-12:
        actual_scale = 1.0

    metrics = {}

    for name in predictions:
        pred = np.asarray(
            predictions[name]
        )

        correlation = spearmanr(
            actual_values,
            pred,
        ).statistic

        if not np.isfinite(
            correlation
        ):
            correlation = 0.0

        rmse = np.sqrt(
            np.mean(
                (
                    actual_values
                    - pred
                ) ** 2
            )
        )

        nrmse = (
            rmse
            / actual_scale
        )

        direction_accuracy = (
            balanced_direction_accuracy(
                actual_improvement,
                predicted_improvement[
                    name
                ],
            )
        )

        metrics[name] = {
            "Spearman": correlation,
            "DirectionAccuracy": direction_accuracy,
            "nRMSE": nrmse,
        }

    round_12_validation[
        function_number
    ] = metrics

    print("=" * 70)
    print(
        f"Function {function_number}"
    )

    for name, result in metrics.items():
        print(
            f"  {name:<10} "
            f"Spearman={result['Spearman']:.3f}  "
            f"Direction={result['DirectionAccuracy']:.3f}  "
            f"nRMSE={result['nRMSE']:.3f}"
        )

    print()

Function 1
  GP         Spearman=0.086  Direction=1.000  nRMSE=4.034
  RBF        Spearman=1.000  Direction=1.000  nRMSE=6.001
  ExtraTrees Spearman=0.943  Direction=1.000  nRMSE=3.472
  Quadratic  Spearman=0.314  Direction=1.000  nRMSE=4.222

Function 2
  GP         Spearman=-0.371  Direction=0.167  nRMSE=1.355
  RBF        Spearman=-0.657  Direction=0.167  nRMSE=2.419
  ExtraTrees Spearman=-0.714  Direction=0.500  nRMSE=1.675
  Quadratic  Spearman=-0.257  Direction=0.500  nRMSE=2.263

Function 3
  GP         Spearman=0.771  Direction=1.000  nRMSE=0.733
  RBF        Spearman=0.371  Direction=0.500  nRMSE=1.166
  ExtraTrees Spearman=0.600  Direction=0.500  nRMSE=0.795
  Quadratic  Spearman=-0.086  Direction=0.300  nRMSE=1.448

Function 4
  GP         Spearman=0.314  Direction=0.500  nRMSE=0.823
  RBF        Spearman=0.714  Direction=0.875  nRMSE=0.976
  ExtraTrees Spearman=0.829  Direction=0.500  nRMSE=2.414
  Quadratic  Spearman=0.600  Direction=0.500  nRMSE=1.765

Function 5
  GP    

## 106. Set Round 12 Model Influence from Recent Validation

In [ ]:
round_12_model_policy = {
    1: {
        "weights": {
            "GP": 0.05,
            "RBF": 0.55,
            "ExtraTrees": 0.40,
        },
        "model_authority": 0.55,
    },

    2: {
        "weights": {},
        "model_authority": 0.00,
    },

    3: {
        "weights": {
            "GP": 0.70,
            "RBF": 0.10,
            "ExtraTrees": 0.20,
        },
        "model_authority": 0.85,
    },

    4: {
        "weights": {
            "GP": 0.10,
            "RBF": 0.65,
            "ExtraTrees": 0.20,
            "Quadratic": 0.05,
        },
        "model_authority": 0.80,
    },

    5: {
        "weights": {
            "GP": 0.75,
            "RBF": 0.10,
            "ExtraTrees": 0.05,
            "Quadratic": 0.10,
        },
        "model_authority": 0.65,
    },

    6: {
        "weights": {
            "GP": 1.00,
        },
        "model_authority": 0.25,
    },

    7: {
        "weights": {
            "GP": 0.40,
            "RBF": 0.45,
            "ExtraTrees": 0.15,
        },
        "model_authority": 0.90,
    },

    8: {
        "weights": {
            "GP": 0.40,
            "RBF": 0.55,
            "ExtraTrees": 0.05,
        },
        "model_authority": 0.85,
    },
}


for function_number in range(1, 9):
    policy = round_12_model_policy[
        function_number
    ]

    print(f"Function {function_number}")

    print(
        f"  Model weights: "
        f"{policy['weights']}"
    )

    print(
        f"  Model authority: "
        f"{policy['model_authority']:.2f}"
    )

    print()

Function 1
  Model weights: {'GP': 0.05, 'RBF': 0.55, 'ExtraTrees': 0.4}
  Model authority: 0.55

Function 2
  Model weights: {}
  Model authority: 0.00

Function 3
  Model weights: {'GP': 0.7, 'RBF': 0.1, 'ExtraTrees': 0.2}
  Model authority: 0.85

Function 4
  Model weights: {'GP': 0.1, 'RBF': 0.65, 'ExtraTrees': 0.2, 'Quadratic': 0.05}
  Model authority: 0.80

Function 5
  Model weights: {'GP': 0.75, 'RBF': 0.1, 'ExtraTrees': 0.05, 'Quadratic': 0.1}
  Model authority: 0.65

Function 6
  Model weights: {'GP': 1.0}
  Model authority: 0.25

Function 7
  Model weights: {'GP': 0.4, 'RBF': 0.45, 'ExtraTrees': 0.15}
  Model authority: 0.90

Function 8
  Model weights: {'GP': 0.4, 'RBF': 0.55, 'ExtraTrees': 0.05}
  Model authority: 0.85



## 107. Fit Final Round 12 Models and Generate Adaptive Candidate Pools

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import Ridge


def fit_final_gp(
    X,
    y,
    random_state=0,
):
    dimension = X.shape[1]

    kernel = (
        ConstantKernel(
            1.0,
            (1e-3, 1e3),
        )
        * Matern(
            length_scale=np.ones(dimension),
            length_scale_bounds=(1e-4, 1e4),
            nu=2.5,
        )
        + WhiteKernel(
            noise_level=1e-8,
            noise_level_bounds=(1e-10, 1e-1),
        )
    )

    model = GaussianProcessRegressor(
        kernel=kernel,
        normalize_y=True,
        n_restarts_optimizer=5,
        random_state=random_state,
    )

    model.fit(
        X,
        y,
    )

    return model


def fit_final_rbf(
    X,
    y,
):
    y_mean = np.mean(y)
    y_std = np.std(y)

    if y_std < 1e-12:
        y_std = 1.0

    y_scaled = (
        y - y_mean
    ) / y_std

    model = RBFInterpolator(
        X,
        y_scaled,
        kernel="thin_plate_spline",
        smoothing=1e-8,
    )

    return {
        "model": model,
        "y_mean": y_mean,
        "y_std": y_std,
    }


def fit_local_quadratic_model(
    X,
    y,
    center,
):
    dimension = X.shape[1]

    distances = np.linalg.norm(
        X - center,
        axis=1,
    )

    local_count = min(
        len(X),
        max(
            12,
            4 * (dimension + 1),
        ),
    )

    local_indices = np.argsort(
        distances
    )[:local_count]

    X_local = X[
        local_indices
    ]

    y_local = y[
        local_indices
    ]

    y_mean = np.mean(
        y_local
    )

    y_std = np.std(
        y_local
    )

    if y_std < 1e-12:
        y_std = 1.0

    polynomial = PolynomialFeatures(
        degree=2,
        include_bias=False,
    )

    X_poly = polynomial.fit_transform(
        X_local
    )

    model = Ridge(
        alpha=1e-3,
    )

    model.fit(
        X_poly,
        (
            y_local - y_mean
        ) / y_std,
    )

    return {
        "model": model,
        "poly": polynomial,
        "y_mean": y_mean,
        "y_std": y_std,
    }


def make_sparse_local_candidates(
    center,
    radius,
    probability,
    n_candidates,
    seed,
):
    rng = np.random.default_rng(
        seed
    )

    dimension = len(center)

    perturb = rng.uniform(
        -radius,
        radius,
        size=(
            n_candidates,
            dimension,
        ),
    )

    active = rng.random(
        size=(
            n_candidates,
            dimension,
        )
    ) < probability

    empty_rows = (
        np.sum(
            active,
            axis=1,
        )
        == 0
    )

    empty_indices = np.where(
        empty_rows
    )[0]

    if len(empty_indices) > 0:
        forced_coordinates = rng.integers(
            0,
            dimension,
            size=len(empty_indices),
        )

        active[
            empty_indices,
            forced_coordinates,
        ] = True

    perturb = (
        perturb
        * active
    )

    candidates = (
        center[None, :]
        + perturb
    )

    candidates = np.clip(
        candidates,
        0.000001,
        0.999999,
    )

    return candidates


def clean_round_12_candidates(
    function_number,
    candidates,
    sources,
):
    candidates = np.clip(
        candidates,
        0.000001,
        0.999999,
    )

    candidates = np.round(
        candidates,
        6,
    )

    candidates = np.clip(
        candidates,
        0.000001,
        0.999999,
    )

    candidates, unique_indices = np.unique(
        candidates,
        axis=0,
        return_index=True,
    )

    sources = sources[
        unique_indices
    ]

    X_existing = datasets[
        function_number
    ]["X"]

    distances = np.min(
        np.linalg.norm(
            candidates[:, None, :]
            - X_existing[None, :, :],
            axis=2,
        ),
        axis=1,
    )

    keep = (
        distances
        > 1e-7
    )

    return (
        candidates[keep],
        sources[keep],
        distances[keep],
    )


round_12_models = {}

for function_number in range(1, 9):
    if function_number == 2:
        continue

    X = datasets[
        function_number
    ]["X"]

    y_raw = datasets[
        function_number
    ]["y"]

    y_model = transform_target(
        function_number,
        y_raw,
    )

    models = {}

    policy = round_12_model_policy[
        function_number
    ]

    if "GP" in policy["weights"]:
        models["GP"] = fit_final_gp(
            X,
            y_model,
            random_state=17000
            + function_number,
        )

    if "RBF" in policy["weights"]:
        models["RBF"] = fit_final_rbf(
            X,
            y_model,
        )

    if "ExtraTrees" in policy["weights"]:
        model = ExtraTreesRegressor(
            n_estimators=1000,
            min_samples_leaf=1,
            max_features=1.0,
            random_state=18000
            + function_number,
            n_jobs=-1,
        )

        model.fit(
            X,
            y_model,
        )

        models[
            "ExtraTrees"
        ] = model

    if (
        "Quadratic"
        in policy["weights"]
        and function_number != 5
    ):
        center = round_12_diagnostics[
            function_number
        ]["current_best_x"]

        models[
            "Quadratic"
        ] = fit_local_quadratic_model(
            X,
            y_model,
            center,
        )

    round_12_models[
        function_number
    ] = models


round_12_candidate_pools = {}


# --------------------------------------------------
# F1: shrunk sparse local search around incumbent
# --------------------------------------------------

f1_center = round_12_diagnostics[
    1
]["current_best_x"]

f1_candidates = make_sparse_local_candidates(
    center=f1_center,
    radius=0.050,
    probability=0.50,
    n_candidates=65536,
    seed=19001,
)

f1_sources = np.array(
    ["sparse_local"]
    * len(f1_candidates)
)

(
    f1_candidates,
    f1_sources,
    f1_distances,
) = clean_round_12_candidates(
    1,
    f1_candidates,
    f1_sources,
)

round_12_candidate_pools[1] = {
    "X": f1_candidates,
    "source": f1_sources,
    "distance": f1_distances,
}


# --------------------------------------------------
# F2: exact six-decimal micro-grid around incumbent
# --------------------------------------------------

f2_best = round_12_diagnostics[
    2
]["current_best_x"]

f2_offsets = np.arange(
    -25,
    26,
    dtype=int,
)

f2_x1 = (
    f2_best[0]
    + f2_offsets
    * 0.000001
)

f2_candidates = np.column_stack([
    f2_x1,
    np.full_like(
        f2_x1,
        0.999999,
    ),
])

f2_sources = np.array(
    ["exact_micro_grid"]
    * len(f2_candidates)
)

(
    f2_candidates,
    f2_sources,
    f2_distances,
) = clean_round_12_candidates(
    2,
    f2_candidates,
    f2_sources,
)

round_12_candidate_pools[2] = {
    "X": f2_candidates,
    "source": f2_sources,
    "distance": f2_distances,
}


# --------------------------------------------------
# F3: expanded local trust region + directional pool
# --------------------------------------------------

f3_center = round_12_diagnostics[
    3
]["current_best_x"]

f3_local = make_sparse_local_candidates(
    center=f3_center,
    radius=0.100,
    probability=1.00,
    n_candidates=65536,
    seed=19003,
)

f3_previous = datasets[
    3
]["X"][-2]

f3_direction = (
    f3_center
    - f3_previous
)

f3_alpha = np.linspace(
    0.02,
    0.50,
    4096,
)

f3_directional = (
    f3_center[None, :]
    + f3_alpha[:, None]
    * f3_direction[None, :]
)

f3_candidates = np.vstack([
    f3_local,
    f3_directional,
])

f3_sources = np.concatenate([
    np.array(
        ["local_trust"]
        * len(f3_local)
    ),
    np.array(
        ["directional"]
        * len(f3_directional)
    ),
])

(
    f3_candidates,
    f3_sources,
    f3_distances,
) = clean_round_12_candidates(
    3,
    f3_candidates,
    f3_sources,
)

round_12_candidate_pools[3] = {
    "X": f3_candidates,
    "source": f3_sources,
    "distance": f3_distances,
}


# --------------------------------------------------
# F4: shrunken sparse region + mirror failed step
# --------------------------------------------------

f4_center = round_12_diagnostics[
    4
]["current_best_x"]

f4_failed = datasets[
    4
]["X"][-1]

f4_local = make_sparse_local_candidates(
    center=f4_center,
    radius=0.015,
    probability=0.50,
    n_candidates=57344,
    seed=19004,
)

f4_mirror_direction = (
    f4_center
    - f4_failed
)

f4_alpha = np.linspace(
    0.05,
    1.50,
    8192,
)

f4_mirror = (
    f4_center[None, :]
    + f4_alpha[:, None]
    * f4_mirror_direction[None, :]
)

f4_candidates = np.vstack([
    f4_local,
    f4_mirror,
])

f4_sources = np.concatenate([
    np.array(
        ["sparse_local"]
        * len(f4_local)
    ),
    np.array(
        ["mirror"]
        * len(f4_mirror)
    ),
])

(
    f4_candidates,
    f4_sources,
    f4_distances,
) = clean_round_12_candidates(
    4,
    f4_candidates,
    f4_sources,
)

round_12_candidate_pools[4] = {
    "X": f4_candidates,
    "source": f4_sources,
    "distance": f4_distances,
}


# --------------------------------------------------
# F5: pure one-dimensional forward search
# --------------------------------------------------

f5_x2 = np.arange(
    0.101001,
    0.110001,
    0.000001,
)

f5_candidates = np.column_stack([
    np.full_like(
        f5_x2,
        0.999999,
    ),
    f5_x2,
    np.full_like(
        f5_x2,
        0.999999,
    ),
    np.full_like(
        f5_x2,
        0.999999,
    ),
])

f5_sources = np.array(
    ["forward_1d"]
    * len(f5_candidates)
)

(
    f5_candidates,
    f5_sources,
    f5_distances,
) = clean_round_12_candidates(
    5,
    f5_candidates,
    f5_sources,
)

round_12_candidate_pools[5] = {
    "X": f5_candidates,
    "source": f5_sources,
    "distance": f5_distances,
}


# --------------------------------------------------
# F6: hard-shrunk DYCORS-style sparse search
# --------------------------------------------------

f6_center = round_12_diagnostics[
    6
]["current_best_x"]

f6_candidates = make_sparse_local_candidates(
    center=f6_center,
    radius=0.040,
    probability=0.35,
    n_candidates=65536,
    seed=19006,
)

f6_sources = np.array(
    ["dycors_sparse"]
    * len(f6_candidates)
)

(
    f6_candidates,
    f6_sources,
    f6_distances,
) = clean_round_12_candidates(
    6,
    f6_candidates,
    f6_sources,
)

round_12_candidate_pools[6] = {
    "X": f6_candidates,
    "source": f6_sources,
    "distance": f6_distances,
}


# --------------------------------------------------
# F7: sparse trust region + continuation direction
# --------------------------------------------------

f7_center = round_12_diagnostics[
    7
]["current_best_x"]

f7_previous = datasets[
    7
]["X"][-2]

f7_local = make_sparse_local_candidates(
    center=f7_center,
    radius=0.060,
    probability=0.40,
    n_candidates=57344,
    seed=19007,
)

f7_direction = (
    f7_center
    - f7_previous
)

f7_alpha = np.linspace(
    0.02,
    0.75,
    8192,
)

f7_directional = (
    f7_center[None, :]
    + f7_alpha[:, None]
    * f7_direction[None, :]
)

f7_candidates = np.vstack([
    f7_local,
    f7_directional,
])

f7_sources = np.concatenate([
    np.array(
        ["sparse_local"]
        * len(f7_local)
    ),
    np.array(
        ["directional"]
        * len(f7_directional)
    ),
])

(
    f7_candidates,
    f7_sources,
    f7_distances,
) = clean_round_12_candidates(
    7,
    f7_candidates,
    f7_sources,
)

round_12_candidate_pools[7] = {
    "X": f7_candidates,
    "source": f7_sources,
    "distance": f7_distances,
}


# --------------------------------------------------
# F8: return to Round-10 best + sparse search + mirror
# --------------------------------------------------

f8_center = round_12_diagnostics[
    8
]["current_best_x"]

f8_failed = datasets[
    8
]["X"][-1]

f8_local = make_sparse_local_candidates(
    center=f8_center,
    radius=0.025,
    probability=0.30,
    n_candidates=61440,
    seed=19008,
)

f8_mirror_direction = (
    f8_center
    - f8_failed
)

f8_alpha = np.linspace(
    0.05,
    1.00,
    8192,
)

f8_mirror = (
    f8_center[None, :]
    + f8_alpha[:, None]
    * f8_mirror_direction[None, :]
)

f8_candidates = np.vstack([
    f8_local,
    f8_mirror,
])

f8_sources = np.concatenate([
    np.array(
        ["sparse_local"]
        * len(f8_local)
    ),
    np.array(
        ["mirror"]
        * len(f8_mirror)
    ),
])

(
    f8_candidates,
    f8_sources,
    f8_distances,
) = clean_round_12_candidates(
    8,
    f8_candidates,
    f8_sources,
)

round_12_candidate_pools[8] = {
    "X": f8_candidates,
    "source": f8_sources,
    "distance": f8_distances,
}


for function_number in range(1, 9):
    pool = round_12_candidate_pools[
        function_number
    ]

    unique, counts = np.unique(
        pool["source"],
        return_counts=True,
    )

    print("=" * 70)
    print(
        f"Function {function_number}"
    )

    print(
        f"  Candidates: "
        f"{len(pool['X'])}"
    )

    print(
        f"  Sources: "
        f"{dict(zip(unique, counts))}"
    )

    print(
        f"  Median tested-point distance: "
        f"{np.median(pool['distance']):.6f}"
    )

    print(
        f"  Maximum tested-point distance: "
        f"{np.max(pool['distance']):.6f}"
    )

    print()

Function 1
  Candidates: 60070
  Sources: {np.str_('sparse_local'): np.int64(60070)}
  Median tested-point distance: 0.029368
  Maximum tested-point distance: 0.070202

Function 2
  Candidates: 50
  Sources: {np.str_('exact_micro_grid'): np.int64(50)}
  Median tested-point distance: 0.000013
  Maximum tested-point distance: 0.000025

Function 3
  Candidates: 69632
  Sources: {np.str_('directional'): np.int64(4096), np.str_('local_trust'): np.int64(65536)}
  Median tested-point distance: 0.089588
  Maximum tested-point distance: 0.170213

Function 4
  Candidates: 62449
  Sources: {np.str_('mirror'): np.int64(6399), np.str_('sparse_local'): np.int64(56050)}
  Median tested-point distance: 0.009551
  Maximum tested-point distance: 0.027471

Function 5
  Candidates: 9001
  Sources: {np.str_('forward_1d'): np.int64(9001)}
  Median tested-point distance: 0.004501
  Maximum tested-point distance: 0.009001

Function 6
  Candidates: 61703
  Sources: {np.str_('dycors_sparse'): np.int64(61703)}
 

## 108. Rank Round 12 Candidates Using Validated Models and Trust-Region Geometry

In [ ]:
from scipy.stats import rankdata


geometry_targets = {
    1: (0.025, 0.018),
    3: (0.050, 0.035),
    4: (0.004, 0.004),
    5: (0.002, 0.002),
    6: (0.015, 0.012),
    7: (0.010, 0.010),
    8: (0.007, 0.006),
}


def predict_round_12_model(
    model_name,
    model_info,
    candidates,
):
    if model_name == "GP":
        return model_info.predict(
            candidates
        )

    if model_name == "RBF":
        prediction_scaled = model_info[
            "model"
        ](
            candidates
        )

        return (
            prediction_scaled
            * model_info["y_std"]
            + model_info["y_mean"]
        )

    if model_name == "ExtraTrees":
        return model_info.predict(
            candidates
        )

    if model_name == "Quadratic":
        X_poly = model_info[
            "poly"
        ].transform(
            candidates
        )

        prediction_scaled = model_info[
            "model"
        ].predict(
            X_poly
        )

        return (
            prediction_scaled
            * model_info["y_std"]
            + model_info["y_mean"]
        )

    raise ValueError(
        f"Unknown model: {model_name}"
    )


round_12_scored_candidates = {}


for function_number in range(1, 9):
    pool = round_12_candidate_pools[
        function_number
    ]

    candidates = pool["X"]

    print("=" * 70)
    print(
        f"Function {function_number}"
    )

    # F2 is intentionally not surrogate-scored.
    if function_number == 2:
        round_12_scored_candidates[
            function_number
        ] = {
            "X": candidates,
            "source": pool["source"],
            "distance": pool["distance"],
        }

        print(
            "  No surrogate scoring."
        )
        print(
            "  Exact micro-grid reserved "
            "for direct evidence-based selection."
        )
        print()

        continue

    models = round_12_models[
        function_number
    ]

    policy = round_12_model_policy[
        function_number
    ]

    requested_weights = policy[
        "weights"
    ]

    available_weights = {
        name: weight
        for name, weight
        in requested_weights.items()
        if name in models
    }

    weight_total = sum(
        available_weights.values()
    )

    available_weights = {
        name: weight / weight_total
        for name, weight
        in available_weights.items()
    }

    predictions = {}
    model_ranks = {}

    for model_name, model_weight in (
        available_weights.items()
    ):
        prediction = predict_round_12_model(
            model_name,
            models[model_name],
            candidates,
        )

        predictions[
            model_name
        ] = prediction

        model_ranks[
            model_name
        ] = (
            rankdata(
                prediction,
                method="average",
            )
            / len(candidates)
        )

    model_score = np.zeros(
        len(candidates)
    )

    for model_name, model_weight in (
        available_weights.items()
    ):
        model_score += (
            model_weight
            * model_ranks[
                model_name
            ]
        )

    target_distance, distance_scale = (
        geometry_targets[
            function_number
        ]
    )

    geometry_score = np.exp(
        -0.5
        * (
            (
                pool["distance"]
                - target_distance
            )
            / distance_scale
        ) ** 2
    )

    authority = policy[
        "model_authority"
    ]

    final_score = (
        authority
        * model_score
        + (
            1.0
            - authority
        )
        * geometry_score
    )

    best_index = np.argmax(
        final_score
    )

    round_12_scored_candidates[
        function_number
    ] = {
        "X": candidates,
        "source": pool["source"],
        "distance": pool["distance"],
        "predictions": predictions,
        "model_ranks": model_ranks,
        "model_score": model_score,
        "geometry_score": geometry_score,
        "final_score": final_score,
        "weights": available_weights,
    }

    print(
        f"  Best candidate: "
        f"{np.round(candidates[best_index], 6)}"
    )

    print(
        f"  Source: "
        f"{pool['source'][best_index]}"
    )

    print(
        f"  Model weights used: "
        f"{available_weights}"
    )

    print(
        f"  Final score: "
        f"{final_score[best_index]:.6f}"
    )

    for model_name in predictions:
        print(
            f"  {model_name}: "
            f"{predictions[model_name][best_index]:.10g}"
        )

    print(
        f"  Distance from tested data: "
        f"{pool['distance'][best_index]:.6f}"
    )

    print(
        f"  Geometry score: "
        f"{geometry_score[best_index]:.6f}"
    )

    print()

Function 1
  Best candidate: [0.752653 0.733   ]
  Source: sparse_local
  Model weights used: {'GP': 0.05, 'RBF': 0.55, 'ExtraTrees': 0.4}
  Final score: 0.921742
  GP: 88.77093232
  RBF: 95.11212462
  ExtraTrees: 68.58622122
  Distance from tested data: 0.021629
  Geometry score: 0.982620

Function 2
  No surrogate scoring.
  Exact micro-grid reserved for direct evidence-based selection.

Function 3
  Best candidate: [0.433414 0.597062 0.064055]
  Source: local_trust
  Model weights used: {'GP': 0.7, 'RBF': 0.1, 'ExtraTrees': 0.2}
  Final score: 0.923589
  GP: -0.0274067716
  RBF: -0.03219447603
  ExtraTrees: -0.03666592206
  Distance from tested data: 0.049582
  Geometry score: 0.999929

Function 4
  Best candidate: [0.362916 0.398573 0.41759  0.428926]
  Source: sparse_local
  Model weights used: {'GP': 0.1, 'RBF': 0.65, 'ExtraTrees': 0.2, 'Quadratic': 0.05}
  Final score: 0.932833
  GP: 0.5687129779
  RBF: 0.5919079729
  ExtraTrees: 0.5792853474
  Quadratic: 0.1072359466
  Distance

## 109. Inspect Critical Round 12 Candidates Before Final Selection

In [ ]:
# --------------------------------------------------
# Function 2: inspect actual observations nearest
# the sharp incumbent around x1 = 0.693780
# --------------------------------------------------

print("=" * 70)
print("FUNCTION 2 — LOCAL OBSERVED HISTORY")

X2 = datasets[2]["X"]
y2 = datasets[2]["y"]

f2_best_x = round_12_diagnostics[
    2
]["current_best_x"]

f2_order = np.argsort(
    np.linalg.norm(
        X2 - f2_best_x,
        axis=1,
    )
)

for index in f2_order[:10]:
    print(
        f"  y={y2[index]:.10g}  "
        f"x={np.round(X2[index], 6)}  "
        f"distance="
        f"{np.linalg.norm(X2[index] - f2_best_x):.6f}"
    )

print()


# --------------------------------------------------
# Function 5: inspect the empirical 1-D boundary
# profile where x1, x3 and x4 are essentially 1
# --------------------------------------------------

print("=" * 70)
print("FUNCTION 5 — EMPIRICAL BOUNDARY PROFILE")

X5 = datasets[5]["X"]
y5 = datasets[5]["y"]

f5_mask = (
    (X5[:, 0] >= 0.999)
    & (X5[:, 2] >= 0.999)
    & (X5[:, 3] >= 0.999)
)

f5_indices = np.where(
    f5_mask
)[0]

f5_indices = f5_indices[
    np.argsort(
        X5[
            f5_indices,
            1,
        ]
    )
]

for index in f5_indices:
    print(
        f"  x2={X5[index, 1]:.6f}  "
        f"y={y5[index]:.10f}"
    )

print()


# --------------------------------------------------
# Functions 3, 4, 7, 8:
# compare best candidate from each search source
# --------------------------------------------------

for function_number in [
    3,
    4,
    7,
    8,
]:
    print("=" * 70)
    print(
        f"FUNCTION {function_number} "
        f"— BEST BY SEARCH SOURCE"
    )

    scored = round_12_scored_candidates[
        function_number
    ]

    sources = scored[
        "source"
    ]

    for source in np.unique(
        sources
    ):
        mask = (
            sources == source
        )

        indices = np.where(
            mask
        )[0]

        best_index = indices[
            np.argmax(
                scored[
                    "final_score"
                ][indices]
            )
        ]

        print(
            f"  {source}"
        )

        print(
            f"    X: "
            f"{np.round(scored['X'][best_index], 6)}"
        )

        print(
            f"    Score: "
            f"{scored['final_score'][best_index]:.6f}"
        )

        for model_name, prediction in (
            scored[
                "predictions"
            ].items()
        ):
            print(
                f"    {model_name}: "
                f"{prediction[best_index]:.10g}"
            )

        print(
            f"    Distance: "
            f"{scored['distance'][best_index]:.6f}"
        )

    print()


# --------------------------------------------------
# Function 6:
# compare overall candidate with best true
# one-coordinate perturbation
# --------------------------------------------------

print("=" * 70)
print(
    "FUNCTION 6 — SPARSE MOVE CHECK"
)

scored6 = round_12_scored_candidates[
    6
]

center6 = round_12_diagnostics[
    6
]["current_best_x"]

deltas6 = (
    scored6["X"]
    - center6
)

changed_coordinates = np.sum(
    np.abs(
        deltas6
    ) > 1e-7,
    axis=1,
)

overall_index = np.argmax(
    scored6[
        "final_score"
    ]
)

one_coordinate_indices = np.where(
    changed_coordinates == 1
)[0]

one_coordinate_index = (
    one_coordinate_indices[
        np.argmax(
            scored6[
                "final_score"
            ][
                one_coordinate_indices
            ]
        )
    ]
)

for label, index in [
    (
        "OVERALL",
        overall_index,
    ),
    (
        "ONE COORDINATE",
        one_coordinate_index,
    ),
]:
    print(
        f"  {label}"
    )

    print(
        f"    X: "
        f"{np.round(scored6['X'][index], 6)}"
    )

    print(
        f"    Delta: "
        f"{np.round(deltas6[index], 6)}"
    )

    print(
        f"    GP: "
        f"{scored6['predictions']['GP'][index]:.10g}"
    )

    print(
        f"    Final score: "
        f"{scored6['final_score'][index]:.6f}"
    )

    print(
        f"    Distance: "
        f"{scored6['distance'][index]:.6f}"
    )

print()


# --------------------------------------------------
# Function 1: show the selected move explicitly
# --------------------------------------------------

print("=" * 70)
print(
    "FUNCTION 1 — SELECTED LOCAL MOVE"
)

scored1 = round_12_scored_candidates[
    1
]

f1_index = np.argmax(
    scored1[
        "final_score"
    ]
)

f1_center = round_12_diagnostics[
    1
]["current_best_x"]

print(
    f"  Current best: "
    f"{np.round(f1_center, 6)}"
)

print(
    f"  Candidate: "
    f"{np.round(scored1['X'][f1_index], 6)}"
)

print(
    f"  Delta: "
    f"{np.round(scored1['X'][f1_index] - f1_center, 6)}"
)

print(
    f"  Distance: "
    f"{scored1['distance'][f1_index]:.6f}"
)

FUNCTION 2 — LOCAL OBSERVED HISTORY
  y=0.6301453245  x=[0.69378  0.999999]  distance=0.000000
  y=0.4467417708  x=[0.69388  0.999999]  distance=0.000100
  y=0.5440923875  x=[0.69175  0.999999]  distance=0.002030
  y=0.4751705597  x=[0.738743 1.      ]  distance=0.044963
  y=0.6112052158  x=[0.702637 0.926564]  distance=0.073967
  y=0.589043136  x=[0.694184 0.825717]  distance=0.174282
  y=0.6265602259  x=[0.69378  0.763493]  distance=0.236506
  y=0.6122414108  x=[0.694873 0.763493]  distance=0.236509
  y=0.0231055498  x=[0.577713 0.771973]  distance=0.255866
  y=0.4205862396  x=[0.877791 0.778628]  distance=0.287863

FUNCTION 5 — EMPIRICAL BOUNDARY PROFILE
  x2=0.000001  y=4440.4808738393
  x2=0.000100  y=4436.3614606108
  x2=0.094500  y=4441.0367834506
  x2=0.100000  y=4441.1475102795
  x2=0.101000  y=4441.1693408423

FUNCTION 3 — BEST BY SEARCH SOURCE
  directional
    X: [0.49526  0.571749 0.061614]
    Score: 0.908522
    GP: -0.02706412696
    RBF: -0.04014256053
    ExtraTrees: 

## 110. Final Evidence-Based Checks for Functions 2, 5 and 6

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge


# ==================================================
# FUNCTION 2
# Exact six-decimal neighbors of the known optimum
# ==================================================

print("=" * 70)
print("FUNCTION 2 — EXACT MICRO-NEIGHBORS")

f2_best = round_12_diagnostics[
    2
]["current_best_x"]

f2_micro_points = np.array([
    [0.693778, 0.999999],
    [0.693779, 0.999999],
    [0.693781, 0.999999],
    [0.693782, 0.999999],
])

X2 = datasets[2]["X"]

for point in f2_micro_points:
    distance = np.min(
        np.linalg.norm(
            X2 - point,
            axis=1,
        )
    )

    print(
        f"  X={np.round(point, 6)}  "
        f"distance={distance:.6f}"
    )

print()


# ==================================================
# FUNCTION 5
# Empirical forward trend + model comparison
# ==================================================

print("=" * 70)
print("FUNCTION 5 — FORWARD 1D LADDER")

X5 = datasets[5]["X"]
y5 = datasets[5]["y"]

x2_a = 0.094500
y_a = 4441.036783450597

x2_b = 0.100000
y_b = 4441.147510279543

x2_c = 0.101000
y_c = 4441.169340842286

slope_ab = (
    y_b - y_a
) / (
    x2_b - x2_a
)

slope_bc = (
    y_c - y_b
) / (
    x2_c - x2_b
)

print(
    f"  Empirical slope .0945→.1000: "
    f"{slope_ab:.6f}"
)

print(
    f"  Empirical slope .1000→.1010: "
    f"{slope_bc:.6f}"
)

print()

f5_test_x2 = np.array([
    0.101500,
    0.102000,
    0.102500,
    0.103000,
    0.103500,
    0.104000,
])

f5_test_points = np.column_stack([
    np.full_like(
        f5_test_x2,
        0.999999,
    ),
    f5_test_x2,
    np.full_like(
        f5_test_x2,
        0.999999,
    ),
    np.full_like(
        f5_test_x2,
        0.999999,
    ),
])

f5_models = round_12_models[5]

f5_predictions = {}

for model_name, model_info in (
    f5_models.items()
):
    f5_predictions[
        model_name
    ] = predict_round_12_model(
        model_name,
        model_info,
        f5_test_points,
    )

for i, x2_value in enumerate(
    f5_test_x2
):
    linear_extrapolation = (
        y_c
        + slope_bc
        * (
            x2_value
            - x2_c
        )
    )

    print(
        f"  x2={x2_value:.6f}"
    )

    print(
        f"    Last-slope extrapolation: "
        f"{linear_extrapolation:.10f}"
    )

    for model_name in (
        f5_predictions
    ):
        print(
            f"    {model_name}: "
            f"{f5_predictions[model_name][i]:.10f}"
        )

print()


# ==================================================
# FUNCTION 6
# Local linear sensitivity + one-coordinate probes
# ==================================================

print("=" * 70)
print("FUNCTION 6 — ONE-COORDINATE LOCAL PROBES")

X6 = datasets[6]["X"]
y6 = datasets[6]["y"]

f6_center = round_12_diagnostics[
    6
]["current_best_x"]

distances6 = np.linalg.norm(
    X6 - f6_center,
    axis=1,
)

local_indices6 = np.argsort(
    distances6
)[:15]

X6_local = X6[
    local_indices6
]

y6_local = y6[
    local_indices6
]

scaler6 = StandardScaler()

X6_scaled = scaler6.fit_transform(
    X6_local
)

linear6 = Ridge(
    alpha=1.0,
)

linear6.fit(
    X6_scaled,
    y6_local,
)

step_sizes = [
    0.003,
    0.006,
    0.010,
]

probe_points = []
probe_labels = []

for coordinate in range(
    X6.shape[1]
):
    for direction in [
        -1,
        1,
    ]:
        for step in step_sizes:
            point = f6_center.copy()

            point[
                coordinate
            ] += (
                direction
                * step
            )

            point = np.clip(
                point,
                0.000001,
                0.999999,
            )

            probe_points.append(
                point
            )

            probe_labels.append(
                (
                    coordinate + 1,
                    direction,
                    step,
                )
            )

probe_points = np.array(
    probe_points
)

probe_scaled = scaler6.transform(
    probe_points
)

linear_predictions = (
    linear6.predict(
        probe_scaled
    )
)

gp_predictions = round_12_models[
    6
]["GP"].predict(
    probe_points
)

X6_existing = datasets[
    6
]["X"]

probe_distances = np.min(
    np.linalg.norm(
        probe_points[:, None, :]
        - X6_existing[None, :, :],
        axis=2,
    ),
    axis=1,
)

linear_rank = rankdata(
    linear_predictions
) / len(
    probe_points
)

gp_rank = rankdata(
    gp_predictions
) / len(
    probe_points
)

probe_score = (
    0.75 * linear_rank
    + 0.25 * gp_rank
)

top_indices = np.argsort(
    probe_score
)[::-1][:10]

for index in top_indices:
    coordinate, direction, step = (
        probe_labels[index]
    )

    sign = (
        "+"
        if direction > 0
        else "-"
    )

    print(
        f"  x{coordinate} {sign}{step:.3f}"
    )

    print(
        f"    X: "
        f"{np.round(probe_points[index], 6)}"
    )

    print(
        f"    Local linear: "
        f"{linear_predictions[index]:.10g}"
    )

    print(
        f"    GP: "
        f"{gp_predictions[index]:.10g}"
    )

    print(
        f"    Distance: "
        f"{probe_distances[index]:.6f}"
    )

FUNCTION 2 — EXACT MICRO-NEIGHBORS
  X=[0.693778 0.999999]  distance=0.000002
  X=[0.693779 0.999999]  distance=0.000001
  X=[0.693781 0.999999]  distance=0.000001
  X=[0.693782 0.999999]  distance=0.000002

FUNCTION 5 — FORWARD 1D LADDER
  Empirical slope .0945→.1000: 20.132151
  Empirical slope .1000→.1010: 21.830563

  x2=0.101500
    Last-slope extrapolation: 4441.1802561237
    GP: 4441.1741782566
    RBF: 4441.1051916954
    ExtraTrees: 4436.8051254801
  x2=0.102000
    Last-slope extrapolation: 4441.1911714050
    GP: 4441.1826709495
    RBF: 4440.9998268187
    ExtraTrees: 4430.2588024367
  x2=0.102500
    Last-slope extrapolation: 4441.2020866864
    GP: 4441.1911648032
    RBF: 4440.8648190394
    ExtraTrees: 4421.8932048059
  x2=0.103000
    Last-slope extrapolation: 4441.2130019678
    GP: 4441.1996598267
    RBF: 4440.7052757662
    ExtraTrees: 4415.3468817626
  x2=0.103500
    Last-slope extrapolation: 4441.2239172491
    GP: 4441.2081560491
    RBF: 4440.5242814744
    E

## 111. Select and Save Round 12 Queries

In [ ]:
round_12_queries = {
    1: [
        0.752653,
        0.733000,
    ],

    2: [
        0.693779,
        0.999999,
    ],

    3: [
        0.433414,
        0.597062,
        0.064055,
    ],

    4: [
        0.362916,
        0.398573,
        0.417590,
        0.428926,
    ],

    5: [
        0.999999,
        0.103500,
        0.999999,
        0.999999,
    ],

    6: [
        0.448596,
        0.283706,
        0.701605,
        0.756821,
        0.000100,
    ],

    7: [
        0.177518,
        0.189461,
        0.439638,
        0.168344,
        0.379098,
        0.766045,
    ],

    8: [
        0.114846,
        0.142684,
        0.147227,
        0.155654,
        0.850812,
        0.495220,
        0.180066,
        0.637573,
    ],
}

import json
import os

save_path = (
    "/content/drive/MyDrive/Imperial_BBO/"
    "data/Week_12/submitted_queries.json"
)

os.makedirs(
    os.path.dirname(save_path),
    exist_ok=True,
)

with open(
    save_path,
    "w",
) as f:
    json.dump(
        round_12_queries,
        f,
        indent=2,
    )

print(
    f"Saved Round 12 queries to:\n{save_path}"
)

print("\nPortal submission strings:\n")

for function_number in range(1, 9):
    query_string = "-".join(
        f"{value:.6f}"
        for value in round_12_queries[
            function_number
        ]
    )

    print(
        f"F{function_number}: "
        f"{query_string}"
    )

Saved Round 12 queries to:
/content/drive/MyDrive/Imperial_BBO/data/Week_12/submitted_queries.json

Portal submission strings:

F1: 0.752653-0.733000
F2: 0.693779-0.999999
F3: 0.433414-0.597062-0.064055
F4: 0.362916-0.398573-0.417590-0.428926
F5: 0.999999-0.103500-0.999999-0.999999
F6: 0.448596-0.283706-0.701605-0.756821-0.000100
F7: 0.177518-0.189461-0.439638-0.168344-0.379098-0.766045
F8: 0.114846-0.142684-0.147227-0.155654-0.850812-0.495220-0.180066-0.637573


## 112. Load Round 12 Returned Results

In [ ]:
round_12_results = {
    1: {
        "x": [0.752653, 0.733000],
        "y": -1.3137323096128262e-19,
    },
    2: {
        "x": [0.693779, 0.999999],
        "y": 0.7567071185005707,
    },
    3: {
        "x": [0.433414, 0.597062, 0.064055],
        "y": -0.04951139021532734,
    },
    4: {
        "x": [0.362916, 0.398573, 0.417590, 0.428926],
        "y": 0.6049585587009223,
    },
    5: {
        "x": [0.999999, 0.103500, 0.999999, 0.999999],
        "y": 4441.226323315354,
    },
    6: {
        "x": [0.448596, 0.283706, 0.701605, 0.756821, 0.000100],
        "y": -0.2355946001700345,
    },
    7: {
        "x": [0.177518, 0.189461, 0.439638, 0.168344, 0.379098, 0.766045],
        "y": 2.5208541465290666,
    },
    8: {
        "x": [0.114846, 0.142684, 0.147227, 0.155654, 0.850812, 0.495220, 0.180066, 0.637573],
        "y": 9.9963337125921,
    },
}

print("Round 12 results loaded.")

Round 12 results loaded.


## 113. Update Datasets with Round 12 Results

In [ ]:
for function_number, result in round_12_results.items():
    x_new = np.array(
        result["x"]
    ).reshape(1, -1)

    y_new = np.array([
        result["y"]
    ])

    datasets[
        function_number
    ]["X"] = np.vstack([
        datasets[
            function_number
        ]["X"],
        x_new,
    ])

    datasets[
        function_number
    ]["y"] = np.concatenate([
        datasets[
            function_number
        ]["y"],
        y_new,
    ])

print(
    "Round 12 results added to datasets."
)

for function_number in range(1, 9):
    print(
        f"Function {function_number}: "
        f"{len(datasets[function_number]['y'])} observations"
    )

Round 12 results added to datasets.
Function 1: 22 observations
Function 2: 22 observations
Function 3: 27 observations
Function 4: 42 observations
Function 5: 32 observations
Function 6: 32 observations
Function 7: 42 observations
Function 8: 52 observations


## 114. Round 12 Performance Diagnostics

In [ ]:
round_13_diagnostics = {}

for function_number in range(1, 9):
    X = datasets[function_number]["X"]
    y = datasets[function_number]["y"]

    round_12_x = X[-1]
    round_12_y = y[-1]

    previous_best = np.max(y[:-1])
    previous_best_index = np.argmax(y[:-1])

    current_best_index = np.argmax(y)
    current_best = y[current_best_index]
    current_best_x = X[current_best_index]

    improved = round_12_y > previous_best
    improvement = round_12_y - previous_best

    round_13_diagnostics[function_number] = {
        "round_12_x": round_12_x,
        "round_12_y": round_12_y,
        "previous_best": previous_best,
        "previous_best_x": X[previous_best_index],
        "current_best": current_best,
        "current_best_x": current_best_x,
        "improved": improved,
        "improvement": improvement,
    }

    print("=" * 70)
    print(f"Function {function_number}")

    print(
        f"  Previous best: "
        f"{previous_best:.12g}"
    )

    print(
        f"  Round 12 result: "
        f"{round_12_y:.12g}"
    )

    print(
        f"  Improved: "
        f"{improved}"
    )

    print(
        f"  Change: "
        f"{improvement:.12g}"
    )

    print(
        f"  Current best X: "
        f"{np.round(current_best_x, 6)}"
    )

    print()

Function 1
  Previous best: 7.7108751145e-16
  Round 12 result: -1.31373230961e-19
  Improved: False
  Change: -7.71218884681e-16
  Current best X: [0.731024 0.733   ]

Function 2
  Previous best: 0.630145324491
  Round 12 result: 0.756707118501
  Improved: True
  Change: 0.126561794009
  Current best X: [0.693779 0.999999]

Function 3
  Previous best: -0.0289486619324
  Round 12 result: -0.0495113902153
  Improved: False
  Change: -0.0205627282829
  Current best X: [0.480004 0.613149 0.058672]

Function 4
  Previous best: 0.590658377338
  Round 12 result: 0.604958558701
  Improved: True
  Change: 0.0143001813625
  Current best X: [0.362916 0.398573 0.41759  0.428926]

Function 5
  Previous best: 4441.16934084
  Round 12 result: 4441.22632332
  Improved: True
  Change: 0.0569824730683
  Current best X: [0.999999 0.1035   0.999999 0.999999]

Function 6
  Previous best: -0.258809205688
  Round 12 result: -0.23559460017
  Improved: True
  Change: 0.0232146055183
  Current best X: [4.48596

## 115. Inspect Final-Round Directional Evidence

In [ ]:
for function_number in range(1, 9):
    info = round_13_diagnostics[
        function_number
    ]

    print("=" * 70)
    print(
        f"Function {function_number}"
    )

    if info["improved"]:
        previous_x = info[
            "previous_best_x"
        ]

        current_x = info[
            "current_best_x"
        ]

        delta = (
            current_x
            - previous_x
        )

        print(
            "  Round 12 improved."
        )

        print(
            f"  Previous best X: "
            f"{np.round(previous_x, 6)}"
        )

        print(
            f"  New best X:      "
            f"{np.round(current_x, 6)}"
        )

        print(
            f"  Successful delta:"
            f" {np.round(delta, 6)}"
        )

        changed = np.where(
            np.abs(delta) > 1e-7
        )[0]

        print(
            "  Changed coordinates:",
            [
                int(i + 1)
                for i in changed
            ],
        )

    else:
        incumbent_x = info[
            "current_best_x"
        ]

        failed_x = info[
            "round_12_x"
        ]

        failed_delta = (
            failed_x
            - incumbent_x
        )

        print(
            "  Round 12 failed."
        )

        print(
            f"  Incumbent X:     "
            f"{np.round(incumbent_x, 6)}"
        )

        print(
            f"  Failed query X:  "
            f"{np.round(failed_x, 6)}"
        )

        print(
            f"  Failed delta:    "
            f"{np.round(failed_delta, 6)}"
        )

        print(
            f"  Mirror direction:"
            f" {np.round(-failed_delta, 6)}"
        )

    print()

Function 1
  Round 12 failed.
  Incumbent X:     [0.731024 0.733   ]
  Failed query X:  [0.752653 0.733   ]
  Failed delta:    [0.021629 0.      ]
  Mirror direction: [-0.021629 -0.      ]

Function 2
  Round 12 improved.
  Previous best X: [0.69378  0.999999]
  New best X:      [0.693779 0.999999]
  Successful delta: [-1.e-06  0.e+00]
  Changed coordinates: [1]

Function 3
  Round 12 failed.
  Incumbent X:     [0.480004 0.613149 0.058672]
  Failed query X:  [0.433414 0.597062 0.064055]
  Failed delta:    [-0.04659  -0.016087  0.005383]
  Mirror direction: [ 0.04659   0.016087 -0.005383]

Function 4
  Round 12 improved.
  Previous best X: [0.362916 0.396669 0.414978 0.428926]
  New best X:      [0.362916 0.398573 0.41759  0.428926]
  Successful delta: [0.       0.001904 0.002612 0.      ]
  Changed coordinates: [2, 3]

Function 5
  Round 12 improved.
  Previous best X: [0.999999 0.101    0.999999 0.999999]
  New best X:      [0.999999 0.1035   0.999999 0.999999]
  Successful delta: [0.

## 116. Build Final-Round Evidence-Based Candidate Ladders

In [ ]:
round_13_candidate_ladders = {}


# ==================================================
# F1
# Round 12 moved +x1 and failed.
# Test smaller moves in the opposite direction.
# ==================================================

f1_center = round_13_diagnostics[
    1
]["current_best_x"].copy()

f1_failed_delta = (
    round_13_diagnostics[1]["round_12_x"]
    - f1_center
)

f1_factors = np.array([
    0.25,
    0.50,
    0.75,
    1.00,
])

f1_candidates = np.array([
    f1_center
    - factor * f1_failed_delta
    for factor in f1_factors
])

round_13_candidate_ladders[1] = (
    f1_candidates
)


# ==================================================
# F2
# One-millionth move left produced a major new best.
# Continue with exact micro-steps only.
# ==================================================

f2_center = round_13_diagnostics[
    2
]["current_best_x"].copy()

f2_candidates = np.array([
    [0.693778, 0.999999],
    [0.693777, 0.999999],
    [0.693776, 0.999999],
    [0.693775, 0.999999],
])

round_13_candidate_ladders[2] = (
    f2_candidates
)


# ==================================================
# F3
# Multi-coordinate Round 12 move failed.
# Test smaller mirror moves around the incumbent.
# ==================================================

f3_center = round_13_diagnostics[
    3
]["current_best_x"].copy()

f3_failed_delta = (
    round_13_diagnostics[3]["round_12_x"]
    - f3_center
)

f3_factors = np.array([
    0.20,
    0.35,
    0.50,
    0.75,
])

f3_mirror = np.array([
    f3_center
    - factor * f3_failed_delta
    for factor in f3_factors
])

# Also test simple one-coordinate alternatives.
f3_single = []

for coordinate in range(3):
    point = f3_center.copy()

    point[coordinate] -= (
        0.25
        * f3_failed_delta[coordinate]
    )

    f3_single.append(point)

f3_candidates = np.vstack([
    f3_mirror,
    np.array(f3_single),
])

round_13_candidate_ladders[3] = (
    f3_candidates
)


# ==================================================
# F4
# Continue the successful x2/x3 direction.
# ==================================================

f4_center = round_13_diagnostics[
    4
]["current_best_x"].copy()

f4_success_delta = (
    f4_center
    - round_13_diagnostics[
        4
    ]["previous_best_x"]
)

f4_factors = np.array([
    0.25,
    0.50,
    0.75,
    1.00,
])

f4_candidates = np.array([
    f4_center
    + factor * f4_success_delta
    for factor in f4_factors
])

round_13_candidate_ladders[4] = (
    f4_candidates
)


# ==================================================
# F5
# Pure x2 continuation.
# ==================================================

f5_center = round_13_diagnostics[
    5
]["current_best_x"].copy()

f5_steps = np.array([
    0.0005,
    0.0010,
    0.0015,
    0.0020,
    0.0025,
    0.0030,
])

f5_candidates = []

for step in f5_steps:
    point = f5_center.copy()

    point[1] += step

    f5_candidates.append(point)

round_13_candidate_ladders[5] = np.array(
    f5_candidates
)


# ==================================================
# F6
# Pure x2 continuation after clear Round 12 success.
# ==================================================

f6_center = round_13_diagnostics[
    6
]["current_best_x"].copy()

f6_steps = np.array([
    0.003,
    0.005,
    0.0075,
    0.010,
    0.015,
])

f6_candidates = []

for step in f6_steps:
    point = f6_center.copy()

    point[1] += step

    f6_candidates.append(point)

round_13_candidate_ladders[6] = np.array(
    f6_candidates
)


# ==================================================
# F7
# Continue the successful x3/x5 direction.
# ==================================================

f7_center = round_13_diagnostics[
    7
]["current_best_x"].copy()

f7_success_delta = (
    f7_center
    - round_13_diagnostics[
        7
    ]["previous_best_x"]
)

f7_factors = np.array([
    0.25,
    0.50,
    0.75,
    1.00,
])

f7_candidates = np.array([
    f7_center
    + factor * f7_success_delta
    for factor in f7_factors
])

round_13_candidate_ladders[7] = (
    f7_candidates
)


# ==================================================
# F8
# Continue the successful x3/x5 direction.
# ==================================================

f8_center = round_13_diagnostics[
    8
]["current_best_x"].copy()

f8_success_delta = (
    f8_center
    - round_13_diagnostics[
        8
    ]["previous_best_x"]
)

f8_factors = np.array([
    0.25,
    0.50,
    0.75,
    1.00,
])

f8_candidates = np.array([
    f8_center
    + factor * f8_success_delta
    for factor in f8_factors
])

round_13_candidate_ladders[8] = (
    f8_candidates
)


# ==================================================
# Clip, round and print
# ==================================================

for function_number in range(1, 9):
    candidates = np.clip(
        round_13_candidate_ladders[
            function_number
        ],
        0.000001,
        0.999999,
    )

    candidates = np.round(
        candidates,
        6,
    )

    round_13_candidate_ladders[
        function_number
    ] = candidates

    print("=" * 70)
    print(
        f"Function {function_number}"
    )

    for i, candidate in enumerate(
        candidates,
        start=1,
    ):
        print(
            f"  Candidate {i}: "
            f"{candidate}"
        )

    print()

Function 1
  Candidate 1: [0.725616 0.733   ]
  Candidate 2: [0.720209 0.733   ]
  Candidate 3: [0.714802 0.733   ]
  Candidate 4: [0.709394 0.733   ]

Function 2
  Candidate 1: [0.693778 0.999999]
  Candidate 2: [0.693777 0.999999]
  Candidate 3: [0.693776 0.999999]
  Candidate 4: [0.693775 0.999999]

Function 3
  Candidate 1: [0.489322 0.616366 0.057595]
  Candidate 2: [0.49631  0.618779 0.056788]
  Candidate 3: [0.503299 0.621192 0.05598 ]
  Candidate 4: [0.514946 0.625214 0.054635]
  Candidate 5: [0.491651 0.613149 0.058672]
  Candidate 6: [0.480004 0.617171 0.058672]
  Candidate 7: [0.480004 0.613149 0.057326]

Function 4
  Candidate 1: [0.362916 0.399049 0.418243 0.428926]
  Candidate 2: [0.362916 0.399525 0.418896 0.428926]
  Candidate 3: [0.362916 0.400001 0.419549 0.428926]
  Candidate 4: [0.362916 0.400477 0.420202 0.428926]

Function 5
  Candidate 1: [0.999999 0.104    0.999999 0.999999]
  Candidate 2: [0.999999 0.1045   0.999999 0.999999]
  Candidate 3: [0.999999 0.105    0

## 117. Score Final-Round Candidate Ladders

In [ ]:
import warnings

from scipy.interpolate import RBFInterpolator
from scipy.stats import rankdata

from sklearn.ensemble import ExtraTreesRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, Matern, WhiteKernel
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import Ridge
from sklearn.exceptions import ConvergenceWarning


warnings.filterwarnings(
    "ignore",
    category=ConvergenceWarning,
)


def final_transform_target(
    function_number,
    y,
):
    y = np.asarray(y)

    if function_number == 1:
        nonzero = np.abs(
            y[y != 0]
        )

        scale = (
            np.median(nonzero)
            if len(nonzero) > 0
            else 1e-12
        )

        scale = max(
            scale,
            1e-300,
        )

        return (
            np.sign(y)
            * np.log1p(
                np.abs(y) / scale
            )
        )

    return y.copy()


def fit_final_gp_model(
    X,
    y,
    seed,
):
    d = X.shape[1]

    kernel = (
        ConstantKernel(
            1.0,
            (1e-3, 1e3),
        )
        * Matern(
            length_scale=np.ones(d),
            length_scale_bounds=(
                1e-4,
                1e4,
            ),
            nu=2.5,
        )
        + WhiteKernel(
            noise_level=1e-8,
            noise_level_bounds=(
                1e-10,
                1e-1,
            ),
        )
    )

    model = GaussianProcessRegressor(
        kernel=kernel,
        normalize_y=True,
        n_restarts_optimizer=4,
        random_state=seed,
    )

    model.fit(
        X,
        y,
    )

    return model


def fit_final_rbf_model(
    X,
    y,
):
    y_mean = np.mean(y)
    y_std = np.std(y)

    if y_std < 1e-12:
        y_std = 1.0

    model = RBFInterpolator(
        X,
        (
            y - y_mean
        ) / y_std,
        kernel="thin_plate_spline",
        smoothing=1e-8,
    )

    return (
        model,
        y_mean,
        y_std,
    )


def fit_final_quadratic(
    X,
    y,
    center,
):
    d = X.shape[1]

    distances = np.linalg.norm(
        X - center,
        axis=1,
    )

    local_count = min(
        len(X),
        max(
            12,
            4 * (d + 1),
        ),
    )

    idx = np.argsort(
        distances
    )[:local_count]

    X_local = X[idx]
    y_local = y[idx]

    poly = PolynomialFeatures(
        degree=2,
        include_bias=False,
    )

    X_poly = poly.fit_transform(
        X_local
    )

    model = Ridge(
        alpha=1e-3,
    )

    model.fit(
        X_poly,
        y_local,
    )

    return (
        model,
        poly,
    )


final_model_weights = {
    1: {
        "GP": 0.05,
        "RBF": 0.55,
        "ET": 0.40,
    },

    2: {},

    3: {
        "GP": 0.70,
        "RBF": 0.10,
        "ET": 0.20,
    },

    4: {
        "GP": 0.10,
        "RBF": 0.65,
        "ET": 0.20,
        "Quadratic": 0.05,
    },

    5: {
        "GP": 0.80,
        "RBF": 0.15,
        "ET": 0.05,
    },

    6: {
        "GP": 1.00,
    },

    7: {
        "GP": 0.40,
        "RBF": 0.45,
        "ET": 0.15,
    },

    8: {
        "GP": 0.40,
        "RBF": 0.55,
        "ET": 0.05,
    },
}


round_13_ladder_scores = {}


for function_number in range(1, 9):
    print("=" * 70)
    print(
        f"Function {function_number}"
    )

    candidates = (
        round_13_candidate_ladders[
            function_number
        ]
    )

    X = datasets[
        function_number
    ]["X"]

    y_raw = datasets[
        function_number
    ]["y"]

    nearest_distance = np.min(
        np.linalg.norm(
            candidates[:, None, :]
            - X[None, :, :],
            axis=2,
        ),
        axis=1,
    )

    # F2 is deliberately evidence-driven,
    # not surrogate-driven.
    if function_number == 2:
        for i, candidate in enumerate(
            candidates,
            start=1,
        ):
            print(
                f"  Candidate {i}: "
                f"{candidate}  "
                f"distance={nearest_distance[i - 1]:.6f}"
            )

        print(
            "  Model scoring skipped for F2."
        )
        print()

        continue

    y_model = final_transform_target(
        function_number,
        y_raw,
    )

    predictions = {}

    weights = final_model_weights[
        function_number
    ]

    if "GP" in weights:
        gp = fit_final_gp_model(
            X,
            y_model,
            seed=21000
            + function_number,
        )

        predictions["GP"] = (
            gp.predict(
                candidates
            )
        )

    if "RBF" in weights:
        (
            rbf,
            rbf_mean,
            rbf_std,
        ) = fit_final_rbf_model(
            X,
            y_model,
        )

        predictions["RBF"] = (
            rbf(
                candidates
            )
            * rbf_std
            + rbf_mean
        )

    if "ET" in weights:
        et = ExtraTreesRegressor(
            n_estimators=1000,
            random_state=22000
            + function_number,
            n_jobs=-1,
        )

        et.fit(
            X,
            y_model,
        )

        predictions["ET"] = (
            et.predict(
                candidates
            )
        )

    if "Quadratic" in weights:
        (
            quad,
            poly,
        ) = fit_final_quadratic(
            X,
            y_model,
            round_13_diagnostics[
                function_number
            ]["current_best_x"],
        )

        predictions[
            "Quadratic"
        ] = quad.predict(
            poly.transform(
                candidates
            )
        )

    consensus = np.zeros(
        len(candidates)
    )

    for model_name, weight in (
        weights.items()
    ):
        ranks = (
            rankdata(
                predictions[
                    model_name
                ],
                method="average",
            )
            / len(candidates)
        )

        consensus += (
            weight
            * ranks
        )

    round_13_ladder_scores[
        function_number
    ] = {
        "candidates": candidates,
        "predictions": predictions,
        "consensus": consensus,
        "distance": nearest_distance,
    }

    order = np.argsort(
        consensus
    )[::-1]

    for index in order:
        print(
            f"  Candidate {index + 1}: "
            f"{candidates[index]}"
        )

        print(
            f"    Consensus: "
            f"{consensus[index]:.4f}"
        )

        for model_name in predictions:
            print(
                f"    {model_name}: "
                f"{predictions[model_name][index]:.10g}"
            )

        print(
            f"    Distance: "
            f"{nearest_distance[index]:.6f}"
        )

    print()

Function 1
  Candidate 3: [0.714802 0.733   ]
    Consensus: 0.7750
    GP: 2.835903552
    RBF: 129.177602
    ET: 22.9391666
    Distance: 0.016222
  Candidate 2: [0.720209 0.733   ]
    Consensus: 0.6125
    GP: 4.991742675
    RBF: 124.5940655
    ET: 26.2596469
    Distance: 0.010815
  Candidate 1: [0.725616 0.733   ]
    Consensus: 0.5875
    GP: 27.31097413
    RBF: 113.4874814
    ET: 31.31408493
    Distance: 0.005408
  Candidate 4: [0.709394 0.733   ]
    Consensus: 0.5250
    GP: 2.692316747
    RBF: 129.0346843
    ET: 18.8541064
    Distance: 0.021630

Function 2
  Candidate 1: [0.693778 0.999999]  distance=0.000001
  Candidate 2: [0.693777 0.999999]  distance=0.000002
  Candidate 3: [0.693776 0.999999]  distance=0.000003
  Candidate 4: [0.693775 0.999999]  distance=0.000004
  Model scoring skipped for F2.

Function 3
  Candidate 6: [0.480004 0.617171 0.058672]
    Consensus: 0.8714
    GP: -0.03934654469
    RBF: -0.0280302725
    ET: -0.02960544536
    Distance: 0.004022

## 118. Select and Save Final Round 13 Queries

In [ ]:
round_13_queries = {
    1: [
        0.714802,
        0.733000,
    ],

    2: [
        0.693778,
        0.999999,
    ],

    3: [
        0.480004,
        0.617171,
        0.058672,
    ],

    4: [
        0.362916,
        0.399525,
        0.418896,
        0.428926,
    ],

    5: [
        0.999999,
        0.106000,
        0.999999,
        0.999999,
    ],

    6: [
        0.448596,
        0.293706,
        0.701605,
        0.756821,
        0.000100,
    ],

    7: [
        0.177518,
        0.189461,
        0.450969,
        0.168344,
        0.372787,
        0.766045,
    ],

    8: [
        0.114846,
        0.142684,
        0.138612,
        0.155654,
        0.847573,
        0.495220,
        0.180066,
        0.637573,
    ],
}

import json
import os

save_path = (
    "/content/drive/MyDrive/Imperial_BBO/"
    "data/Week_13/submitted_queries.json"
)

os.makedirs(
    os.path.dirname(save_path),
    exist_ok=True,
)

with open(
    save_path,
    "w",
) as f:
    json.dump(
        round_13_queries,
        f,
        indent=2,
    )

print(
    f"Saved final Round 13 queries to:\n{save_path}"
)

print("\nPortal submission strings:\n")

for function_number in range(1, 9):
    query_string = "-".join(
        f"{value:.6f}"
        for value in round_13_queries[
            function_number
        ]
    )

    print(
        f"F{function_number}: "
        f"{query_string}"
    )

Saved final Round 13 queries to:
/content/drive/MyDrive/Imperial_BBO/data/Week_13/submitted_queries.json

Portal submission strings:

F1: 0.714802-0.733000
F2: 0.693778-0.999999
F3: 0.480004-0.617171-0.058672
F4: 0.362916-0.399525-0.418896-0.428926
F5: 0.999999-0.106000-0.999999-0.999999
F6: 0.448596-0.293706-0.701605-0.756821-0.000100
F7: 0.177518-0.189461-0.450969-0.168344-0.372787-0.766045
F8: 0.114846-0.142684-0.138612-0.155654-0.847573-0.495220-0.180066-0.637573


## 120. Final BBO Results Summary

In [1]:
for function_number, result in round_13_results.items():
    x_new = np.array(result["x"]).reshape(1, -1)
    y_new = np.array([result["y"]])

    datasets[function_number]["X"] = np.vstack([
        datasets[function_number]["X"],
        x_new,
    ])

    datasets[function_number]["y"] = np.concatenate([
        datasets[function_number]["y"],
        y_new,
    ])

print("Final Round 13 results added.\n")

for function_number in range(1, 9):
    X = datasets[function_number]["X"]
    y = datasets[function_number]["y"]

    best_index = np.argmax(y)

    print("=" * 70)
    print(f"Function {function_number}")
    print(f"  Final observations: {len(y)}")
    print(f"  Best output: {y[best_index]:.12g}")
    print(f"  Best input: {np.round(X[best_index], 6)}")

NameError: name 'round_13_results' is not defined